In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# RDKit install in Colab
!pip install rdkit

In [ ]:
%%bash
set -euo pipefail

python -m pip install -U pip
python -m pip uninstall -y torch torchvision torchaudio || true

# GPU(CUDA 12.6).CPU cu126  cpu .
python -m pip install --no-cache-dir torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 \
  --index-url https://download.pytorch.org/whl/cu126

In [ ]:
%%bash
set -euo pipefail

python -m pip install -U pip

python - <<'PY'
import torch, sys, subprocess
ver = torch.__version__.split('+')[0]
cuda = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".","")
url  = f"https://data.pyg.org/whl/torch-{ver}+{cuda}.html"
pkgs = ["pyg_lib","torch_scatter","torch_sparse","torch_cluster","torch_spline_conv"]
print(f"-> torch={ver} cuda={cuda} | {url}")
subprocess.check_call([sys.executable,"-m","pip","install","-U","--no-cache-dir",*pkgs,"-f",url])
PY

python -m pip install -U --no-cache-dir torch_geometric rdkit captum matplotlib pandas

python - <<'PY'
import torch, torch_geometric
from rdkit import Chem
print("[OK] torch", torch.__version__)
print("[OK] torch_geometric", torch_geometric.__version__)
print("[OK] rdkit OK", Chem.MolFromSmiles("CCO") is not None)
PY

In [ ]:
%%bash
set -euo pipefail
echo "[RUN]  Installing PyTorch-Geometric + RDKit (Py3.12 friendly)…"

# Latest pip
python -m pip install -q --upgrade pip

# PyG(Torch/CUDAfetch)
python - << 'PY'
import torch, sys, subprocess
ver = torch.__version__.split('+')[0]
cuda = 'cu' + (torch.version.cuda or '0.0').replace('.','') if torch.cuda.is_available() else 'cpu'
if cuda == 'cu0': cuda = 'cpu'
url  = f"https://data.pyg.org/whl/torch-{ver}%2B{cuda}.html"
pkgs = ["pyg_lib","torch_scatter","torch_sparse","torch_cluster","torch_spline_conv"]
print(f"-> PyTorch {ver} / {cuda} | index: {url}")
subprocess.check_call([sys.executable,"-m","pip","install","-q",*pkgs,"-f",url])
PY

# remaining(RDKitPyPI `rdkit` )
python -m pip install -q torch_geometric rdkit captum matplotlib pandas

# Smoke test
python - << 'PY'
import torch, torch_geometric, captum
from rdkit import Chem, RDLogger; RDLogger.DisableLog('rdApp.*')
print("[OK] torch", torch.__version__)
print("[OK] torch_geometric", torch_geometric.__version__)
print("[OK] captum", captum.__version__)
print("[OK] rdkit import OK | MolFromSmiles:", Chem.MolFromSmiles("CCO") is not None)
PY

echo "[DONE]  All deps installed"

In [ ]:
# ============================================
# AID 1671200 (hERG FluxOR) expanded labeling workflow(main=QC + PAINS exclusion)
#  + Additional: also write a balanced main set with strict QC for positives and relaxed QC for negatives
# - Input:  /content/drive/MyDrive/Chemoinfo_QT/AID_1671200_datatable.csv
# - Output:
#   (0) mainset(legacy)      -> AID_1671200_labeled_MAIN.csv            ← strict QC(all rows) + PAINS/Brenk/NIH drop
#   (0b)mainset(recommended)  -> AID_1671200_labeled_MAIN_balanced.csv  ← [positive=strict QC／negative=relaxed QC]+ PAINS/Brenk/NIH drop
#   (1) base labeling        -> AID_1671200_labeled.csv
#   (2) QC(label)-> AID_1671200_labeled_QC.csv
#   (3) QC + PAINS-filtered version      -> AID_1671200_labeled_QC_noPAINS.csv
#   audit: AID_1671200_label_audit.json(balancedauditrecorded together)
# Defaultpolicy:
#   Active AND Phenotype == "Inhibitor" in at least one replicate -> 1
#   Inactive -> 0
#   Inconclusive/missing -> drop (INCONCLUSIVE_POLICY="drop")
#   ※ INCONCLUSIVE_POLICY="zero"  0 can be treated as 0 if needed
# ============================================

import os, json, re
import numpy as np
import pandas as pd

# ---- RDKit  PAINS/Brenk/NIH (SMILES) ----
try:
    from rdkit import Chem
    from rdkit.Chem import FilterCatalog, FilterCatalogParams
    RDKit_AVAILABLE = True
except Exception:
    RDKit_AVAILABLE = False

# ===== Configuration =====
ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
INPUT_CSV  = f"{ROOT}/AID_1671200_datatable.csv"

OUT_MAIN     = f"{ROOT}/AID_1671200_labeled_MAIN.csv"                # legacy: all rowsstrict QC + reactivitydrop
OUT_MAIN_BAL = f"{ROOT}/AID_1671200_labeled_MAIN_balanced.csv"       # Additional: positive/negativerelaxed QC + reactivitydrop
OUT_BASE     = f"{ROOT}/AID_1671200_labeled.csv"                     # base
OUT_QC       = f"{ROOT}/AID_1671200_labeled_QC.csv"                  # strict QC(all rows)
OUT_QC_NP    = f"{ROOT}/AID_1671200_labeled_QC_noPAINS.csv"          # strict QC + reactivitydrop
AUDIT_JSON   = f"{ROOT}/AID_1671200_label_audit.json"

# Inconclusive/missingtreated: "drop"=drop, "zero"=0
INCONCLUSIVE_POLICY = "drop"

# QCpolicy(columnautomatic)
MIN_ACTIVITY_SCORE      = 40.0      # Active(recommended)
REQUIRE_FIT_LOGAC50     = True      # Fit_LogAC50 
MIN_ABS_MAX_RESPONSE    = 10.0      # Max_Response ()
REQUIRE_FULL_OR_PARTIAL = True      # Curvecolumn full/partial only(columnskip if missing)

# Phenotype "Inhibitor" (recommended=True)
REQUIRE_INHIBITOR_PHENOTYPE = True

# SMILES column
SMILES_COL = "PUBCHEM_EXT_DATASOURCE_SMILES"

# ====== base ======
def _norm_text(x):
    if pd.isna(x): return None
    return str(x).strip()

def _any_prefix(values, prefix):
    pref = prefix.lower()
    for v in values:
        if v is None: continue
        s = str(v).lower()
        if s.startswith(pref):
            return True
    return False

def _detect_curve_ok(df):
    curve_cols = [c for c in df.columns if re.search(r'curve', c, re.I)]
    if not curve_cols:
        return pd.Series(True, index=df.index)  # column(True)
    txt = df[curve_cols].astype(str).apply(lambda r: " ".join(r.values.tolist()).lower(), axis=1)
    # pandas 1.x/2.x (str_contains)
    mask = txt.str.contains("full") | txt.str.contains("partial")
    return mask.fillna(False)

def _apply_qc_strict(df):
    """legacystrict QCall rows"""
    keep = pd.Series(True, index=df.index)
    stats = {}

    # 
    if (MIN_ACTIVITY_SCORE is not None) and ("PUBCHEM_ACTIVITY_SCORE" in df.columns):
        score = pd.to_numeric(df["PUBCHEM_ACTIVITY_SCORE"], errors="coerce")
        m = score >= float(MIN_ACTIVITY_SCORE)
        keep &= m.fillna(False)
        stats["score_ge_threshold"] = int(m.sum())

    # Fit_LogAC50
    if REQUIRE_FIT_LOGAC50 and ("Fit_LogAC50" in df.columns):
        m = df["Fit_LogAC50"].notna()
        keep &= m
        stats["has_Fit_LogAC50"] = int(m.sum())

    # Max_Response
    if (MIN_ABS_MAX_RESPONSE is not None) and ("Max_Response" in df.columns):
        mx = pd.to_numeric(df["Max_Response"], errors="coerce").abs()
        m = mx >= float(MIN_ABS_MAX_RESPONSE)
        keep &= m.fillna(False)
        stats["abs_Max_Response_ge_threshold"] = int(m.sum())

    # Curve (full/partial)
    if REQUIRE_FULL_OR_PARTIAL:
        m = _detect_curve_ok(df)
        keep &= m
        stats["curve_full_or_partial"] = int(m.sum())

    return keep, stats

def _apply_qc_balanced(df):
    """
    positive=strict QC / negative=relaxed QC
    - positive: strict 
    - negative: PUBCHEM_ACTIVITY_OUTCOME=='Inactive'  records.score/AC50/Max_Response.
            curve recordscolumn(skip if missing)
    """
    is_pos = (df["label_herg_inhibit"] == 1)
    keep = pd.Series(True, index=df.index)
    stats = {"pos_strict": {}, "neg_relaxed": {}}

    # --- positive: strict QC ---
    pos_idx = df.index[is_pos]
    if len(pos_idx):
        pos_keep = pd.Series(True, index=pos_idx)

        # score
        if (MIN_ACTIVITY_SCORE is not None) and ("PUBCHEM_ACTIVITY_SCORE" in df.columns):
            score = pd.to_numeric(df.loc[pos_idx, "PUBCHEM_ACTIVITY_SCORE"], errors="coerce")
            m = score >= float(MIN_ACTIVITY_SCORE)
            pos_keep &= m.fillna(False)
            stats["pos_strict"]["score_ge_threshold"] = int(m.sum())

        # Fit_LogAC50
        if REQUIRE_FIT_LOGAC50 and ("Fit_LogAC50" in df.columns):
            m = df.loc[pos_idx, "Fit_LogAC50"].notna()
            pos_keep &= m
            stats["pos_strict"]["has_Fit_LogAC50"] = int(m.sum())

        # Max_Response
        if (MIN_ABS_MAX_RESPONSE is not None) and ("Max_Response" in df.columns):
            mx = pd.to_numeric(df.loc[pos_idx, "Max_Response"], errors="coerce").abs()
            m = mx >= float(MIN_ABS_MAX_RESPONSE)
            pos_keep &= m.fillna(False)
            stats["pos_strict"]["abs_Max_Response_ge_threshold"] = int(m.sum())

        # Curve
        if REQUIRE_FULL_OR_PARTIAL:
            m_all = _detect_curve_ok(df)
            m = m_all.loc[pos_idx]
            pos_keep &= m
            stats["pos_strict"]["curve_full_or_partial"] = int(m.sum())

        keep.loc[pos_idx] &= pos_keep

    # --- negative: relaxed QC ---
    neg_idx = df.index[~is_pos]
    if len(neg_idx):
        neg_keep = pd.Series(True, index=neg_idx)

        # Inactive 
        outcome = df.get("PUBCHEM_ACTIVITY_OUTCOME", pd.Series(index=df.index, dtype=object)).astype(str)
        m = outcome.loc[neg_idx] == "Inactive"
        neg_keep &= m
        stats["neg_relaxed"]["outcome_inactive"] = int(m.sum())

        # Curve(column)
        if REQUIRE_FULL_OR_PARTIAL:
            m_all = _detect_curve_ok(df)
            m = m_all.loc[neg_idx]
            neg_keep &= m
            stats["neg_relaxed"]["curve_full_or_partial"] = int(m.sum())

        keep.loc[neg_idx] &= neg_keep

    return keep, stats

def _build_label(df):
    phen_cols = [c for c in df.columns if c.startswith("Phenotype-Replicate_")]
    if not phen_cols and "Phenotype" in df.columns:
        phen_cols = ["Phenotype"]

    use_phen = len(phen_cols) > 0
    if use_phen:
        phen = df[phen_cols].applymap(_norm_text)
        has_inhib = phen.apply(lambda r: _any_prefix(r.values, "Inhibitor"), axis=1)
        has_actv  = phen.apply(lambda r: _any_prefix(r.values, "Activator"), axis=1)
    else:
        has_inhib = pd.Series(False, index=df.index)
        has_actv  = pd.Series(False, index=df.index)

    outcome = df.get("PUBCHEM_ACTIVITY_OUTCOME", pd.Series(index=df.index, dtype=object)).fillna("").astype(str)

    label = np.where(outcome == "Inactive", 0, np.nan)

    active_mask = (outcome == "Active")
    if REQUIRE_INHIBITOR_PHENOTYPE and use_phen:
        active_mask = active_mask & (has_inhib == True)

    label = np.where(active_mask, 1, label)
    return label, has_inhib, has_actv, phen_cols

def _apply_inconclusive(df, label, policy="drop"):
    if policy == "drop":
        keep = ~pd.isna(label)
        out = df.loc[keep].copy()
        out["label_herg_inhibit"] = label[keep].astype(int)
        return out
    elif policy == "zero":
        lab2 = np.where(pd.isna(label), 0, label).astype(int)
        out = df.copy()
        out["label_herg_inhibit"] = lab2
        return out
    else:
        raise ValueError("INCONCLUSIVE_POLICY  'drop'  'zero'.")

def _add_reactive_flags(df, smiles_col):
    if not RDKit_AVAILABLE or smiles_col not in df.columns:
        return df.assign(PAINS_flag=False, BRENK_flag=False, NIH_flag=False), {"rdkit_used": False}

    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.NIH)
    catalog = FilterCatalog.FilterCatalog(params)

    pains, brenk, nih = [], [], []
    for smi in df[smiles_col].fillna(""):
        try:
            mol = Chem.MolFromSmiles(str(smi))
            if mol is None:
                pains.append(False); brenk.append(False); nih.append(False); continue
            matches = catalog.GetMatches(mol)
            names = [m.GetDescription() for m in matches]
            u = [n.upper() for n in names]
            pains.append(any("PAINS" in n for n in u))
            brenk.append(any("BRENK" in n for n in u))
            nih.append(any("NIH"   in n for n in u))
        except Exception:
            pains.append(False); brenk.append(False); nih.append(False)

    df2 = df.copy()
    df2["PAINS_flag"] = pains
    df2["BRENK_flag"] = brenk
    df2["NIH_flag"]   = nih
    return df2, {"rdkit_used": True}

# ===== Run =====
os.makedirs(os.path.dirname(OUT_BASE), exist_ok=True)
df = pd.read_csv(INPUT_CSV, low_memory=False)

# (1) base
label, has_inhib, has_actv, phen_cols = _build_label(df)
df_base = _apply_inconclusive(df, label, policy=INCONCLUSIVE_POLICY)
df_base.to_csv(OUT_BASE, index=False, encoding="utf-8-sig")

# (2) strict QC(legacy)
keep_qc_strict, qc_stats_strict = _apply_qc_strict(df_base)
df_qc = df_base.loc[keep_qc_strict].copy()
df_qc.to_csv(OUT_QC, index=False, encoding="utf-8-sig")

# (3) strict QC + reactivity(legacyMAIN)
df_qc_flags, flag_meta = _add_reactive_flags(df_qc, SMILES_COL)
if flag_meta.get("rdkit_used", False):
    keep_np = ~(df_qc_flags[["PAINS_flag","BRENK_flag","NIH_flag"]].any(axis=1))
    df_qc_np = df_qc_flags.loc[keep_np].copy()
else:
    df_qc_np = df_qc_flags.copy()
df_qc_np.to_csv(OUT_QC_NP, index=False, encoding="utf-8-sig")
df_qc_np.to_csv(OUT_MAIN,   index=False, encoding="utf-8-sig")  # legacyMAIN

# (4) [Additional]balanced: positive=/negative=relaxed QC
keep_bal, qc_stats_bal = _apply_qc_balanced(df_base)
df_bal = df_base.loc[keep_bal].copy()

# (5) balanced reactivity
df_bal_flags, flag_meta_bal = _add_reactive_flags(df_bal, SMILES_COL)
if flag_meta_bal.get("rdkit_used", False):
    keep_bal_np = ~(df_bal_flags[["PAINS_flag","BRENK_flag","NIH_flag"]].any(axis=1))
    df_bal_np = df_bal_flags.loc[keep_bal_np].copy()
else:
    df_bal_np = df_bal_flags.copy()
df_bal_np.to_csv(OUT_MAIN_BAL, index=False, encoding="utf-8-sig")

# ---- audit ----
audit = {
    "input_rows": int(len(df)),
    "phenotype_columns_detected": phen_cols,
    "inconclusive_policy": INCONCLUSIVE_POLICY,

    "base_rows": int(len(df_base)),
    "base_label_1_n": int((df_base["label_herg_inhibit"]==1).sum()),
    "base_label_0_n": int((df_base["label_herg_inhibit"]==0).sum()),

    "strict_qc": {
        "applied": True,
        "stats": qc_stats_strict,
        "rows_after_qc": int(len(df_qc)),
        "label_1_n": int((df_qc["label_herg_inhibit"]==1).sum()),
        "label_0_n": int((df_qc["label_herg_inhibit"]==0).sum())
    },
    "strict_qc_noPAINS": {
        "rdkit_available": RDKit_AVAILABLE,
        "smiles_col_present": (SMILES_COL in df_qc.columns),
        "rows_after_qc_noPAINS": int(len(df_qc_np)),
        "label_1_n": int((df_qc_np["label_herg_inhibit"]==1).sum()),
        "label_0_n": int((df_qc_np["label_herg_inhibit"]==0).sum()),
        "flagged_counts": {
            "PAINS_true_n": int(df_qc_flags["PAINS_flag"].sum()) if "PAINS_flag" in df_qc_flags.columns else 0,
            "BRENK_true_n": int(df_qc_flags["BRENK_flag"].sum()) if "BRENK_flag" in df_qc_flags.columns else 0,
            "NIH_true_n":   int(df_qc_flags["NIH_flag"].sum())   if "NIH_flag"   in df_qc_flags.columns else 0
        }
    },
    "balanced_qc": {
        "applied": True,
        "stats": qc_stats_bal,
        "rows_after_qc": int(len(df_bal)),
        "label_1_n": int((df_bal["label_herg_inhibit"]==1).sum()),
        "label_0_n": int((df_bal["label_herg_inhibit"]==0).sum())
    },
    "balanced_qc_noPAINS": {
        "rows_after_qc_noPAINS": int(len(df_bal_np)),
        "label_1_n": int((df_bal_np["label_herg_inhibit"]==1).sum()),
        "label_0_n": int((df_bal_np["label_herg_inhibit"]==0).sum())
    },
    "outputs": {
        "main_strict": OUT_MAIN,
        "main_balanced": OUT_MAIN_BAL,
        "base": OUT_BASE,
        "qc_strict": OUT_QC,
        "qc_strict_noPAINS": OUT_QC_NP
    }
}

with open(AUDIT_JSON, "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)

print("=== Finished ===")
print(json.dumps(audit, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# POS=strict QC_noPAINS / NEG=BASE(Inactive) set
# Input:
#  - /content/drive/MyDrive/Chemoinfo_QT/AID_1671200_labeled.csv            (BASE)
#  - /content/drive/MyDrive/Chemoinfo_QT/AID_1671200_labeled_QC_noPAINS.csv (strict QC + PAINS exclusion)
# Output:
#  - AID_1671200_labeled_MIX_POSstrict_NEGbase.csv
#  - AID_1671200_mix_audit.json
# ============================================================
import os, json, pandas as pd

ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
BASE   = f"{ROOT}/AID_1671200_labeled.csv"
QC_NP  = f"{ROOT}/AID_1671200_labeled_QC_noPAINS.csv"
OUT    = f"{ROOT}/AID_1671200_labeled_MIX_POSstrict_NEGbase.csv"
AUDIT  = f"{ROOT}/AID_1671200_mix_audit.json"

SMILES_COL = "PUBCHEM_EXT_DATASOURCE_SMILES"

df_base  = pd.read_csv(BASE, low_memory=False)
df_qcnp  = pd.read_csv(QC_NP, low_memory=False)

# Safety checks
for need in ["label_herg_inhibit"]:
    if need not in df_base.columns or need not in df_qcnp.columns:
        raise RuntimeError(f"required columns {need}  not found.")

# POS = strict QC_noPAINS positive
pos = df_qcnp[df_qcnp["label_herg_inhibit"]==1].copy()

# NEG = BASE  Inactive(=0).PAINSdrop, dropRulesAdditional.
neg = df_base[df_base["label_herg_inhibit"]==0].copy()

# column(optional): CID/SMILES/label 
keep_cols = []
for c in ["PUBCHEM_CID", SMILES_COL, "label_herg_inhibit"]:
    if c in pos.columns: keep_cols.append(c)
pos = pos[keep_cols].drop_duplicates()

keep_cols_neg = [c for c in keep_cols if c in neg.columns]
neg = neg[keep_cols_neg].drop_duplicates()

# SMILESPOS(POSSMILESNEG)
if SMILES_COL in pos.columns and SMILES_COL in neg.columns:
    neg = neg[~neg[SMILES_COL].isin(set(pos[SMILES_COL].dropna().unique()))]

df_mix = pd.concat([pos, neg], ignore_index=True).drop_duplicates()

# audit
audit = {
  "pos_from_qc_noPAINS_n": int(len(pos)),
  "neg_from_base_inactive_n": int(len(neg)),
  "mix_total_n": int(len(df_mix)),
  "mix_label_counts": df_mix["label_herg_inhibit"].value_counts(dropna=False).to_dict()
}
with open(AUDIT, "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)

df_mix.to_csv(OUT, index=False, encoding="utf-8-sig")
print("Saved:", OUT)
print(json.dumps(audit, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# ror_results_with_cids.csv  CID -> Canonical SMILES (version)
# 1) JSON property -> 2) TXT property -> 3) SDF + RDKit fallback
# Input : /content/drive/MyDrive/Chemoinfo_QT/ror_results_with_cids.csv
# Output :
#   - ror_with_smiles_long.csv  (long format (drug x CID) with SMILES)
#   - ror_with_smiles_best.csv  (one representative SMILES per drug)
#   - ror_with_smiles_audit.json(audit: HTTP)
# ============================================================

import os, re, json, time, requests, io
import pandas as pd
from typing import Dict, Any, List, Optional

# ----- environment configuration -----
ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
IN_CSV = f"{ROOT}/ror_results_with_cids.csv"
OUT_LONG = f"{ROOT}/ror_with_smiles_long.csv"
OUT_BEST = f"{ROOT}/ror_with_smiles_best.csv"
OUT_AUDIT = f"{ROOT}/ror_with_smiles_audit.json"
CACHE_PROPS = f"{ROOT}/cache_pubchem_cid2props.json"

PUBCHEM = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
TIMEOUT = 25
RETRY = 4
SLEEP = 0.25
BATCH = 200
PROPS = "CanonicalSMILES,IUPACName,IsomericSMILES,InChIKey"

# RDKit(SDF->SMILESlast resort)
try:
    from rdkit import Chem
    RD_AVAILABLE = True
except Exception:
    RD_AVAILABLE = False

# ----- utilities -----
def load_json(p):
    if os.path.exists(p):
        try:
            with open(p, "r", encoding="utf-8") as f: return json.load(f)
        except Exception: return {}
    return {}

def save_json(p, obj):
    tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, p)

def backoff(i): time.sleep(min(2**i, 8) + 0.1*i)

def http_get(url, params=None, as_text=False):
    for i in range(RETRY):
        try:
            r = requests.get(url, params=params, timeout=TIMEOUT)
            if r.status_code == 200:
                return (200, r.text if as_text else r.json())
            code = r.status_code
            if code in (400,404):   # clear failure
                return (code, None)
            # 429, 5xx retry
            backoff(i)
        except requests.RequestException:
            backoff(i)
    return (0, None)  # timeout or related issue

# ----- PubChem fetch: 3fallback -----
http_stats = {"json_ok":0,"json_404":0,"json_other":0,
              "txt_ok":0,"txt_404":0,"txt_other":0,
              "sdf_ok":0,"sdf_fail":0}

cid_cache: Dict[str, Any] = load_json(CACHE_PROPS)

def get_props_json_by_cids(cids: List[int]) -> Dict[str, Optional[dict]]:
    """batch JSON property.None(fallback)"""
    out = {str(c): cid_cache.get(str(c)) for c in cids}
    need = [c for c in cids if str(c) not in cid_cache or cid_cache[str(c)] is None]
    if not need:
        return out
    # batch request
    cid_list = ",".join(str(int(x)) for x in need)
    url = f"{PUBCHEM}/compound/cid/{cid_list}/property/{PROPS}/JSON"
    code, js = http_get(url)
    if code == 200 and js:
        props_list = js.get("PropertyTable",{}).get("Properties",[])
        for p in props_list:
            cid_cache[str(p.get("CID"))] = p
            http_stats["json_ok"] += 1
        save_json(CACHE_PROPS, cid_cache)
    elif code in (400,404):
        http_stats["json_404"] += 1
    else:
        http_stats["json_other"] += 1
    # leave unresolved entries as None
    for c in need:
        out[str(c)] = cid_cache.get(str(c))
    return out

def get_smiles_txt(cid: int, which="CanonicalSMILES") -> Optional[str]:
    """TXTversion property(JSONcan work in environments where JSON does not)"""
    url = f"{PUBCHEM}/compound/cid/{cid}/property/{which}/TXT"
    code, txt = http_get(url, as_text=True)
    if code == 200 and txt:
        http_stats["txt_ok"] += 1
        return txt.strip()
    elif code in (400,404):
        http_stats["txt_404"] += 1
    else:
        http_stats["txt_other"] += 1
    return None

def get_smiles_sdf(cid: int) -> Optional[str]:
    """SDF->RDKitconvert to SMILES(last resort)"""
    if not RD_AVAILABLE:
        http_stats["sdf_fail"] += 1
        return None
    url = f"{PUBCHEM}/compound/cid/{cid}/SDF"
    code, sdf = http_get(url, as_text=True)
    if code == 200 and sdf:
        try:
            mol = Chem.MolFromMolBlock(sdf, sanitize=True)
            if mol is None:
                http_stats["sdf_fail"] += 1
                return None
            smi = Chem.MolToSmiles(mol, canonical=True)
            http_stats["sdf_ok"] += 1
            return smi
        except Exception:
            http_stats["sdf_fail"] += 1
            return None
    else:
        http_stats["sdf_fail"] += 1
        return None

def resolve_cid_to_props(cid: int) -> Dict[str, Optional[str]]:
    """1) JSON-> 2) TXT Canonical -> 2b) TXT Isomeric -> 3) SDF"""
    # 1) JSON cache- JSONfetch
    p = cid_cache.get(str(cid))
    if not p:
        # .remaining
        url = f"{PUBCHEM}/compound/cid/{cid}/property/{PROPS}/JSON"
        code, js = http_get(url)
        if code == 200 and js:
            props_list = js.get("PropertyTable",{}).get("Properties",[])
            p = props_list[0] if props_list else None
            if p:
                cid_cache[str(cid)] = p
                save_json(CACHE_PROPS, cid_cache)
                http_stats["json_ok"] += 1
        elif code in (400,404):
            http_stats["json_404"] += 1
        else:
            http_stats["json_other"] += 1

    can_smi = p.get("CanonicalSMILES") if p else None
    iso_smi = p.get("IsomericSMILES")  if p else None
    iupac   = p.get("IUPACName")       if p else None
    inchik  = p.get("InChIKey")        if p else None

    # 2) TXTfallback
    if not can_smi:
        can_smi = get_smiles_txt(cid, "CanonicalSMILES")
        time.sleep(SLEEP)
    if not can_smi and not iso_smi:
        iso_smi = get_smiles_txt(cid, "IsomericSMILES")
        time.sleep(SLEEP)

    # 3) SDF->RDKit
    if not can_smi and not iso_smi:
        can_smi = get_smiles_sdf(cid)
        time.sleep(SLEEP)

    return {
        "CanonicalSMILES": can_smi or None,
        "IsomericSMILES": iso_smi or None,
        "IUPACName": iupac,
        "InChIKey": inchik
    }

# ----- Load input and extract CIDs -----
df = pd.read_csv(IN_CSV, dtype=str)
cid_cols = [c for c in df.columns if re.fullmatch(r"cid(\d+)?", c, flags=re.I)]
if not cid_cols:
    raise RuntimeError("cid column(cid, cid1, cid2, ...) not found.")

id_cols = [c for c in df.columns if c not in cid_cols]
rows = []
for _, row in df.iterrows():
    base = {k: row[k] for k in id_cols}
    for ccol in cid_cols:
        v = row.get(ccol)
        if v is None or str(v).strip()=="" or str(v).lower()=="na":
            continue
        try:
            cid = int(float(str(v)))
        except Exception:
            continue
        r = dict(base)
        r["cid"] = cid
        rows.append(r)

df_long = pd.DataFrame(rows).drop_duplicates()
if df_long.empty:
    raise RuntimeError("CID could not be extracted.")

# -----  JSON  -----
unique_cids = sorted(df_long["cid"].dropna().astype(int).unique().tolist())
get_props_json_by_cids(unique_cids)  # 

# -----  3 -----
props_rows = []
ok_count = 0
for cid in unique_cids:
    pr = resolve_cid_to_props(cid)
    pr["CID"] = cid
    if pr.get("CanonicalSMILES") or pr.get("IsomericSMILES"):
        ok_count += 1
    props_rows.append(pr)

df_props = pd.DataFrame(props_rows)
# Canonical  Isomeric , CanonicalSMILES(Training)
mask_fill = df_props["CanonicalSMILES"].isna() & df_props["IsomericSMILES"].notna()
df_props.loc[mask_fill, "CanonicalSMILES"] = df_props.loc[mask_fill, "IsomericSMILES"]

# ----- join & Output -----
df_long2 = df_long.merge(df_props.rename(columns={"CID":"cid"}), on="cid", how="left")
df_long2.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

# drug 1(SMILESfetchminimumCID, minimumCID)
def pick_best(g: pd.DataFrame) -> pd.Series:
    g2 = g.dropna(subset=["CanonicalSMILES"])
    if not g2.empty:
        g2 = g2.sort_values(["cid"]).head(1)
        s = g2.iloc[0].copy()
        s["status"] = "ok"
        return s
    g3 = g.sort_values(["cid"]).head(1)
    s = g3.iloc[0].copy()
    s["status"] = "no_smiles"
    return s

key = "drug" if "drug" in df_long2.columns else None
if key:
    df_best = df_long2.groupby(key, as_index=False).apply(lambda x: pick_best(x)).reset_index(drop=True)
else:
    df_long2["_rowid"] = range(len(df_long2))
    df_best = df_long2.groupby("_rowid", as_index=False).apply(lambda x: pick_best(x)).reset_index(drop=True).drop(columns=["_rowid"])

df_best.to_csv(OUT_BEST, index=False, encoding="utf-8-sig")

# ----- audit -----
audit = {
    "input_rows": int(len(df)),
    "cid_columns_detected": cid_cols,
    "long_rows": int(len(df_long2)),
    "unique_cids": int(len(unique_cids)),
    "resolved_smiles_long": int(df_long2["CanonicalSMILES"].notna().sum()),
    "resolved_smiles_best": int(df_best["CanonicalSMILES"].notna().sum()),
    "http_stats": http_stats,
    "rdkit_available": RD_AVAILABLE,
    "outputs": {
        "long_csv": os.path.basename(OUT_LONG),
        "best_csv": os.path.basename(OUT_BEST),
        "cache": os.path.basename(CACHE_PROPS),
    }
}
with open(OUT_AUDIT, "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)

print("=== DONE: robust CID -> SMILES mapping ===")
print(json.dumps(audit, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# Training data merge(FAERS-priority version)
# 1) AID_1671200_labeled_MIX_POSstrict_NEGbase.csv minimumcolumn(CID, label)
# 2) ror_with_smiles_best.csv  signal==1 
# 3) 1)2)merge(SMILES**FAERS**AIDdrop)
#    ※ AIDlabel.only.
# Output:
#   - aid_minimal.csv / faers_signal1.csv / merged_for_model.csv / merged_for_model_audit.json
# ============================================================

import os, json
import pandas as pd
from typing import Optional

ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
AID_MAIN = f"{ROOT}/AID_1671200_labeled_MIX_POSstrict_NEGbase.csv"
FAERS_BEST = f"{ROOT}/ror_with_smiles_best.csv"

OUT_AID_MIN = f"{ROOT}/aid_minimal.csv"
OUT_FAERS_SIG = f"{ROOT}/faers_signal1.csv"
OUT_MERGED = f"{ROOT}/merged_for_model.csv"
OUT_AUDIT = f"{ROOT}/merged_for_model_audit.json"

def choose_smiles_column(df: pd.DataFrame) -> Optional[str]:
    for c in ["PUBCHEM_EXT_DATASOURCE_SMILES","CanonicalSMILES","SMILES","IsomericSMILES"]:
        if c in df.columns: return c
    return None

def normalize_columns(df: pd.DataFrame, cid_col: str, smiles_col: Optional[str], carry_cols=()):
    out = pd.DataFrame(index=df.index)
    out["PUBCHEM_CID"] = pd.to_numeric(df[cid_col], errors="coerce").astype("Int64")
    if smiles_col and smiles_col in df.columns:
        out["SMILES"] = df[smiles_col].astype(str).str.strip()
        out.loc[out["SMILES"].isin(["", "nan", "None"]), "SMILES"] = pd.NA
    for c in carry_cols:
        if c in df.columns and c not in out.columns:
            out[c] = df[c]
    # only()
    out = out.drop_duplicates().reset_index(drop=True)
    return out

# 1) AID: minimumcolumn(CID/SMILES/label_herg_inhibit)
df_aid = pd.read_csv(AID_MAIN, low_memory=False)
if "PUBCHEM_CID" not in df_aid.columns:
    raise RuntimeError("PUBCHEM_CID column is missing in the AID file.")

if "label_herg_inhibit" not in df_aid.columns:
    raise RuntimeError("label_herg_inhibit column is missing in the AID file.")

aid_smiles_col = choose_smiles_column(df_aid)
df_aid_min = normalize_columns(
    df_aid,
    cid_col="PUBCHEM_CID",
    smiles_col=aid_smiles_col,
    carry_cols=["label_herg_inhibit"]
)
df_aid_min["source"] = "AID1671200"

# (audit)
aid_label_counts = df_aid_min["label_herg_inhibit"].value_counts(dropna=False).to_dict()

# 2) FAERS: signal==1 only
df_faers = pd.read_csv(FAERS_BEST, low_memory=False)
if "signal" not in df_faers.columns:
    raise RuntimeError("signal column is missing in the FAERS file.")

df_faers_sig = df_faers.loc[df_faers["signal"] == 1].copy()

faers_cid_col = None
for c in ["PUBCHEM_CID","cid","CID"]:
    if c in df_faers_sig.columns:
        faers_cid_col = c; break
if faers_cid_col is None:
    raise RuntimeError("CID column (PUBCHEM_CID/cid/CID) was not found in the FAERS file.")

faers_smiles_col = choose_smiles_column(df_faers_sig)
df_faers_sig_min = normalize_columns(
    df_faers_sig,
    cid_col=faers_cid_col,
    smiles_col=faers_smiles_col,
    carry_cols=["signal","drug"]
)
df_faers_sig_min["source"] = "FAERS_signal"

# 3) : SMILES**FAERS**AIDdrop(AIDlabel)
faers_smiles_set = set(df_faers_sig_min["SMILES"].dropna().unique()) if "SMILES" in df_faers_sig_min.columns else set()

if "SMILES" in df_aid_min.columns and faers_smiles_set:
    # FAERSSMILESAID -> FAERS
    before = len(df_aid_min)
    df_aid_min_nodup = df_aid_min.loc[~df_aid_min["SMILES"].isin(faers_smiles_set)].copy()
    removed_from_aid_by_smiles_dup = before - len(df_aid_min_nodup)
else:
    df_aid_min_nodup = df_aid_min.copy()
    removed_from_aid_by_smiles_dup = 0

#  FAERS -> AID()
df_merged = pd.concat([df_faers_sig_min, df_aid_min_nodup], ignore_index=True)
# only(CIDSMILEScolumn)
df_merged = df_merged.drop_duplicates().reset_index(drop=True)

# Output
df_aid_min.to_csv(OUT_AID_MIN, index=False, encoding="utf-8-sig")
df_faers_sig_min.to_csv(OUT_FAERS_SIG, index=False, encoding="utf-8-sig")
df_merged.to_csv(OUT_MERGED, index=False, encoding="utf-8-sig")

# audit
audit = {
    "paths": {
        "aid_main": AID_MAIN,
        "faers_best": FAERS_BEST,
        "aid_minimal": OUT_AID_MIN,
        "faers_signal1": OUT_FAERS_SIG,
        "merged_for_model": OUT_MERGED
    },
    "aid": {
        "rows_in": int(len(df_aid)),
        "rows_minimal": int(len(df_aid_min)),
        "label_counts_in_minimal": aid_label_counts,
        "has_smiles": ("SMILES" in df_aid_min.columns),
        "unique_smiles": int(df_aid_min["SMILES"].nunique()) if "SMILES" in df_aid_min.columns else None,
        "removed_by_faers_smiles_dup": int(removed_from_aid_by_smiles_dup)
    },
    "faers": {
        "rows_in": int(len(df_faers)),
        "rows_signal1": int(len(df_faers_sig)),
        "rows_signal1_minimal": int(len(df_faers_sig_min)),
        "has_smiles": ("SMILES" in df_faers_sig_min.columns),
        "unique_smiles": int(df_faers_sig_min["SMILES"].nunique()) if "SMILES" in df_faers_sig_min.columns else None
    },
    "merged": {
        "rows": int(len(df_merged)),
        "unique_smiles": int(df_merged["SMILES"].nunique()) if "SMILES" in df_merged.columns else None,
        "class_counts_if_present": (
            df_merged["label_herg_inhibit"].value_counts(dropna=False).to_dict()
            if "label_herg_inhibit" in df_merged.columns else None
        )
    },
    "policy": {
        "dedup_key": "SMILES",
        "priority": "FAERS_over_AID_on_SMILES_dup",
        "note": "SMILESRules(only)"
    }
}
with open(OUT_AUDIT, "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)

print("=== DONE: merge (FAERS priority on SMILES duplicates) ===")
print(json.dumps(audit, ensure_ascii=False, indent=2))


In [ ]:
# ============================================================
# merged_for_model.csv add the final label column to
# Rules:
#  1) (label_herg_inhibit == 1)  (signal == 1) -> label = 1
#  2) otherwise, if (label_herg_inhibit == 0) -> label = 0
#  3) () NaN
# Input/Output:
#  Input : /content/drive/MyDrive/Chemoinfo_QT/merged_for_model.csv
#  Output : /content/drive/MyDrive/Chemoinfo_QT/merged_for_model_with_label.csv
#        /content/drive/MyDrive/Chemoinfo_QT/merged_for_model_label_freq.csv
#        /content/drive/MyDrive/Chemoinfo_QT/merged_for_model_label_audit.json
# ============================================================

import pandas as pd, numpy as np, json, os

ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
IN_CSV  = f"{ROOT}/merged_for_model.csv"
OUT_CSV = f"{ROOT}/merged_for_model_with_label.csv"

df = pd.read_csv(IN_CSV, low_memory=False)

def find_col(name):
    for c in df.columns:
        if c.strip().lower()==name.lower():
            return c
    return None

c_herg = find_col("label_herg_inhibit")
c_sig  = find_col("signal")
c_src  = find_col("source")

herg = pd.Series(pd.NA, index=df.index, dtype="Int64")
if c_herg:
    herg = pd.to_numeric(df[c_herg], errors="coerce").astype("Int64")

sig = pd.Series(pd.NA, index=df.index, dtype="Int64")
if c_sig:
    sig_clean = df[c_sig].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})
    sig = pd.to_numeric(sig_clean, errors="coerce").astype("Int64")

src = df[c_src].astype(str) if c_src else pd.Series(["UNKNOWN"]*len(df))

# FAERS correction: source==FAERS_signal and missing signal -> set to 1
faers_mask = src.str.strip().str.lower().eq("faers_signal")
sig = sig.mask(faers_mask & sig.isna(), 1)

label = pd.Series(pd.NA, index=df.index, dtype="Int64")
label[(herg==1) | (sig==1)] = 1
label[label.isna() & (herg==0)] = 0
df["label"] = label

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

# Check(: 1=805, 0=6581, NaN=0)
freq = df["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count")
print("=== label frequency ===")
print(freq.to_string(index=False))
print("Saved:", OUT_CSV)

In [ ]:
# ============================================================
# merged_for_model_with_label.csv deduplicate by SMILES
# Representative-row priority: AID1671200 > label(1>0>NaN) > CID present > original row order
# Additional: after deduplication df_kept  label 
# Output:
#   - merged_for_model_dedup_smiles.csv
#   - merged_for_model_dedup_audit.json
#   - (optional)removed_rows_by_smiles.csv
# ============================================================

import os, json
import pandas as pd
import numpy as np

ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
IN_CSV   = f"{ROOT}/merged_for_model_with_label.csv"
OUT_CSV  = f"{ROOT}/merged_for_model_dedup_smiles.csv"
OUT_RM   = f"{ROOT}/removed_rows_by_smiles.csv"
OUT_AUD  = f"{ROOT}/merged_for_model_dedup_audit.json"

df = pd.read_csv(IN_CSV, low_memory=False)

# --- Required-column check(SMILES handle naming variants) ---
if "SMILES" not in df.columns:
    cand = [c for c in df.columns if c.strip().lower() == "smiles"]
    if cand:
        df = df.rename(columns={cand[0]: "SMILES"})
    else:
        raise RuntimeError("SMILES column was not found. Please check for naming variants such as 'Smiles'.")

# Cleanup
if "label" in df.columns:
    df["label"] = pd.to_numeric(df["label"], errors="coerce").astype("Int64")
else:
    df["label"] = pd.NA

src_col = "source" if "source" in df.columns else None
cid_col = "PUBCHEM_CID" if "PUBCHEM_CID" in df.columns else None

# --- Deduplication key(strip leading and trailing whitespace；empty string/nan/None NaN treated)---
smiles_norm = df["SMILES"].astype(str).str.strip()
smiles_norm = smiles_norm.where(~smiles_norm.isin(["", "nan", "None"]), np.nan)
df["_smiles_norm"] = smiles_norm

df["_rowid"] = np.arange(len(df))  # for stable sorting

# Source priority
def source_rank(s):
    if src_col is None:
        return 1
    v = str(s).strip().lower()
    return 0 if v == "aid1671200" else 1

df["_src_rank"] = df[src_col].map(source_rank) if src_col else 1

# label(1>0>NaN)
def label_rank(x):
    if pd.isna(x): return 2
    return 0 if int(x)==1 else (1 if int(x)==0 else 2)

df["_lbl_rank"] = df["label"].map(label_rank)

# CID availability(prefer present CIDs)
df["_cid_rank"] = 0
if cid_col:
    df["_cid_rank"] = df[cid_col].notna().astype(int).rsub(1)  # CID present->0, ->1

# Only non-NaN SMILES values are considered for duplicate detection.
mask_dup_candidate = df["_smiles_norm"].notna()

# Representative-row selection: after sorting by priority, _smiles_norm  drop_duplicates
sort_keys = ["_src_rank", "_lbl_rank", "_cid_rank", "_rowid"]
df_sorted = df.sort_values(sort_keys)

rep_rowids = (
    df_sorted[mask_dup_candidate]
    .drop_duplicates(subset=["_smiles_norm"])["_rowid"]
    .tolist()
)

rep_flag = pd.Series(False, index=df.index)
rep_flag.loc[rep_rowids] = True
# SMILES NaN exclude from duplicate detection and keep all rows
rep_flag = rep_flag | (~mask_dup_candidate)

df_kept = df.loc[rep_flag].drop(columns=["_rowid","_src_rank","_lbl_rank","_cid_rank"])
df_removed = df.loc[~rep_flag].drop(columns=["_rowid","_src_rank","_lbl_rank","_cid_rank"])

# --- Output ---
df_kept.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
if len(df_removed):
    df_removed.to_csv(OUT_RM, index=False, encoding="utf-8-sig")

# --- audit ---
aud = {
    "input_rows": int(len(df)),
    "kept_rows": int(len(df_kept)),
    "removed_rows": int(len(df_removed)),
    "unique_smiles_kept": int(df_kept["SMILES"].nunique(dropna=True)),
    "duplicates_removed_by_smiles": int(df[mask_dup_candidate].shape[0] - df_kept["SMILES"].notna().sum()),
    "rules": {
        "group_by": "SMILES (trimonly; )",
        "priority": ["source=='AID1671200'", "label: 1 > 0 > NaN", "PUBCHEM_CID present", "original order"],
        "smiles_nan": "()"
    },
    "outputs": {
        "dedup_csv": OUT_CSV,
        "removed_rows_csv": OUT_RM if len(df_removed) else None
    }
}
with open(OUT_AUD, "w", encoding="utf-8") as f:
    json.dump(aud, f, ensure_ascii=False, indent=2)

# --- Additional: label(df_kept ) ---
n_total = len(df_kept)
freq = df_kept["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count")
freq["label"] = freq["label"].astype(object)
freq.loc[freq["label"].isna(), "label"] = "NaN"
freq["percent"] = (freq["count"] / max(n_total, 1) * 100).round(2)

print("=== DONE: SMILES de-duplicated ===")
print(json.dumps(aud, ensure_ascii=False, indent=2))
print("\n=== Label frequency after dedup (kept rows) ===")
print(f"Total kept rows: {n_total}")
print(freq.to_string(index=False))
print("\nSaved:", OUT_CSV)


In [ ]:
# ============================================================
# merged_for_model_dedup_smiles.csv parent-structure consolidation of(merging salts, hydrates, and solvates)
# - RDKit use MolStandardize to generate Parent_SMILES / Parent_Key
# - -> label aggregation( any 1 -> 1, else any 0 -> 0, else NaN )
# - Representative-row priority starts with AID > label(1>0>NaN) > CID present > original row order
# - Sample_Weight  source assign from the source(create if missing)
# Output:
#   1) merged_for_model_consolidated.csv     …… one representative row per parent structure
#   2) merged_for_model_children_expanded.csv …… ->child-to-parent expansion table(audit)
#   3) merged_for_model_parent_audit.json    …… audit
# Dependencies:
#   pip install rdkit-pypi  ()
# ============================================================

import os, json, csv, re
import pandas as pd
import numpy as np

ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
IN_CSV = f"{ROOT}/merged_for_model_dedup_smiles.csv"

OUT_BASE   = IN_CSV.rsplit(".",1)[0].replace("_dedup_smiles","")
OUT_CONS   = f"{OUT_BASE}_consolidated.csv"
OUT_CHILD  = f"{OUT_BASE}_children_expanded.csv"
OUT_AUDIT  = f"{OUT_BASE}_parent_audit.json"

SOURCE_WEIGHT = {"aid1671200": 1.0, "faers_signal": 1.0}
DEFAULT_WEIGHT = 1.0

from rdkit import RDLogger
# RDKit (suppress INFO/DEBUG messages)
RDLogger.DisableLog('rdApp.info')
RDLogger.DisableLog('rdApp.debug')

# MolStandardize ("Running Normalizer" )
try:
    from rdkit.Chem.MolStandardize import rdMolStandardize as STD
    # , as a precautionso the fallback also applies globally:
    RDLogger.DisableLog('rdMolStandardize')
except Exception:
    pass

# ---------- JSON ----------
def py_scalar(x):
    """numpy/NAPython"""
    if x is None or x is pd.NA:
        return None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        # NaN  None 
        return None if np.isnan(x) else float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x

def counts_dict(series: pd.Series):
    """value_counts  JSON dict (NaN 'NaN')"""
    vc = series.value_counts(dropna=False)
    out = {}
    for k, v in vc.items():
        if pd.isna(k):
            key = "NaN"
        elif isinstance(k, (np.integer,)):
            key = int(k)
        elif isinstance(k, (np.floating,)):
            key = float(k)
        else:
            key = str(k)
        out[key] = int(py_scalar(v))
    return out

def describe_dict(series: pd.Series):
    """Return a JSON-safe version of describe()."""
    desc = series.describe()
    return {str(k): py_scalar(v) for k, v in desc.items()}

# ---------- RDKit ----------
try:
    from rdkit import Chem
    from rdkit.Chem.MolStandardize import rdMolStandardize as STD
    RDKit_OK = True
    RD_ERR = ""
except Exception as e:
    RDKit_OK = False
    RD_ERR = str(e)

def _canon_smiles(mol):
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)
    except Exception:
        return None

def _parent_mol_from_smiles(smi: str):
    if not smi or str(smi).strip().lower() in {"nan","none"}:
        return None
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    try:
        mol = STD.Normalizer().normalize(mol)
        mol = STD.Reionizer().reionize(mol)
        mol = STD.Uncharger().uncharge(mol)
        mol = STD.LargestFragmentChooser().choose(mol)
    except Exception:
        pass
    return mol

def parent_key_and_smiles(smi: str):
    if not RDKit_OK:
        return (None, None)
    mol = _parent_mol_from_smiles(smi)
    if mol is None:
        return (None, None)
    p_smi = _canon_smiles(mol)
    if not p_smi:
        return (None, None)
    return (p_smi, p_smi)



# ---------- utilities ----------
def find_col(df, candidates):
    low = {str(c).strip().lower(): c for c in df.columns}
    for name in candidates:
        c = low.get(name.strip().lower())
        if c: return c
    return None

def label_rank(v):
    if pd.isna(v): return 2
    try:
        iv = int(v)
        return 0 if iv==1 else (1 if iv==0 else 2)
    except Exception:
        return 2

def source_rank(v):
    if pd.isna(v): return 1
    key = str(v).strip().lower()
    return 0 if key=="aid1671200" else 1

# ---------- Run ----------
df = pd.read_csv(IN_CSV, low_memory=False)

col_smi = find_col(df, ["SMILES"])
if col_smi is None:
    raise RuntimeError("SMILES column was not found. Please check for naming variants such as 'Smiles'.")
col_label = find_col(df, ["label"])
col_cid   = find_col(df, ["PUBCHEM_CID","CID"])
col_src   = find_col(df, ["source"])

# Type cleanup
df["label"] = pd.to_numeric(df[col_label], errors="coerce").astype("Int64") if col_label else pd.Series(pd.NA, index=df.index, dtype="Int64")
df["PUBCHEM_CID"] = pd.to_numeric(df[col_cid], errors="coerce") if col_cid else np.nan
df["source"] = df[col_src].astype(str) if col_src else "UNKNOWN"

# Sample_Weight
col_sw_in = find_col(df, ["Sample_Weight","SampleWeight","sampleweight"])
if col_sw_in:
    df["Sample_Weight"] = pd.to_numeric(df[col_sw_in], errors="coerce").fillna(DEFAULT_WEIGHT).astype(float).clip(lower=0.0)
else:
    def _sw_from_source(s):
        key = str(s).strip().lower()
        return float(SOURCE_WEIGHT.get(key, DEFAULT_WEIGHT))
    df["Sample_Weight"] = df["source"].map(_sw_from_source).astype(float)

# parent keys
if RDKit_OK:
    parent_keys, parent_smiles = [], []
    for i, smi in enumerate(df[col_smi].astype(str).tolist()):
        pkey, psm = parent_key_and_smiles(smi)
        parent_keys.append(pkey)
        parent_smiles.append(psm)
        if (i+1) % 500 == 0:
            print(f"[prog] parentizing {i+1}/{len(df)}")
    df["Parent_Key"] = parent_keys
    df["Parent_SMILES"] = parent_smiles
else:
    print(f"[WARN] RDKit not available: {RD_ERR}")
    df["Parent_Key"] = df[col_smi]
    df["Parent_SMILES"] = df[col_smi]

# -> 
child_rows = []
for _, r in df.iterrows():
    child_rows.append({
        "Child_SMILES": r[col_smi],
        "Child_CID": r["PUBCHEM_CID"] if pd.notna(r["PUBCHEM_CID"]) else "",
        "Child_label": r["label"],
        "Child_source": r["source"],
        "Sample_Weight": float(r["Sample_Weight"]),
        "Parent_Key": r["Parent_Key"],
        "Parent_SMILES": r["Parent_SMILES"],
    })
child_df = pd.DataFrame(child_rows)

# aggregation
df["_rowid"] = np.arange(len(df))
df["_src_rank"] = df["source"].map(source_rank)
df["_lbl_rank"] = df["label"].map(label_rank)
df["_cid_rank"] = df["PUBCHEM_CID"].notna().astype(int).rsub(1)  # CID present->0, ->1

cons_rows = []

for pkey, g in df.groupby("Parent_Key", dropna=True):
    parent_smiles = g["Parent_SMILES"].iloc[0] if "Parent_SMILES" in g.columns else pkey

    # labelaggregation:  any 1 -> 1, else any 0 -> 0, else NaN
    vals = g["label"].dropna().astype(int)
    if (vals == 1).any():
        parent_label = 1
    elif (vals == 0).any():
        parent_label = 0
    else:
        parent_label = pd.NA

    parent_sw = float(g["Sample_Weight"].fillna(DEFAULT_WEIGHT).max())

    g_sorted = g.sort_values(["_src_rank","_lbl_rank","_cid_rank","_rowid"])
    rep = g_sorted.iloc[0]

    cons_rows.append({
        "SMILES": parent_smiles,
        "Parent_Key": pkey,
        "PUBCHEM_CID": int(rep["PUBCHEM_CID"]) if pd.notna(rep["PUBCHEM_CID"]) else "",
        "source_rep": rep["source"],
        "label": parent_label,
        "Sample_Weight": parent_sw,
        "Children_N": int(len(g)),
    })

cons_df = pd.DataFrame(cons_rows).sort_values(["Children_N"], ascending=False)

# Save
child_cols = ["Child_SMILES","Child_CID","Child_label","Child_source","Sample_Weight","Parent_Key","Parent_SMILES"]
child_df[child_cols].to_csv(OUT_CHILD, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)
cons_df.to_csv(OUT_CONS, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

# audit(JSON)
aud = {
    "input": {
        "path": IN_CSV,
        "rows": int(len(df)),
        "unique_smiles_in": int(df[col_smi].nunique(dropna=True)),
        "has_rdkit": bool(RDKit_OK),
        "rdkit_error": RD_ERR if not RDKit_OK else ""
    },
    "parentization": {
        "parents_found": int(cons_df.shape[0]),
        "unique_parent_smiles": int(cons_df["SMILES"].nunique(dropna=True)),
    },
    "labels": {
        "child_label_counts": counts_dict(df["label"]),
        "parent_label_counts": counts_dict(cons_df["label"]),
    },
    "weights": {
        "source_weight_policy": {str(k): float(v) for k,v in SOURCE_WEIGHT.items()},
        "default_weight": float(DEFAULT_WEIGHT),
        "parent_weight_summary": describe_dict(cons_df["Sample_Weight"])
    },
    "outputs": {
        "consolidated_csv": OUT_CONS,
        "children_expanded_csv": OUT_CHILD,
        "audit_json": OUT_AUDIT
    }
}
with open(OUT_AUDIT, "w", encoding="utf-8") as f:
    json.dump(aud, f, ensure_ascii=False, indent=2)

print("[OK] Saved after consolidation at the parent-structure level")
print(json.dumps(aud, ensure_ascii=False, indent=2))
print(f"- CONS:  {OUT_CONS}")
print(f"- CHILD: {OUT_CHILD}")
print(f"- AUDIT: {OUT_AUDIT}")


# ========= label frequency for merged_for_model_consolidated.csv =========
import pandas as pd
import numpy as np

ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
PATH = f"{ROOT}/merged_for_model_consolidated.csv"

df = pd.read_csv(PATH, low_memory=False)

# labelcolumn(- )
label_col = None
for c in df.columns:
    if c.strip().lower() == "label":
        label_col = c
        break
if label_col is None:
    raise RuntimeError("label column not found. label columnCheck.")

# Normalization(Int64 )
lab = pd.to_numeric(df[label_col], errors="coerce").astype("Int64")
n = len(lab)
n_nan = int(lab.isna().sum())

# (NaN)
freq = lab.value_counts(dropna=False).rename_axis("label").reset_index(name="count")
#  NaN column
freq["label"] = freq["label"].astype(object)
freq.loc[freq["label"].isna(), "label"] = "NaN"
freq["percent"] = (freq["count"] / max(n, 1) * 100).round(2)

print("=== label frequency (merged_for_model_consolidated.csv) ===")
print(f"Path       : {PATH}")
print(f"Total rows : {n}  |  NaN labels: {n_nan}")
print(freq.to_string(index=False))

In [ ]:
# =========================================
# PyG data generation(d.x + d.g + d.r + cid + sample_weight)
# Input: /content/drive/MyDrive/Chemoinfo_QT/merged_for_model_consolidated.csv
# - required columns: SMILES, label(0/1)
# - optionalcolumn: Sample_Weight or SampleWeight( 1.0), CID/PubChem_CID( -1)
# Specification:
# - label  NaN Trainingdrop
# - positive (label==1) only SMILES (AUG_POS_K times; default=5)
# - RDKit descriptors (gfeat7, rdesc10)  d.g, d.r 
# - save the sample weight to d.sample_weight( 1.0)
# - save CID to d.cid( -1, CID->PubChem_CID )
# Output:
#   /content/drive/MyDrive/Chemoinfo_QT/data_graph_with_smiles.pt
#   /content/drive/MyDrive/Chemoinfo_QT/data_graph_with_smiles_index.csv
# =========================================

import os, json, random
import numpy as np
import pandas as pd

# ---- RDKit (suppress Normalizer and related output)----
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.info')
RDLogger.DisableLog('rdApp.debug')

from rdkit import Chem
from rdkit.Chem import Descriptors, rdchem

# ---- PyTorch / PyG ----
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

# ========== Configuration ==========
ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
DATA_PATH = f"{ROOT}/merged_for_model_consolidated.csv"
SAVE_PATH = f"{ROOT}/data_graph_with_smiles.pt"
INDEX_CSV = f"{ROOT}/data_graph_with_smiles_index.csv"

SEED = 42
AUG_POS_K = 5  # augmentation multiplier for positive SMILES
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ========== utilities ==========
def find_col(df, names):
    low = {str(c).strip().lower(): c for c in df.columns}
    for n in names:
        c = low.get(n.lower())
        if c: return c
    return None

# --- molecular features(d.g, d.r) ---
def gfeat7(mol):
    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.HeavyAtomCount(mol),
        Descriptors.RingCount(mol),
    ]

def rdesc10(mol):
    return [
        Descriptors.FpDensityMorgan1(mol),
        Descriptors.FpDensityMorgan2(mol),
        Descriptors.FpDensityMorgan3(mol),
        Descriptors.NumAliphaticRings(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumValenceElectrons(mol),
        Descriptors.BalabanJ(mol),
        Descriptors.BertzCT(mol),
        Descriptors.FractionCSP3(mol),
    ]

# --- -  ---
def atom_f(atom: rdchem.Atom):
    return torch.tensor([
        atom.GetAtomicNum(),
        atom.GetTotalDegree(),
        atom.GetFormalCharge(),
        float(atom.GetIsAromatic()),
        atom.GetTotalNumHs(includeNeighbors=True),
    ], dtype=torch.float)

def bond_f(bond: rdchem.Bond):
    return torch.tensor([
        bond.GetBondTypeAsDouble(),
        float(bond.GetIsConjugated()),
        float(bond.IsInRing()),
        int(bond.GetStereo()),
        bond.GetBeginAtom().GetAtomicNum(),
        bond.GetEndAtom().GetAtomicNum(),
    ], dtype=torch.float)

def aug_smiles(sm, k=5):
    m = Chem.MolFromSmiles(sm)
    if m is None: return []
    return [Chem.MolToSmiles(m, doRandom=True) for _ in range(k)]

# ========== Run ==========
df0 = pd.read_csv(DATA_PATH, low_memory=False)

col_smiles = find_col(df0, ["SMILES"])
col_label  = find_col(df0, ["label"])
if not col_smiles or not col_label:
    raise ValueError("Required columns SMILES and label were not found.")

col_sw  = find_col(df0, ["Sample_Weight","SampleWeight","sample_weight","sampleweight"])
col_cid = find_col(df0, ["CID","PubChem_CID"])

# Cleanup
df0["SMILES"] = df0[col_smiles].astype(str)
df0["label"]  = pd.to_numeric(df0[col_label], errors="coerce").astype("Int64")
if col_sw:
    df0["SampleWeight"] = pd.to_numeric(df0[col_sw], errors="coerce").fillna(1.0).astype(float).clip(lower=0.0)
else:
    df0["SampleWeight"] = 1.0

if col_cid:
    # : CID -> PubChem_CID
    if col_cid.lower() == "cid":
        df0["CID"] = pd.to_numeric(df0[col_cid], errors="coerce")
    else:
        # handle the case where only one of the two columns is available
        df0["CID"] = pd.to_numeric(df0[col_cid], errors="coerce")
else:
    df0["CID"] = np.nan

# NaN labeldrop, 0/1only
before = len(df0)
df0 = df0[df0["label"].isin([0,1])]
df0["label"] = df0["label"].astype(int)
after = len(df0)
print(f"[INFO] rows: {before} -> {after} (drop NaN/non-binary labels)")

# CID fallback( -1)
def cid_to_int(x):
    if pd.isna(x): return -1
    try:
        return int(x)
    except Exception:
        return -1
df0["_cid_int"] = df0["CID"].apply(cid_to_int)

# Label distribution()
vc = df0["label"].value_counts().to_dict()
print(f"[CHECK] label counts (orig): {vc}")

# augment only positive samples
rows = []
for sm, y, sw, cidv in zip(df0["SMILES"], df0["label"], df0["SampleWeight"], df0["_cid_int"]):
    sm = "" if pd.isna(sm) else str(sm)
    if not sm: continue
    if y == 1 and AUG_POS_K > 0:
        aug = aug_smiles(sm, k=AUG_POS_K)
        for s in [sm] + aug:
            rows.append((s, y, float(sw), int(cidv)))
    else:
        rows.append((sm, y, float(sw), int(cidv)))

df = pd.DataFrame(rows, columns=["SMILES","Label","SampleWeight","CID"])
print(f"[OK] Augmented size: {len(df)} (original {len(df0)})")

# RDKit  & descriptors
mols, keep, g7_raw, r10_raw = [], [], [], []
for i, (sm, _, _, _) in enumerate(df.itertuples(index=False)):
    m = Chem.MolFromSmiles(sm)
    if m is None or m.GetNumAtoms() == 0:
        continue
    mols.append(m); keep.append(i)
    g7_raw.append(gfeat7(m)); r10_raw.append(rdesc10(m))
df = df.iloc[keep].reset_index(drop=True)

# 
scaler_g = StandardScaler().fit(g7_raw)
scaler_r = StandardScaler().fit(r10_raw)
g7_scaled = scaler_g.transform(g7_raw)
r10_scaled = scaler_r.transform(r10_raw)

# graph conversion
data_list = []
for row_idx, m in enumerate(mols):
    y  = int(df.loc[row_idx, "Label"])
    sw = float(df.loc[row_idx, "SampleWeight"])
    cid_val = int(df.loc[row_idx, "CID"])

    if m.GetNumBonds() == 0:
        continue

    # 
    x = torch.stack([atom_f(a) for a in m.GetAtoms()])  # [N,5]

    # ()
    src, dst, eattr = [], [], []
    for b in m.GetBonds():
        i = b.GetBeginAtomIdx(); j = b.GetEndAtomIdx()
        f = bond_f(b)
        src += [i, j]; dst += [j, i]
        eattr += [f, f]
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr  = torch.stack(eattr)

    d = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([y], dtype=torch.float32)
    )
    d.g = torch.tensor(g7_scaled[row_idx], dtype=torch.float32).unsqueeze(0)   # [1,7]
    d.r = torch.tensor(r10_scaled[row_idx], dtype=torch.float32).unsqueeze(0)  # [1,10]
    d.sample_weight = torch.tensor([sw], dtype=torch.float32)
    d.smiles = df.loc[row_idx, "SMILES"]
    d.cid = cid_val  # -1 or int
    data_list.append(d)

print(f"[OK] graphs: {len(data_list)}")
lbl = [int(d.y.item()) for d in data_list]
print(f"[CHECK] label counts (graphs): {pd.Series(lbl).value_counts().to_dict()}")

# Save
os.makedirs(ROOT, exist_ok=True)
torch.save(data_list, SAVE_PATH)
print(f"[OK] saved: {SAVE_PATH}")

# 
idx_df = pd.DataFrame({
    "SMILES":        [d.smiles for d in data_list],
    "Label":         [int(d.y.item()) for d in data_list],
    "SampleWeight":  [float(d.sample_weight.item()) for d in data_list],
    "CID":           [int(getattr(d, "cid", -1)) for d in data_list],
})
idx_df.to_csv(INDEX_CSV, index=False)
print(f"[OK] index saved: {INDEX_CSV}")

# (optional)pos_weight 
n_pos = (idx_df["Label"]==1).sum()
n_neg = (idx_df["Label"]==0).sum()
pos_w = (n_neg / max(n_pos,1)) if n_pos>0 else None
print(f"[INFO] pos_weight (neg/pos): {round(pos_w,4) if pos_w else 'N/A'}")

In [ ]:
# --- Save(compressed with metadata) ---
import os, json, joblib, sklearn
ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
os.makedirs(ROOT, exist_ok=True)

meta = {
    "scaler_g": {"type": "StandardScaler", "features": "gfeat7(7)"},
    "scaler_r": {"type": "StandardScaler", "features": "rdesc10(10)"},
    "sklearn_version": sklearn.__version__
}

joblib.dump(scaler_g, f"{ROOT}/scaler_g.joblib", compress=3)
joblib.dump(scaler_r, f"{ROOT}/scaler_r.joblib", compress=3)
with open(f"{ROOT}/scaler_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("[OK] Saved:", f"{ROOT}/scaler_g.joblib", f"{ROOT}/scaler_r.joblib", "and scaler_meta.json")

In [ ]:
# =============================================================
# 4Model comparison (GINE / GATv2 / NNConv / TransformerConv)
# d.x + d.g + d.r , edge_attr 
# Input: /content/drive/MyDrive/Chemoinfo_QT/data_graph_with_smiles.pt
# Output: /content/drive/MyDrive/Chemoinfo_QT/results_gnn_dx_dg_dr_cv_edgeall/
# Notes:
#  -  d.sample_weight (d.w )
#  - pos_weight()(reduction='none' + Normalization)
#  - priority-CID fixing is optional(PRIORITY_CIDS )
# =============================================================

import os, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.serialization import add_safe_globals
from torch_geometric.nn import GINEConv, GATv2Conv, NNConv, TransformerConv, AttentionalAggregation
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import pandas as pd

# ====== Configuration ======
ROOT   = "/content/drive/MyDrive/Chemoinfo_QT"
IN_PYGDATA = f"{ROOT}/data_graph_with_smiles.pt"
SAVE_BASE  = f"{ROOT}/results_gnn_dx_dg_dr_cv_edgeall"
os.makedirs(SAVE_BASE, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
KFOLD  = 3
BATCH  = 64
HID    = 256
LR     = 3e-4
PAT    = 5
TMAX   = 5
CLIP   = 5
WEIGHT_DECAY = 1e-4
ARCHES = ["GINE", "GATv2", "NNConv", "Transformer"]
for arch in ARCHES:
    os.makedirs(os.path.join(SAVE_BASE, arch), exist_ok=True)

# ★optional: CIDs to always include in training()
PRIORITY_CIDS = set()

# ====== Reproducibility ======
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ====== Data loading(PyTorch 2.6) ======
def load_pyg_data(path):
    try:
        try:
            from torch_geometric.data.data import DataEdgeAttr
            add_safe_globals([DataEdgeAttr])
        except Exception:
            pass
        obj = torch.load(path)
        return obj
    except Exception as e:
        print(f"[WARN] Safe loading failed({e}).falling back to weights_only=False.")
        obj = torch.load(path, weights_only=False)
        return obj

obj = load_pyg_data(IN_PYGDATA)
if isinstance(obj, list):
    data_list = obj
elif isinstance(obj, dict) and "data_list" in obj:
    data_list = obj["data_list"]
else:
    data_list = list(obj)

if len(data_list) == 0:
    raise RuntimeError("data_list  is empty.Please confirm that the upstream .pt generation step completed successfully.")

sample = data_list[0]
req = ["x","edge_index","edge_attr","g","r","y"]
missing = [k for k in req if not hasattr(sample, k)]
if missing:
    raise RuntimeError(f"Required attributes are missing from the Data object: {missing}")

def infer_feat_dim(t):
    if t is None: return 0
    if t.dim() == 1: return t.numel()
    if t.dim() in (2,3): return t.size(-1)
    raise RuntimeError(f"Unknown tensor dimensionality: dim={t.dim()}")

in_dim   = sample.x.size(-1)
edge_dim = sample.edge_attr.size(-1)
g_dim    = infer_feat_dim(sample.g)
r_dim    = infer_feat_dim(sample.r)

labels = [int(d.y.item()) for d in data_list]
pos = sum(labels); neg = len(labels) - pos
print(f"[INFO] Graphs: {len(data_list)} | x_dim={in_dim} | edge_dim={edge_dim} | g_dim={g_dim} | r_dim={r_dim}")
print(f"[INFO] Label balance: pos={pos} neg={neg} (pos_ratio={pos/len(labels):.3f})")

cids = [getattr(d, "cid", -1) for d in data_list]
has_cid_attr = any(getattr(d, "cid", None) is not None for d in data_list)
priority_idx = [i for i, c in enumerate(cids) if c in PRIORITY_CIDS] if has_cid_attr else []
priority_idx_set = set(priority_idx)
print(f"[INFO] Priority CIDs present: {len(priority_idx)} / {len(PRIORITY_CIDS)}")

# ====== Convolution blocks ======
def make_edge_mlp(in_attr_dim, out_channels, in_channels):
    hidden = max(64, min(256, in_attr_dim * 8))
    return nn.Sequential(
        nn.Linear(in_attr_dim, hidden), nn.ReLU(),
        nn.Linear(hidden, in_channels * out_channels)
    )

def get_conv(name, c_in, c_out, edge_dim):
    if name == "GINE":
        return GINEConv(nn.Sequential(
            nn.Linear(c_in, c_out), nn.ReLU(), nn.Linear(c_out, c_out)
        ), edge_dim=edge_dim)
    if name == "GATv2":
        return GATv2Conv(c_in, c_out, heads=4, concat=False,
                         edge_dim=edge_dim, fill_value='mean')
    if name == "NNConv":
        edge_nn = make_edge_mlp(edge_dim, c_out, c_in)
        return NNConv(c_in, c_out, edge_nn, aggr='mean')
    if name == "Transformer":
        return TransformerConv(c_in, c_out, heads=4, concat=False,
                               edge_dim=edge_dim, dropout=0.0, beta=False)
    raise ValueError(f"unknown arch: {name}")

# ====== Shared model definitions ======
class Net(nn.Module):
    def __init__(self, arch, in_dim, edge_dim, g_dim, r_dim, hid=256, p_drop=0.2):
        super().__init__()
        self.arch = arch
        self.c1 = get_conv(arch, in_dim, hid, edge_dim)
        self.c2 = get_conv(arch, hid, hid, edge_dim)
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(hid, 1))
        self.drop = nn.Dropout(p_drop)
        self.fc   = nn.Linear(hid + g_dim + r_dim, 1)

    def forward(self, d):
        x = F.relu(self.c1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.c2(x, d.edge_index, d.edge_attr))
        x = self.drop(x)
        x = self.pool(x, d.batch)

        g = d.g; r = d.r
        if g.dim() == 3: g = g.squeeze(1)
        elif g.dim() == 1: g = g.view(d.num_graphs, -1)
        if r.dim() == 3: r = r.squeeze(1)
        elif r.dim() == 1: r = r.view(d.num_graphs, -1)

        z = torch.cat([x.float(), g.float(), r.float()], dim=1)
        return self.fc(z).view(-1)

# ====== Training function(pos_weight + sample_weight) ======
def train_fold(net, tr_loader, va_loader, pos_weight, save_path):
    bce = nn.BCEWithLogitsLoss(reduction='none')
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, TMAX)

    best_auc, wait, best_state = 0.0, 0, None
    max_epochs = 200

    for ep in range(1, max_epochs + 1):
        net.train()
        for bt in tr_loader:
            bt = bt.to(DEVICE)
            opt.zero_grad()

            logits = net(bt)
            y = bt.y.float().view(-1)

            loss_vec = bce(logits, y)

            # class-imbalance correction
            cls_w = torch.where(
                y > 0.5,
                torch.tensor(pos_weight, device=DEVICE, dtype=loss_vec.dtype),
                torch.tensor(1.0,       device=DEVICE, dtype=loss_vec.dtype),
            )

            # (d.w  1, d.sample_weight already stored upstream)
            w1 = getattr(bt, "w", torch.ones_like(y)).view(-1)
            w2 = getattr(bt, "sample_weight", torch.ones_like(y)).view(-1)
            samp_w = (w1 * w2).to(loss_vec.dtype)

            loss = (loss_vec * cls_w * samp_w).sum() / (samp_w.sum() + 1e-12)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), CLIP)
            opt.step()
        sch.step()

        # Validation
        net.eval(); ys, ps = [], []
        with torch.no_grad():
            for bt in va_loader:
                bt = bt.to(DEVICE)
                ys += bt.y.cpu().tolist()
                ps += torch.sigmoid(net(bt)).cpu().tolist()
        auc = roc_auc_score(ys, ps)

        if auc > best_auc:
            best_auc, wait = auc, 0
            best_state = net.state_dict()
            torch.save(best_state, save_path)
        else:
            wait += 1
            if wait >= PAT:
                break

    if best_state is not None:
        net.load_state_dict(best_state)
    return net

# ====== K-Fold Training- Evaluation ======
skf = StratifiedKFold(n_splits=KFOLD, shuffle=True, random_state=SEED)
all_summary_rows = []

def enforce_priority_in_train(tr_idx, va_idx):
    if len(PRIORITY_CIDS) == 0 or len(priority_idx_set) == 0:
        return tr_idx, va_idx, 0, 0
    tr_list, va_list = tr_idx.tolist(), va_idx.tolist()
    pri_in_val = [i for i in va_list if i in priority_idx_set]
    moved_to_train = 0; swapped = 0
    for i_pri in pri_in_val:
        if i_pri in va_list:
            va_list.remove(i_pri)
        if i_pri not in tr_list:
            tr_list.append(i_pri); moved_to_train += 1
        y_pri = labels[i_pri]
        cand = None
        for j in tr_list:
            if j in priority_idx_set: continue
            if labels[j] == y_pri and j not in va_list:
                cand = j; break
        if cand is not None:
            tr_list.remove(cand); va_list.append(cand); swapped += 1
    return np.array(sorted(set(tr_list))), np.array(sorted(set(va_list))), moved_to_train, swapped

for arch in ARCHES:
    print(f"\n================ {arch} (d.x + d.g + d.r + edge_attr) ================")
    fold_rows = []
    arch_dir = os.path.join(SAVE_BASE, arch)

    for f, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        tr_idx, va_idx, moved_cnt, swap_cnt = enforce_priority_in_train(tr_idx, va_idx)
        if moved_cnt > 0:
            print(f"[Fold {f}] moved priority from val->train: {moved_cnt} | swapped: {swap_cnt}")

        tr_ds = [data_list[i] for i in tr_idx]
        va_ds = [data_list[i] for i in va_idx]

        ytr = torch.cat([d.y.view(-1) for d in tr_ds]).float()
        neg = (ytr == 0).sum().item(); posc = (ytr == 1).sum().item()
        w = min((neg / max(posc, 1.0)), 10.0)

        tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,
                               generator=torch.Generator().manual_seed(SEED))
        va_loader = DataLoader(va_ds, batch_size=BATCH, shuffle=False)

        net = Net(arch, in_dim, edge_dim, g_dim, r_dim, hid=HID).to(DEVICE)
        save_model_path = os.path.join(arch_dir, f"fold{f}.pt")
        net = train_fold(net, tr_loader, va_loader, pos_weight=w, save_path=save_model_path)

        # within-fold validation scores
        net.eval(); ys, ps = [], []
        with torch.no_grad():
            for bt in va_loader:
                bt = bt.to(DEVICE)
                ys += bt.y.cpu().tolist()
                ps += torch.sigmoid(net(bt)).cpu().tolist()

        y_pred = [int(p >= 0.5) for p in ps]
        metrics = {
            "Fold": f,
            "ROC-AUC": roc_auc_score(ys, ps),
            "PR-AUC": average_precision_score(ys, ps),
            "Precision": precision_score(ys, y_pred, zero_division=0),
            "Recall": recall_score(ys, y_pred, zero_division=0),
            "F1-Score": f1_score(ys, y_pred, zero_division=0),
            "Val_Size": len(va_idx),
            "Train_Size": len(tr_idx),
        }
        print("Fold{}: ROC {:.4f} | PR {:.4f} | P {:.4f} | R {:.4f} | F1 {:.4f} | Train {} | Val {}".format(
            f, metrics["ROC-AUC"], metrics["PR-AUC"], metrics["Precision"], metrics["Recall"], metrics["F1-Score"],
            metrics["Train_Size"], metrics["Val_Size"]
        ))
        fold_rows.append(metrics)

    df_metrics = pd.DataFrame(fold_rows)
    df_metrics.to_csv(os.path.join(arch_dir, "metrics_foldwise.csv"), index=False)

    mean_row = df_metrics[["ROC-AUC","PR-AUC","Precision","Recall","F1-Score"]].mean()
    std_row  = df_metrics[["ROC-AUC","PR-AUC","Precision","Recall","F1-Score"]].std()

    df_summary = pd.DataFrame({
        "Metric": ["ROC-AUC","PR-AUC","Precision","Recall","F1-Score"],
        "Mean":   [mean_row[m] for m in mean_row.index],
        "SD":     [std_row[m]  for m in std_row.index],
    })
    df_summary.to_csv(os.path.join(arch_dir, "metrics_summary.csv"), index=False)

    print("\n[{}] mean ± SD".format(arch))
    for col in ["ROC-AUC", "PR-AUC", "Precision", "Recall", "F1-Score"]:
        print(f"{col}: {mean_row[col]:.4f} ± {std_row[col]:.4f}")

    all_summary_rows.append({
        "Arch": arch,
        **{f"{k}_Mean": mean_row[k] for k in mean_row.index},
        **{f"{k}_SD":   std_row[k]  for k in std_row.index},
    })

pd.DataFrame(all_summary_rows).to_csv(
    os.path.join(SAVE_BASE, "metrics_all_summary.csv"), index=False
)

print(f"\n[OK] done: Save path {SAVE_BASE}")

In [ ]:
# =============================================================
# TransformerConv + GATv2 ensemble(d.x + d.g + d.r, edge_attr)
# 5-fold CV / training curves- ROC- PRcurve / logit-weighted ensemble
# Input: /content/drive/MyDrive/Chemoinfo_QT/data_graph_with_smiles.pt
# Output: /content/drive/MyDrive/Chemoinfo_QT/results_ens_trans_gatv2_5fold/
# Notes:
#  - CIDfixed(use standard stratified K-fold only)
#  -  pos_weight × per-sample weight(d.w, sample_weight )
#  - apply weights to validation metrics(APPLY_WEIGHT_IN_VAL=True)※set to False if preferred
# =============================================================

import os, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.serialization import add_safe_globals
from torch_geometric.nn import TransformerConv, GATv2Conv, AttentionalAggregation
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    roc_curve, precision_recall_curve, auc
)
import pandas as pd
import matplotlib.pyplot as plt

# ====== Configuration ======
ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
DATA_PATH = f"{ROOT}/data_graph_with_smiles.pt"
SAVE_DIR  = f"{ROOT}/results_ens_trans_gatv2_5fold"
PLOT_DIR  = os.path.join(SAVE_DIR, "plots")
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# (use in order if present in the Data object)
WEIGHT_ATTR_CANDIDATES = ["sample_weight","weight","w","SampleWeight","Weight"]
APPLY_WEIGHT_IN_VAL = True  # Validation

# Training hyperparameters
KFOLD = 5
BATCH_TRAIN = 64
BATCH_VAL   = 128
HID = 256
LR  = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10
TMAX = 10
CLIP = 5
MAX_EPOCHS = 200

# ===== Reproducibility =====
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ===== PyG data loading(PyTorch 2.6 ) =====
def load_pyg_list(path):
    try:
        from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
        add_safe_globals([DataEdgeAttr, DataTensorAttr])
    except Exception as e:
        print(f"[WARN] add_safe_globals failed: {e}")
    try:
        obj = torch.load(path)  # weights_only=True Default
    except Exception as e:
        print(f"[WARN] Safe loading failed({e}).retrying with weights_only=False.")
        obj = torch.load(path, weights_only=False)

    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    try:
        return list(obj)
    except Exception as e:
        raise RuntimeError(f": {type(obj)}; failed to convert to list: {e}")

data_list = load_pyg_list(DATA_PATH)
if len(data_list) == 0:
    raise RuntimeError("data_list  is empty.")

# ===== label =====
labels = [int(d.y.item()) for d in data_list]
print(f"[INFO] Graphs: {len(data_list)} | pos={sum(labels)} neg={len(labels)-sum(labels)}")

# =====  =====
sample = data_list[0]
def infer_feat_dim(t):
    if t is None: return 0
    if t.dim() == 1: return t.numel()
    if t.dim() in (2,3): return t.size(-1)
    raise RuntimeError(f"Unknown tensor dimensionality: dim={t.dim()}")

X_DIM = sample.x.size(-1)
E_DIM = sample.edge_attr.size(-1)
G_DIM = infer_feat_dim(sample.g)
R_DIM = infer_feat_dim(sample.r)
print(f"[INFO] Dims: x={X_DIM}, edge={E_DIM}, g={G_DIM}, r={R_DIM}")

# =====  =====
class Transformer_DX_DG_DR(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = TransformerConv(X_DIM, HID, heads=4, concat=False, edge_dim=E_DIM, dropout=0.0, beta=False)
        self.c2 = TransformerConv(HID,  HID, heads=4, concat=False, edge_dim=E_DIM, dropout=0.0, beta=False)
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(HID + G_DIM + R_DIM, 1)

    def forward(self, d):
        x = F.relu(self.c1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.c2(x, d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim() == 3 else d.g
        r = d.r.squeeze(1) if d.r.dim() == 3 else d.r
        return self.fc(torch.cat([x, g, r], dim=1)).view(-1)  # logits

class GATv2_DX_DG_DR(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = GATv2Conv(X_DIM, HID, heads=4, concat=False, edge_dim=E_DIM, fill_value='mean')
        self.c2 = GATv2Conv(HID,  HID, heads=4, concat=False, edge_dim=E_DIM, fill_value='mean')
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(HID + G_DIM + R_DIM, 1)

    def forward(self, d):
        x = F.relu(self.c1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.c2(x, d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim() == 3 else d.g
        r = d.r.squeeze(1) if d.r.dim() == 3 else d.r
        return self.fc(torch.cat([x, g, r], dim=1)).view(-1)  # logits

# ===== utilities =====
def _to_1d_weight(t):
    if t is None: return None
    if t.dim() == 2 and t.size(1) == 1:
        return t.view(-1)
    return t.view(-1)

def get_batch_weights(batch, device=None):
    for cand in WEIGHT_ATTR_CANDIDATES:
        if hasattr(batch, cand):
            w = getattr(batch, cand)
            w = _to_1d_weight(w).to(device or batch.y.device).float()
            return w, cand
    n = batch.y.numel()
    return torch.ones(n, dtype=torch.float, device=device or batch.y.device), "CONST1"

def plot_curve(xs, ys, title, xlabel, ylabel, save_path):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.figure(); plt.plot(xs, ys)
    plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.tight_layout(); plt.savefig(save_path); plt.close()

def save_roc_pr_curves(y_true, y_prob, title_prefix, out_dir, sample_weight=None):
    os.makedirs(out_dir, exist_ok=True)
    fpr, tpr, _ = roc_curve(y_true, y_prob, sample_weight=sample_weight)
    roc_auc = auc(fpr, tpr)
    plt.figure(); plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--', linewidth=1)
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"{title_prefix} ROC")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{title_prefix}_ROC.png")); plt.close()

    prec, rec, _ = precision_recall_curve(y_true, y_prob, sample_weight=sample_weight)
    ap = average_precision_score(y_true, y_prob, sample_weight=sample_weight)
    plt.figure(); plt.plot(rec, prec, label=f"AP={ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"{title_prefix} PR")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{title_prefix}_PR.png")); plt.close()

def best_f1_threshold(y_true, y_prob, sample_weight=None):
    prec, rec, thr = precision_recall_curve(y_true, y_prob, sample_weight=sample_weight)
    thr = np.append(thr, 1.0)
    f1s = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    i = np.nanargmax(f1s)
    return float(thr[i]), float(f1s[i])

# ===== Training function(pos_weight × per-sample weight) =====
def train_fold(model, tr_loader, va_loader, fold, save_path, model_name, apply_weight_in_val=True):
    #  pos_weight  recordscompute(maximum10)
    y_list, w_list = [], []
    for d in tr_loader.dataset:
        y_list.append(int(d.y.item()))
        w = None
        for cand in WEIGHT_ATTR_CANDIDATES:
            if hasattr(d, cand):
                a = getattr(d, cand)
                w = float(a.item()) if torch.is_tensor(a) else float(a)
                break
        if w is None: w = 1.0
        w_list.append(w)
    ytr = torch.tensor(y_list, dtype=torch.float)
    wtr = torch.tensor(w_list, dtype=torch.float)
    pos_w = (wtr[ytr==0].sum() / torch.clamp(wtr[ytr==1].sum(), min=1e-8)).item()
    pos_w = float(min(max(pos_w, 1.0), 10.0))
    print(f"[INFO] fold{fold} pos_weight(eff)={pos_w:.3f}")

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], device=DEVICE), reduction='none')
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TMAX)

    history = {"epoch": [], "train_loss": [], "val_auc": [], "val_prauc": []}
    best_auc, best_state, wait = 0.0, None, 0

    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); epoch_losses = []
        for batch in tr_loader:
            batch = batch.to(DEVICE)
            logits = model(batch)
            loss_vec = criterion(logits, batch.y.float())
            sw, used_attr = get_batch_weights(batch, DEVICE)
            loss = (loss_vec * sw).sum() / (sw.sum() + 1e-8)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()
            epoch_losses.append(loss.item())
        sch.step()

        # Validation
        model.eval(); ys, ps, ws = [], [], []
        with torch.no_grad():
            for batch in va_loader:
                batch = batch.to(DEVICE)
                ys += batch.y.cpu().tolist()
                ps += torch.sigmoid(model(batch)).cpu().tolist()
                sw, _ = get_batch_weights(batch, DEVICE)
                ws += sw.detach().cpu().tolist()

        if apply_weight_in_val:
            auc_val = roc_auc_score(ys, ps, sample_weight=ws)
            pr_val  = average_precision_score(ys, ps, sample_weight=ws)
        else:
            auc_val = roc_auc_score(ys, ps)
            pr_val  = average_precision_score(ys, ps)

        history["epoch"].append(ep)
        history["train_loss"].append(float(np.mean(epoch_losses)))
        history["val_auc"].append(float(auc_val))
        history["val_prauc"].append(float(pr_val))

        if auc_val > best_auc:
            best_auc, wait = auc_val, 0
            best_state = model.state_dict()
            torch.save(best_state, save_path)
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"[OK] EarlyStopped [{model_name}] (fold{fold}) at epoch {ep}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

# ===== Cross-validation(5-fold)standard stratified: split the full sample set directly =====
skf = StratifiedKFold(n_splits=KFOLD, shuffle=True, random_state=SEED)
metrics_tr_list, metrics_ga_list, metrics_ens_list = [], [], []
fold_counter = 0

for tr_idx, va_idx in skf.split(np.zeros(len(labels)), labels):
    tr_data = [data_list[i] for i in tr_idx]
    va_data = [data_list[i] for i in va_idx]

    # ()
    def _stats(indices):
        ys = np.array([int(data_list[i].y.item()) for i in indices])
        ws = []
        for i in indices:
            w = None
            for cand in WEIGHT_ATTR_CANDIDATES:
                if hasattr(data_list[i], cand):
                    a = getattr(data_list[i], cand)
                    w = float(a.item()) if torch.is_tensor(a) else float(a)
                    break
            if w is None: w = 1.0
            ws.append(w)
        ws = np.array(ws, dtype=float)
        return ys.mean(), ws.mean(), ws.min(), ws.max()

    y_bar_tr, w_bar_tr, w_min_tr, w_max_tr = _stats(tr_idx)
    y_bar_va, w_bar_va, w_min_va, w_max_va = _stats(va_idx)

    print(f"\n[FOLD] Fold {fold_counter} | train={len(tr_idx)} val={len(va_idx)} | "
          f"train_w(mean/min/max)={w_bar_tr:.2f}/{w_min_tr:.2f}/{w_max_tr:.2f} | "
          f"val_w(mean/min/max)={w_bar_va:.2f}/{w_min_va:.2f}/{w_max_va:.2f}")

    tr_loader = DataLoader(tr_data, batch_size=BATCH_TRAIN, shuffle=True, pin_memory=True)
    va_loader = DataLoader(va_data, batch_size=BATCH_VAL,   shuffle=False, pin_memory=True)

    fold_plot_dir = os.path.join(PLOT_DIR, f"fold{fold_counter}")
    os.makedirs(fold_plot_dir, exist_ok=True)

    # ---- Transformer ----
    model_tr = Transformer_DX_DG_DR().to(DEVICE)
    path_tr  = f"{SAVE_DIR}/transformer_fold{fold_counter}.pt"
    model_tr, hist_tr = train_fold(model_tr, tr_loader, va_loader, fold_counter, path_tr, "Transformer",
                                   apply_weight_in_val=APPLY_WEIGHT_IN_VAL)
    pd.DataFrame(hist_tr).to_csv(f"{SAVE_DIR}/history_transformer_fold{fold_counter}.csv", index=False)
    plot_curve(hist_tr["epoch"], hist_tr["train_loss"], f"Transformer Fold{fold_counter} Train Loss", "Epoch", "Loss",
               os.path.join(fold_plot_dir, "Transformer_Loss.png"))
    plot_curve(hist_tr["epoch"], hist_tr["val_auc"], f"Transformer Fold{fold_counter} Val ROC-AUC", "Epoch", "ROC-AUC",
               os.path.join(fold_plot_dir, "Transformer_ValAUC.png"))
    plot_curve(hist_tr["epoch"], hist_tr["val_prauc"], f"Transformer Fold{fold_counter} Val PR-AUC", "Epoch", "PR-AUC",
               os.path.join(fold_plot_dir, "Transformer_ValPRAUC.png"))

    # ---- GATv2 ----
    model_ga = GATv2_DX_DG_DR().to(DEVICE)
    path_ga  = f"{SAVE_DIR}/gatv2_fold{fold_counter}.pt"
    model_ga, hist_ga = train_fold(model_ga, tr_loader, va_loader, fold_counter, path_ga, "GATv2",
                                   apply_weight_in_val=APPLY_WEIGHT_IN_VAL)
    pd.DataFrame(hist_ga).to_csv(f"{SAVE_DIR}/history_gatv2_fold{fold_counter}.csv", index=False)
    plot_curve(hist_ga["epoch"], hist_ga["train_loss"], f"GATv2 Fold{fold_counter} Train Loss", "Epoch", "Loss",
               os.path.join(fold_plot_dir, "GATv2_Loss.png"))
    plot_curve(hist_ga["epoch"], hist_ga["val_auc"], f"GATv2 Fold{fold_counter} Val ROC-AUC", "Epoch", "ROC-AUC",
               os.path.join(fold_plot_dir, "GATv2_ValAUC.png"))
    plot_curve(hist_ga["epoch"], hist_ga["val_prauc"], f"GATv2 Fold{fold_counter} Val PR-AUC", "Epoch", "PR-AUC",
               os.path.join(fold_plot_dir, "GATv2_ValPRAUC.png"))

    # ---- Evaluation:  & ensemble(PR-AUC)----
    def eval_logits_and_weights(m):
        m.eval(); ys, logits, ws = [], [], []
        with torch.no_grad():
            for batch in va_loader:
                batch = batch.to(DEVICE)
                ys += batch.y.cpu().tolist()
                logits += m(batch).cpu().tolist()  # 
                sw, _ = get_batch_weights(batch, DEVICE)
                ws += sw.detach().cpu().tolist()
        return np.array(ys), np.array(logits), np.array(ws, dtype=float)

    ys_tr, lg_tr, w_va = eval_logits_and_weights(model_tr)
    ys_ga, lg_ga, w_va2 = eval_logits_and_weights(model_ga)
    assert np.allclose(ys_tr, ys_ga)
    ys = ys_tr.tolist()

    ps_tr = torch.sigmoid(torch.tensor(lg_tr)).numpy()
    ps_ga = torch.sigmoid(torch.tensor(lg_ga)).numpy()
    if APPLY_WEIGHT_IN_VAL:
        pr_tr = average_precision_score(ys, ps_tr, sample_weight=w_va)
        pr_ga = average_precision_score(ys, ps_ga, sample_weight=w_va)
    else:
        pr_tr = average_precision_score(ys, ps_tr)
        pr_ga = average_precision_score(ys, ps_ga)

    a_tr = pr_tr / (pr_tr + pr_ga + 1e-12)
    a_ga = pr_ga / (pr_tr + pr_ga + 1e-12)

    lg_ens = a_tr * lg_tr + a_ga * lg_ga
    ps_ens = torch.sigmoid(torch.tensor(lg_ens)).numpy()

    def pack_metrics(y, p, name, sw=None):
        thr_opt, f1_opt = best_f1_threshold(y, p, sample_weight=sw if APPLY_WEIGHT_IN_VAL else None)
        y_pred05 = (p >= 0.5).astype(int)
        if APPLY_WEIGHT_IN_VAL:
            roc = roc_auc_score(y, p, sample_weight=sw)
            pr  = average_precision_score(y, p, sample_weight=sw)
        else:
            roc = roc_auc_score(y, p)
            pr  = average_precision_score(y, p)
        return {
            "Fold": fold_counter, "Model": name,
            "ROC-AUC": roc, "PR-AUC": pr,
            "Precision@0.5": precision_score(y, y_pred05, zero_division=0),
            "Recall@0.5":    recall_score(y, y_pred05, zero_division=0),
            "F1@0.5":        f1_score(y, y_pred05, zero_division=0),
            "BestThr(F1)": thr_opt, "F1@BestThr": f1_opt
        }

    mt = pack_metrics(ys, ps_tr,  "Transformer", w_va)
    mg = pack_metrics(ys, ps_ga,  "GATv2",       w_va)
    me = pack_metrics(ys, ps_ens, "Ensemble",    w_va)
    print(f"[Transformer] {mt}")
    print(f"[GATv2]       {mg}")
    print(f"[Ensemble]    {me} (alpha={a_tr:.3f}/{a_ga:.3f})")

    fold_dir = os.path.join(fold_plot_dir)
    save_roc_pr_curves(ys, ps_tr,  "Transformer", fold_dir, sample_weight=w_va if APPLY_WEIGHT_IN_VAL else None)
    save_roc_pr_curves(ys, ps_ga,  "GATv2",       fold_dir, sample_weight=w_va if APPLY_WEIGHT_IN_VAL else None)
    save_roc_pr_curves(ys, ps_ens, "Ensemble",    fold_dir, sample_weight=w_va if APPLY_WEIGHT_IN_VAL else None)

    metrics_tr_list.append(mt)
    metrics_ga_list.append(mg)
    metrics_ens_list.append(me)
    fold_counter += 1

# ===== Save results =====
def save_metrics(rows, csv_name):
    df = pd.DataFrame(rows)
    mean = df.select_dtypes(include=[np.number]).mean()
    std  = df.select_dtypes(include=[np.number]).std()
    path = f"{SAVE_DIR}/{csv_name}"
    df.to_csv(path, index=False)
    print(f"[FILE] Saved: {path}")
    print("[METRIC] :")
    for k in ["ROC-AUC","PR-AUC","Precision@0.5","Recall@0.5","F1@0.5","BestThr(F1)","F1@BestThr"]:
        if k in mean.index:
            print(f"  {k}: {float(mean[k]):.4f} ± {float(std[k]):.4f}")

save_metrics(metrics_tr_list,  "transformer_cv_results.csv")
save_metrics(metrics_ga_list,  "gatv2_cv_results.csv")
save_metrics(metrics_ens_list, "ensemble_cv_results.csv")

# Save(CID)
summary = {
    "data_path": DATA_PATH,
    "save_dir": SAVE_DIR,
    "apply_weight_in_val": APPLY_WEIGHT_IN_VAL,
    "kfold": KFOLD,
    "seed": SEED,
    "hparams": {
        "HID": HID, "LR": LR, "WEIGHT_DECAY": WEIGHT_DECAY,
        "PATIENCE": PATIENCE, "TMAX": TMAX, "CLIP": CLIP,
        "BATCH_TRAIN": BATCH_TRAIN, "BATCH_VAL": BATCH_VAL
    }
}
with open(os.path.join(SAVE_DIR, "run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n[OK] done: Save path {SAVE_DIR}")
print(f"[INFO] Weights applied in validation: {APPLY_WEIGHT_IN_VAL}")

In [ ]:
# ================================================
# TransformerConv + GATv2 (d.x + d.g + d.r, edge_attr)
# Scaffold 5-Fold CV + Ensemble (logit-weighted by Val PR-AUC)
# training curves(Loss/Val AUC/Val PR-AUC)＆ROC/PRplots with file export
# Notes:
#  - NNConv -> GATv2 replace
#  - CID/priority scaffoldfixed(KFoldonly)
#  - Training(BCEWithLogitsLoss(reduction='none') -> )
#  - pos_weight effective weighted ratio(neg_sum/pos_sum, capped at 10)
#  - ValidationROC/PR- plots- ensemble(Val PR-AUC), 
#  - : ["sample_weight","weight","w","SampleWeight","Weight"](1.0)
# ================================================
import os, random
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.serialization import add_safe_globals
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, GATv2Conv, AttentionalAggregation

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, f1_score,
    auc, roc_curve, average_precision_score
)
from collections import defaultdict

# ----- Configuration -----
ROOT      = "/content/drive/MyDrive/Chemoinfo_QT"
PT_PATH   = f"{ROOT}/data_graph_with_smiles.pt"
SAVE_DIR  = f"{ROOT}/results_ens_trans_gatv2_scaffold5fold"
PLOT_DIR  = os.path.join(SAVE_DIR, "plots")
NUM_FOLDS = 5
BATCH_TR  = 64
BATCH_TE  = 128
LR        = 3e-4
WD        = 1e-4
TMAX      = 20
PATIENCE  = 15
MAX_EPOCH = 200
CLIP      = 5
SEED      = 42

# SampleWeight Configuration
WEIGHT_ATTR_CANDIDATES = ["sample_weight","weight","w","SampleWeight","Weight"]
APPLY_WEIGHT_IN_VAL  = True   # Validation AUC/AP/plots/ensemble
APPLY_WEIGHT_IN_TEST = False  # test evaluation True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(PLOT_DIR, exist_ok=True)

# ----- utilities -----
def _ensure_dir(p): os.makedirs(p, exist_ok=True)
def plot_curve(xs, ys, title, xlabel, ylabel, save_path):
    plt.figure(); plt.plot(xs, ys); plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.tight_layout(); plt.savefig(save_path); plt.close()

def save_roc_pr_curves(y_true, y_prob, title_prefix, out_dir, sample_weight=None):
    _ensure_dir(out_dir)
    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_prob, sample_weight=sample_weight)
    roc_auc = auc(fpr, tpr)
    plt.figure(); plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--', linewidth=1)
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"{title_prefix} ROC")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{title_prefix}_ROC.png")); plt.close()
    # PR
    prec, rec, _ = precision_recall_curve(y_true, y_prob, sample_weight=sample_weight)
    ap = average_precision_score(y_true, y_prob, sample_weight=sample_weight)
    plt.figure(); plt.plot(rec, prec, label=f"AP={ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"{title_prefix} PR")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{title_prefix}_PR.png")); plt.close()

def _get_weight_from_data(d):
    for nm in WEIGHT_ATTR_CANDIDATES:
        if hasattr(d, nm):
            v = getattr(d, nm)
            return float(v.item()) if torch.is_tensor(v) else float(v)
    return 1.0

def _to_1d(w):
    if w is None: return None
    if torch.is_tensor(w) and w.dim() == 2 and w.size(1) == 1:
        return w.view(-1)
    return w.view(-1) if torch.is_tensor(w) else torch.tensor(w, dtype=torch.float32)

def get_batch_weights(batch, device=None):
    for nm in WEIGHT_ATTR_CANDIDATES:
        if hasattr(batch, nm):
            w = getattr(batch, nm); w = _to_1d(w).to(device or batch.y.device).float()
            return w, nm
    return torch.ones(batch.y.numel(), device=device or batch.y.device, dtype=torch.float32), "CONST1"

# ----- PyG data loading(PyTorch 2.6 ) -----
def load_pyg_list(path):
    try:
        from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
        from torch_geometric.data.storage import GlobalStorage
        add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
    except Exception as e:
        print(f"[WARN] add_safe_globals failed: {e}")
    try:
        obj = torch.load(path)  # default: weights_only=True
    except Exception as e:
        print(f"[WARN] Safe loading failed({e}).retrying with weights_only=False.")
        obj = torch.load(path, weights_only=False)
    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    try:
        return list(obj)
    except Exception as e:
        raise RuntimeError(f": {type(obj)}; failed to convert to list: {e}")

data = load_pyg_list(PT_PATH)
if len(data) == 0: raise RuntimeError("data  is empty.")
print(f"[OK] Data loading: {len(data)} records")
print("Label distribution:", pd.Series([int(d.y) for d in data]).value_counts().to_dict())

# ----- Murcko scaffold compute -> KFold -----
def smiles_to_scaffold(smi, fallback):
    m = Chem.MolFromSmiles(smi)
    if m:
        try:
            return MurckoScaffold.MurckoScaffoldSmiles(mol=m)
        except Exception:
            pass
    return fallback

scaf_of = []
for i, d in enumerate(data):
    smi = getattr(d, "smiles", "")
    scaf_of.append(smiles_to_scaffold(smi, f"_NOSCAF_{i}"))

scaf_to_indices = defaultdict(list)
for i, sc in enumerate(scaf_of):
    scaf_to_indices[sc].append(i)

def scaffold_split_kfold(k=5, seed=42):
    scafs = list(scaf_to_indices.keys())
    random.seed(seed); random.shuffle(scafs)
    folds = [[] for _ in range(k)]
    for i, sc in enumerate(scafs):
        folds[i % k].extend(scaf_to_indices[sc])
    return folds

folds = scaffold_split_kfold(k=NUM_FOLDS, seed=SEED)

# ----- Input dimensions -----
sample = data[0]
def infer_dim(t):
    if t.dim() == 1: return t.numel()
    if t.dim() in (2,3): return t.size(-1)
    raise RuntimeError(f": dim={t.dim()}")

X_DIM = sample.x.size(-1)
E_DIM = sample.edge_attr.size(-1)
G_DIM = infer_dim(sample.g)
R_DIM = infer_dim(sample.r)
HID   = 256
print(f"[INFO] Dims: x={X_DIM}, edge={E_DIM}, g={G_DIM}, r={R_DIM}")

# -----  -----
class Transformer_DX_DG_DR(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = TransformerConv(X_DIM, HID, heads=4, concat=False, edge_dim=E_DIM, dropout=0.0, beta=False)
        self.c2 = TransformerConv(HID,  HID, heads=4, concat=False, edge_dim=E_DIM, dropout=0.0, beta=False)
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(HID + G_DIM + R_DIM, 1)
    def forward(self, d):
        x = F.relu(self.c1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.c2(x, d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim() == 3 else d.g
        r = d.r.squeeze(1) if d.r.dim() == 3 else d.r
        return self.fc(torch.cat([x, g, r], dim=1)).view(-1)  # logits

class GATv2_DX_DG_DR(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = GATv2Conv(X_DIM, HID, heads=4, concat=False, edge_dim=E_DIM, fill_value='mean')
        self.c2 = GATv2Conv(HID,  HID, heads=4, concat=False, edge_dim=E_DIM, fill_value='mean')
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(HID + G_DIM + R_DIM, 1)
    def forward(self, d):
        x = F.relu(self.c1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.c2(x, d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim() == 3 else d.g
        r = d.r.squeeze(1) if d.r.dim() == 3 else d.r
        return self.fc(torch.cat([x, g, r], dim=1)).view(-1)  # logits

# ----- Training/Evaluationutilities -----
def pos_weight_from(dataset):
    ys = torch.tensor([float(d.y.item()) for d in dataset])
    ws = torch.tensor([_get_weight_from_data(d) for d in dataset], dtype=torch.float32)
    pos_sum = ws[ys == 1].sum(); neg_sum = ws[ys == 0].sum()
    pw = (neg_sum / torch.clamp(pos_sum, min=1e-8)).item()
    return torch.tensor([min(max(pw, 1.0), 10.0)], dtype=torch.float32)

@torch.no_grad()
def eval_logits(model, ds, batch_size=BATCH_TE):
    model.eval(); ys, lg, ws = [], [], []
    for bt in DataLoader(ds, batch_size=batch_size, shuffle=False):
        bt = bt.to(DEVICE)
        ys.append(bt.y.detach().cpu())
        lg.append(model(bt).detach().cpu())
        w, _ = get_batch_weights(bt, DEVICE)
        ws.append(w.detach().cpu())
    y  = torch.cat(ys).numpy()
    l  = torch.cat(lg).numpy()
    sw = torch.cat(ws).numpy()
    return y, l, sw

def metrics_from(y, p, thr=0.5, sample_weight=None):
    precision, recall, _ = precision_recall_curve(y, p, sample_weight=sample_weight)
    pr_auc = auc(recall, precision)
    roc    = roc_auc_score(y, p, sample_weight=sample_weight)
    f1     = f1_score(y, (p >= thr).astype(int))  # display an unweighted value(manual weighting can be added if needed)
    return roc, pr_auc, f1

def train_one(model, train_set, val_set, save_path, tag, plot_dir):
    model.to(DEVICE)
    pw = pos_weight_from(train_set).to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw, reduction='none')
    opt  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TMAX)

    history = {"epoch": [], "train_loss": [], "val_auc": [], "val_prauc": []}
    best_auc, wait = 0.0, 0

    print(f"[{tag}] pos_weight(eff)={float(pw.item()):.3f}")

    for ep in range(1, MAX_EPOCH + 1):
        model.train(); epoch_losses = []
        for bt in DataLoader(train_set, batch_size=BATCH_TR, shuffle=True):
            bt = bt.to(DEVICE)
            logits = model(bt)
            loss_vec = crit(logits, bt.y.float())
            sw, _nm = get_batch_weights(bt, DEVICE)
            loss = (loss_vec * sw).sum() / (sw.sum() + 1e-8)

            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()
            epoch_losses.append(loss.item())
        sch.step()

        yv, lv, wv = eval_logits(model, val_set)
        pv = torch.sigmoid(torch.tensor(lv)).numpy()
        if APPLY_WEIGHT_IN_VAL:
            roc_v, pr_v, f1_v = metrics_from(yv, pv, sample_weight=wv)
        else:
            roc_v, pr_v, f1_v = metrics_from(yv, pv)

        history["epoch"].append(ep)
        history["train_loss"].append(float(np.mean(epoch_losses)))
        history["val_auc"].append(float(roc_v))
        history["val_prauc"].append(float(pr_v))

        print(f"[{tag}] Epoch {ep:03d} | ROC-AUC {roc_v:.4f} | PR-AUC {pr_v:.4f} | F1 {f1_v:.4f}")

        if roc_v > best_auc:
            best_auc, wait = roc_v, 0
            torch.save(model.state_dict(), save_path)
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"[OK] EarlyStopping [{tag}]")
                break

    model.load_state_dict(torch.load(save_path, map_location=DEVICE))

    # training curvesSave
    df_hist = pd.DataFrame(history)
    df_hist.to_csv(save_path.replace(".pt", f"_history.csv"), index=False)
    _ensure_dir(plot_dir)
    plot_curve(df_hist["epoch"], df_hist["train_loss"], f"{tag} Train Loss", "Epoch", "Loss",
               os.path.join(plot_dir, f"{tag.replace('/','_')}_Loss.png"))
    plot_curve(df_hist["epoch"], df_hist["val_auc"], f"{tag} Val ROC-AUC", "Epoch", "ROC-AUC",
               os.path.join(plot_dir, f"{tag.replace('/','_')}_ValAUC.png"))
    plot_curve(df_hist["epoch"], df_hist["val_prauc"], f"{tag} Val PR-AUC", "Epoch", "PR-AUC",
               os.path.join(plot_dir, f"{tag.replace('/','_')}_ValPRAUC.png"))

    # ValidationPR-AUC(ensemble)
    yv, lv, wv = eval_logits(model, val_set)
    pv = torch.sigmoid(torch.tensor(lv)).numpy()
    pr_v = average_precision_score(yv, pv, sample_weight=wv) if APPLY_WEIGHT_IN_VAL else average_precision_score(yv, pv)
    return model, float(pr_v)

# ----- K-Fold(Scaffold) -> Training- Evaluation -----
res_tr, res_ga, res_en = [], [], []

for k in range(NUM_FOLDS):
    # simple 3-way: test=fold k, val=fold k+1, train=remaining
    test_idx = folds[k]
    val_idx  = folds[(k + 1) % NUM_FOLDS]
    train_idx = [i for j in range(NUM_FOLDS) if j not in [k, (k + 1) % NUM_FOLDS] for i in folds[j]]

    print(f"\n[FOLD] Fold {k} | train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

    train_set = [data[i] for i in train_idx]
    val_set   = [data[i] for i in val_idx]
    test_set  = [data[i] for i in test_idx]

    fold_plot_dir = os.path.join(PLOT_DIR, f"fold{k}"); _ensure_dir(fold_plot_dir)

    # --- Training(ValidationPR-AUC) ---
    m_tr = Transformer_DX_DG_DR().to(DEVICE)
    m_ga = GATv2_DX_DG_DR().to(DEVICE)
    path_tr = f"{SAVE_DIR}/transformer_fold{k}.pt"
    path_ga = f"{SAVE_DIR}/gatv2_fold{k}.pt"

    m_tr, pr_tr_val = train_one(m_tr, train_set, val_set, path_tr, f"Transformer/F{k}", fold_plot_dir)
    m_ga, pr_ga_val = train_one(m_ga, train_set, val_set, path_ga, f"GATv2/F{k}",      fold_plot_dir)

    # foldensemble(ValidationPR-AUCNormalization)
    s = pr_tr_val + pr_ga_val + 1e-12
    a_tr, a_ga = pr_tr_val/s, pr_ga_val/s

    # --- () ---
    yt, lg_tr, wt = eval_logits(m_tr, test_set)
    _,  lg_ga, wt2 = eval_logits(m_ga, test_set)
    assert np.allclose(wt, wt2)

    ps_tr = torch.sigmoid(torch.tensor(lg_tr)).numpy()
    ps_ga = torch.sigmoid(torch.tensor(lg_ga)).numpy()
    lg_en = a_tr * lg_tr + a_ga * lg_ga
    ps_en = torch.sigmoid(torch.tensor(lg_en)).numpy()

    def met(y, p, sw=None): return metrics_from(y, p, sample_weight=sw if APPLY_WEIGHT_IN_TEST else None)
    roc_tr, pr_tr, f1_tr = met(yt, ps_tr, wt)
    roc_ga, pr_ga, f1_ga = met(yt, ps_ga, wt)
    roc_en, pr_en, f1_en = met(yt, ps_en, wt)

    print(f"[OK] Fold{k} [Transformer] ROC {roc_tr:.4f} | PR {pr_tr:.4f} | F1 {f1_tr:.4f}")
    print(f"[OK] Fold{k} [GATv2]      ROC {roc_ga:.4f} | PR {pr_ga:.4f} | F1 {f1_ga:.4f}")
    print(f"[OK] Fold{k} [Ensemble]   ROC {roc_en:.4f} | PR {pr_en:.4f} | F1 {f1_en:.4f} (alpha={a_tr:.3f}/{a_ga:.3f})")

    save_roc_pr_curves(yt, ps_tr,  f"Transformer_F{k}", fold_plot_dir, sample_weight=wt if APPLY_WEIGHT_IN_TEST else None)
    save_roc_pr_curves(yt, ps_ga,  f"GATv2_F{k}",       fold_plot_dir, sample_weight=wt if APPLY_WEIGHT_IN_TEST else None)
    save_roc_pr_curves(yt, ps_en,  f"Ensemble_F{k}",    fold_plot_dir, sample_weight=wt if APPLY_WEIGHT_IN_TEST else None)

    res_tr.append([roc_tr, pr_tr, f1_tr])
    res_ga.append([roc_ga, pr_ga, f1_ga])
    res_en.append([roc_en, pr_en, f1_en])

# -----  -----
def to_df(rows, name):
    arr = np.array(rows)
    df = pd.DataFrame({
        "Fold": [f"Fold{i}" for i in range(NUM_FOLDS)],
        "ROC-AUC": arr[:, 0], "PR-AUC": arr[:, 1], "F1": arr[:, 2]
    })
    df.index = df["Fold"]; df = df.drop(columns=["Fold"])
    df.loc["Mean"] = df.mean(numeric_only=True)
    df.loc["Std"]  = df.std(numeric_only=True)
    out = f"{SAVE_DIR}/cv_results_{name}.csv"; df.to_csv(out, index=True)
    print(f"[FILE] Saved: {out}"); print("[METRIC] :"); print(df.loc["Mean"])
    return df

df_tr = to_df(res_tr, "transformer")
df_ga = to_df(res_ga, "gatv2")
df_en = to_df(res_en, "ensemble")

rows_order = [f"Fold{i}" for i in range(NUM_FOLDS)] + ["Mean", "Std"]
def pick(df, col): return df.reindex(rows_order)[col].tolist()

df_comb = pd.DataFrame({
    "Fold": rows_order,
    "Transformer_ROC": pick(df_tr, "ROC-AUC"),
    "Transformer_PR":  pick(df_tr, "PR-AUC"),
    "Transformer_F1":  pick(df_tr, "F1"),
    "GATv2_ROC":       pick(df_ga, "ROC-AUC"),
    "GATv2_PR":        pick(df_ga, "PR-AUC"),
    "GATv2_F1":        pick(df_ga, "F1"),
    "Ensemble_ROC":    pick(df_en, "ROC-AUC"),
    "Ensemble_PR":     pick(df_en, "PR-AUC"),
    "Ensemble_F1":     pick(df_en, "F1"),
})
df_comb.to_csv(f"{SAVE_DIR}/cv_results_combined.csv", index=False)

print("\n[OK] done:", SAVE_DIR)
print(f"[INFO] Weights applied in validation: {APPLY_WEIGHT_IN_VAL} / Weights applied in test: {APPLY_WEIGHT_IN_TEST}")

In [ ]:
# ================================================
# LCO-CV (K=5) - TransformerConv & GATv2 Ensemble
#  - features: d.x + d.g + d.r (+ edge_attr)
#  - only when positives are scarce SMILES 
#  - that maximizes validation PR-AUC Ensemble select the ensemble weight alpha automatically()
#  - ★ CID(Trainingfixed)[]
#  - ★ SampleWeight support: 
#      - Training(BCEWithLogitsLoss(reduction='none') -> )
#      - pos_weight effective weighted ratio(neg_sum/pos_sum, capped at 10)
#      - validation/test ROC- PR- F1(plots- ) α* (Val PR-AUC)
#      - : ["sample_weight","weight","w","SampleWeight","Weight"]( 1.0)
# ================================================
import os, random, math, warnings as _warnings
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
from rdkit import RDLogger

from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, GATv2Conv, AttentionalAggregation
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score, auc, average_precision_score, roc_curve
from torch.serialization import add_safe_globals

# --- quiet ---
RDLogger.DisableLog('rdApp.*')
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# ----- Reproducibility -----
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ----- Configuration -----
ROOT     = "/content/drive/MyDrive/Chemoinfo_QT"
PT_PATH  = f"{ROOT}/data_graph_with_smiles.pt"
SAVE_DIR = f"{ROOT}/results_lco5_trans_gatv2_ens_posaug"
os.makedirs(SAVE_DIR, exist_ok=True)
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

K          = 5
BATCH_TR   = 64
BATCH_TE   = 128
LR         = 3e-4
WD         = 1e-4
TMAX       = 20
PATIENCE   = 10
MAX_EPOCH  = 200
CLIP       = 5.0
ALPHAS     = np.linspace(0, 1, 21)   # 0.00, 0.05, ..., 1.00

# --- Positive oversampling policy ---
AUG_TARGET_POS_RATE = 0.30
AUG_MAX_PER_POS     = 3

# --- SampleWeight Configuration ---
WEIGHT_ATTR_CANDIDATES = ["sample_weight","weight","w","SampleWeight","Weight"]
APPLY_WEIGHT_IN_VAL  = True    # apply weights in validation(α*search)
APPLY_WEIGHT_IN_TEST = False   # ( True)

# ===== Utils =====
def pr_auc_score(y, p, sample_weight=None):
    prec, rec, _ = precision_recall_curve(y, p, sample_weight=sample_weight)
    return auc(rec, prec)

def _get_weight_from_data(d):
    for nm in WEIGHT_ATTR_CANDIDATES:
        if hasattr(d, nm):
            v = getattr(d, nm)
            return float(v.item()) if torch.is_tensor(v) else float(v)
    return 1.0

def _to_1d(w):
    if torch.is_tensor(w):
        if w.dim() == 2 and w.size(1) == 1: return w.view(-1)
        return w.view(-1)
    return torch.tensor(w, dtype=torch.float32)

def get_batch_weights(batch, device=None):
    for nm in WEIGHT_ATTR_CANDIDATES:
        if hasattr(batch, nm):
            w = getattr(batch, nm)
            w = _to_1d(w).to(device or batch.y.device).float()
            return w, nm
    return torch.ones(batch.y.numel(), device=device or batch.y.device, dtype=torch.float32), "CONST1"

def pos_weight_from(dataset):
    ys = torch.tensor([float(d.y.item()) for d in dataset])
    ws = torch.tensor([_get_weight_from_data(d) for d in dataset], dtype=torch.float32)
    pos_sum = ws[ys == 1].sum()
    neg_sum = ws[ys == 0].sum()
    w = (neg_sum / torch.clamp(pos_sum, min=1e-8)).item()
    return torch.tensor([min(max(w, 1.0), 10.0)], dtype=torch.float32)

@torch.no_grad()
def eval_probs_model(m, ds, batch_size=BATCH_TE):
    if len(ds) == 0:
        return np.array([]), np.array([]), np.array([])
    m.eval(); ys, ps, ws = [], [], []
    for bt in DataLoader(ds, batch_size=batch_size, shuffle=False):
        bt = bt.to(DEVICE)
        ys.append(bt.y.detach().cpu())
        ps.append(torch.sigmoid(m(bt)).detach().cpu())
        w, _ = get_batch_weights(bt, DEVICE)
        ws.append(w.detach().cpu())
    return torch.cat(ys).numpy(), torch.cat(ps).numpy(), torch.cat(ws).numpy()

def best_alpha_for_prauc(y_val, p1, p2, alphas=ALPHAS, sample_weight=None):
    best_a, best_s = 0.5, -1.0
    for a in alphas:
        pv = a * p1 + (1 - a) * p2
        s = pr_auc_score(y_val, pv, sample_weight=sample_weight)
        if s > best_s:
            best_s, best_a = s, a
    return best_a, best_s

def infer_dim(t):
    if t.dim() == 1: return t.numel()
    if t.dim() in (2,3): return t.size(-1)
    raise RuntimeError(f"Unknown tensor dimensionality: {t.size()}")

# ----- Data loading(PyTorch 2.6 ) -----
def load_pyg_list(path):
    try:
        from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
        from torch_geometric.data.storage import GlobalStorage
        add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
    except Exception as e:
        print(f"[WARN] add_safe_globals failed: {e}")
    try:
        obj = torch.load(path)  # default weights_only=True
    except Exception as e:
        print(f"[WARN] Safe loading failed({e}).weights_only=False (assuming a trusted file).")
        obj = torch.load(path, weights_only=False)

    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    try: return list(obj)
    except Exception as e:
        raise RuntimeError(f"Unknown format: {type(obj)}; failed to convert to list: {e}")

# =====  =====
class Transformer_DX_DG_DR(nn.Module):
    def __init__(self, X_DIM, G_DIM, R_DIM, E_DIM, HID=256):
        super().__init__()
        self.c1 = TransformerConv(X_DIM, HID, heads=4, concat=False, edge_dim=E_DIM, dropout=0.0, beta=False)
        self.c2 = TransformerConv(HID,  HID, heads=4, concat=False, edge_dim=E_DIM, dropout=0.0, beta=False)
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(HID + G_DIM + R_DIM, 1)
    def forward(self, d):
        x = F.relu(self.c1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.c2(x, d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim() == 3 else d.g
        r = d.r.squeeze(1) if d.r.dim() == 3 else d.r
        return self.fc(torch.cat([x.float(), g.float(), r.float()], dim=1)).view(-1)

class GATv2_DX_DG_DR(nn.Module):
    def __init__(self, X_DIM, G_DIM, R_DIM, E_DIM, HID=256):
        super().__init__()
        self.c1 = GATv2Conv(X_DIM, HID, heads=4, concat=False, edge_dim=E_DIM, fill_value='mean')
        self.c2 = GATv2Conv(HID,  HID, heads=4, concat=False, edge_dim=E_DIM, fill_value='mean')
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(HID + G_DIM + R_DIM, 1)
    def forward(self, d):
        x = F.relu(self.c1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.c2(x, d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim() == 3 else d.g
        r = d.r.squeeze(1) if d.r.dim() == 3 else d.r
        return self.fc(torch.cat([x.float(), g.float(), r.float()], dim=1)).view(-1)

# ===== main procedure =====
def main():
    # 1) data loading
    data = load_pyg_list(PT_PATH)
    if len(data) == 0:
        raise RuntimeError("data  is empty.")
    print(f"[OK] Number of graphs: {len(data)}")

    # 2) dimension check
    sample = data[0]
    X_DIM = sample.x.size(-1)
    if not hasattr(sample, "edge_attr") or sample.edge_attr is None:
        raise RuntimeError("edge_attr is required.")
    E_DIM = sample.edge_attr.size(-1)
    G_DIM = infer_dim(sample.g)
    R_DIM = infer_dim(sample.r)
    HID   = 256
    print(f"[INFO] Dims: x={X_DIM}, edge={E_DIM}, g={G_DIM}, r={R_DIM}")

    # 3) Morgan FP -> KMeans(LCO clusters)
    fps = []
    for d in data:
        m = Chem.MolFromSmiles(d.smiles)
        arr = np.zeros((2048,), dtype=np.float32)
        fp = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) if m is not None else None
        if fp is None:
            fps.append(arr)
        else:
            DataStructs.ConvertToNumpyArray(fp, arr)
            fps.append(arr)

    km = KMeans(n_clusters=K, random_state=SEED, n_init=10).fit(fps)
    for d, c in zip(data, km.labels_):
        d.cluster = int(c)

    # 4) clusters -> 
    cluster_to_indices = {cl: [] for cl in range(K)}
    for i, d in enumerate(data):
        cluster_to_indices[int(d.cluster)].append(i)

    # 5) LCO-CV(test=foldclusters, val=(fold+1)%K, train=remaining)
    rows = []
    for fold in range(K):
        test_cls = fold
        val_cls  = (fold + 1) % K
        test_idx = list(cluster_to_indices[test_cls])
        val_idx  = list(cluster_to_indices[val_cls])
        train_idx = [i for cl, idxs in cluster_to_indices.items() if cl not in (test_cls, val_cls) for i in idxs]

        train_base = [data[i] for i in sorted(train_idx)]
        val_data   = [data[i] for i in sorted(val_idx)]
        test_data  = [data[i] for i in sorted(test_idx)]

        # ---- positiveonly(trainonly)----
        n_train = len(train_base)
        pos_cnt = sum(int(d.y) for d in train_base)
        neg_cnt = n_train - pos_cnt
        pos_rate = pos_cnt / max(1, n_train)
        target_pos = math.ceil(AUG_TARGET_POS_RATE * n_train)
        need_pos   = max(0, target_pos - pos_cnt)

        if pos_cnt > 0 and need_pos > 0:
            per_pos_aug = max(1, min(AUG_MAX_PER_POS, math.ceil(need_pos / pos_cnt)))
            train_aug = []
            for d in train_base:
                if int(d.y) == 1:
                    m = Chem.MolFromSmiles(d.smiles)
                    if m is None:
                        continue
                    for _ in range(per_pos_aug):
                        dd = d.clone()
                        dd.smiles = Chem.MolToSmiles(m, doRandom=True)
                        train_aug.append(dd)
            train_data = train_base + train_aug
            print(f"[AUG] Fold {fold}: pos-aug enabled | base_pos={pos_cnt}, neg={neg_cnt}, "
                  f"pos_rate={pos_rate:.3f} -> target={AUG_TARGET_POS_RATE:.2f}, "
                  f"per_pos_aug={per_pos_aug}, +aug={len(train_aug)}, new_train={len(train_data)}")
        else:
            train_data = train_base
            print(f"[AUG] Fold {fold}: pos-aug skipped | base_pos={pos_cnt}, neg={neg_cnt}, pos_rate={pos_rate:.3f}")

        print(f"\n[FOLD] Fold {fold} | train={len(train_data)} val={len(val_data)} test={len(test_data)}")

        # ---- Training(1) ----
        def train_one(model, train_set, val_set, save_path, tag):
            model.to(DEVICE)
            pw = pos_weight_from(train_set).to(DEVICE)
            crit = nn.BCEWithLogitsLoss(pos_weight=pw, reduction='none')  # per-sample
            opt  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
            sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TMAX)

            best_auc, wait = 0.0, 0
            for ep in range(1, MAX_EPOCH + 1):
                model.train()
                for bt in DataLoader(train_set, batch_size=BATCH_TR, shuffle=True):
                    bt = bt.to(DEVICE)
                    logits = model(bt)
                    loss_vec = crit(logits, bt.y.float())
                    sw, _nm = get_batch_weights(bt, DEVICE)
                    loss = (loss_vec * sw).sum() / (sw.sum() + 1e-8)

                    opt.zero_grad(); loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                    opt.step()
                sch.step()

                yv, pv, wv = eval_probs_model(model, val_set)
                if yv.size > 0:
                    if APPLY_WEIGHT_IN_VAL:
                        roc = roc_auc_score(yv, pv, sample_weight=wv)
                        prc = pr_auc_score(yv, pv, sample_weight=wv)
                    else:
                        roc = roc_auc_score(yv, pv)
                        prc = pr_auc_score(yv, pv)
                    f1  = f1_score(yv, (pv >= 0.5).astype(int))
                else:
                    roc = prc = f1 = 0.0
                print(f"[{tag}] Ep{ep:03d} | ROC={roc:.4f} PR={prc:.4f} F1={f1:.4f}")

                if roc > best_auc:
                    best_auc, wait = roc, 0
                    torch.save(model.state_dict(), save_path)
                else:
                    wait += 1
                    if wait >= PATIENCE:
                        print(f"[OK] EarlyStopping [{tag}]")
                        break

            model.load_state_dict(torch.load(save_path, map_location=DEVICE))
            yv, pv, wv = eval_probs_model(model, val_set)
            return model, yv, pv, wv

        # --- Training(Transformer / GATv2) ---
        m_tr = Transformer_DX_DG_DR(X_DIM, G_DIM, R_DIM, E_DIM, HID)
        m_ga = GATv2_DX_DG_DR(      X_DIM, G_DIM, R_DIM, E_DIM, HID)
        p_tr = os.path.join(SAVE_DIR, f"trans_fold{fold}.pt")
        p_ga = os.path.join(SAVE_DIR, f"gatv2_fold{fold}.pt")

        m_tr, yv, pv_tr, wv = train_one(m_tr, train_data, val_data, p_tr, f"Transformer/F{fold}")
        m_ga, _,  pv_ga, _  = train_one(m_ga, train_data, val_data, p_ga, f"GATv2/F{fold}")

        # --- Validation α*: PR-AUC maximum(；) ---
        if yv.size == 0:
            alpha_star, pr_val = 0.5, 0.0
        else:
            alpha_star, pr_val = best_alpha_for_prauc(
                yv, pv_tr, pv_ga,
                alphas=ALPHAS,
                sample_weight=wv if APPLY_WEIGHT_IN_VAL else None
            )
        print(f"⭐ Fold{fold} best α={alpha_star:.2f} (Val PR-AUC={pr_val:.4f}{' [weighted]' if APPLY_WEIGHT_IN_VAL else ''})")

        # ---  ---
        yt, pt_tr, wt = eval_probs_model(m_tr, test_data)
        _,  pt_ga, wt2 = eval_probs_model(m_ga, test_data)
        if yt.size == 0:
            print(f"[WARN] Fold{fold}: Test split is empty; skipping evaluation."); continue
        assert np.allclose(wt, wt2)

        pt_en = alpha_star * pt_tr + (1 - alpha_star) * pt_ga

        def pack(prefix, y, p, sw=None):
            if sw is not None:
                return {
                    f"{prefix}_ROC": roc_auc_score(y, p, sample_weight=sw),
                    f"{prefix}_PR":  pr_auc_score(y, p, sample_weight=sw),
                    f"{prefix}_F1":  f1_score(y, (p >= 0.5).astype(int)),  # display an unweighted value
                }
            else:
                return {
                    f"{prefix}_ROC": roc_auc_score(y, p),
                    f"{prefix}_PR":  pr_auc_score(y, p),
                    f"{prefix}_F1":  f1_score(y, (p >= 0.5).astype(int)),
                }

        sw_use = wt if APPLY_WEIGHT_IN_TEST else None
        rec = {"Fold": fold, "Alpha": float(alpha_star)}
        rec.update(pack("Transformer", yt, pt_tr, sw_use))
        rec.update(pack("GATv2",      yt, pt_ga, sw_use))
        rec.update(pack("Ensemble",   yt, pt_en, sw_use))
        rows.append(rec)

        print(f"[OK] Fold{fold} [Transformer] ROC={rec['Transformer_ROC']:.4f} PR={rec['Transformer_PR']:.4f} F1={rec['Transformer_F1']:.4f}")
        print(f"[OK] Fold{fold} [GATv2]      ROC={rec['GATv2_ROC']:.4f} PR={rec['GATv2_PR']:.4f} F1={rec['GATv2_F1']:.4f}")
        print(f"[OK] Fold{fold} [Ensemble]   ROC={rec['Ensemble_ROC']:.4f} PR={rec['Ensemble_PR']:.4f} F1={rec['Ensemble_F1']:.4f} (α={alpha_star:.2f})")

        # plotsSave()
        def save_roc_pr(y_true, y_prob, title_prefix, fold_dir, sample_weight=None):
            os.makedirs(fold_dir, exist_ok=True)
            fpr, tpr, _ = roc_curve(y_true, y_prob, sample_weight=sample_weight)
            roc_auc = auc(fpr, tpr)
            plt.figure(); plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
            plt.plot([0,1],[0,1],'--', linewidth=1)
            plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"{title_prefix} ROC")
            plt.legend(); plt.tight_layout()
            plt.savefig(os.path.join(fold_dir, f"{title_prefix}_ROC.png")); plt.close()

            prec, rec, _ = precision_recall_curve(y_true, y_prob, sample_weight=sample_weight)
            ap = average_precision_score(y_true, y_prob, sample_weight=sample_weight)
            plt.figure(); plt.plot(rec, prec, label=f"AP={ap:.3f}")
            plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"{title_prefix} PR")
            plt.legend(); plt.tight_layout()
            plt.savefig(os.path.join(fold_dir, f"{title_prefix}_PR.png")); plt.close()

        fold_dir = os.path.join(SAVE_DIR, f"plots_fold{fold}")
        save_roc_pr(yt, pt_tr, "Transformer", fold_dir, sample_weight=sw_use)
        save_roc_pr(yt, pt_ga, "GATv2",      fold_dir, sample_weight=sw_use)
        save_roc_pr(yt, pt_en, "Ensemble",   fold_dir, sample_weight=sw_use)

    # 6) Save- summary
    if len(rows) == 0:
        print("\n[WARN] All folds were skipped during evaluation."); return

    df = pd.DataFrame(rows)
    out_csv = os.path.join(SAVE_DIR, "lco_cv_metrics_trans_gatv2_ensemble.csv")
    df.to_csv(out_csv, index=False)

    metrics_cols = [c for c in df.columns if c not in ["Fold", "Alpha"]]
    mean_row = df[metrics_cols].mean().to_dict()
    std_row  = df[metrics_cols].std().to_dict()
    print("\n[DONE] LCO-CV done！")
    print("mean:", {k: float(v) for k, v in mean_row.items()})
    print("standard deviation:", {k: float(v) for k, v in std_row.items()})
    print("α*", df["Alpha"].round(2).tolist())
    print(f"[FILE] Saved: {out_csv}")
    print(f"[INFO] Weights applied in validation: {APPLY_WEIGHT_IN_VAL} / Weights applied in test: {APPLY_WEIGHT_IN_TEST}")

if __name__ == "__main__":
    main()

In [ ]:
# ================================================
# LCO-CV with Advanced Clustering & Fold Constraints (QT project)
#  - clustering: Butina(Tanimoto) / K-Medoids(Tanimoto) / Murcko Scaffold(+subcluster)
#  - fold constraints: minimum positives/negatives per fold (with relaxation)
#  - models: TransformerConv & GATv2Conv, PR-AUC-based early stopping
#  - ensemble alpha chosen to maximize validation PR-AUC
# --- Additional notes(shared specification) ---
#  - ★ SampleWeight support: 
#      - Training(BCEWithLogitsLoss(reduction='none') -> )
#      - pos_weight effective weighted ratio(neg_sum/pos_sum, capped at 10)compute
#      - validation/test PR-AUC- ROC() α* (Val PR-AUC)
#      - : [
#          "sample_weight","weight","w","SampleWeight","Weight"
#        ]( 1.0)
#      - : APPLY_WEIGHT_IN_VAL(Validation), APPLY_WEIGHT_IN_TEST()
# ================================================

import os, random, math, gc
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, GATv2Conv, AttentionalAggregation
from torch.serialization import add_safe_globals

from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score, auc

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina

# ====== baseConfiguration(for QT) ======
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

ROOT      = "/content/drive/MyDrive/Chemoinfo_QT"
PT_PATH   = f"{ROOT}/data_graph_with_smiles.pt"  # d.x + d.g + d.r + edge_attr 
SAVE_DIR  = f"{ROOT}/results_lco5_trans_gat_ens_posaug_advanced"
os.makedirs(SAVE_DIR, exist_ok=True)
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- CV / TrainingConfiguration ----
K                  = 5
BATCH_TR           = 64
BATCH_TE           = 128
LR                 = 3e-4
WD                 = 1e-4
PATIENCE           = 10
MAX_EPOCH          = 200
CLIP               = 5.0
ALPHAS             = np.linspace(0,1,11)   # 0.0, 0.1, ..., 1.0

# ---- (positiveonly)----
AUG_TARGET_POS_RATE = 0.30
AUG_MAX_PER_POS     = 3

# ---- clusters ----
# "butina" | "kmedoids" | "scaffold"
CLUSTER_METHOD   = "butina"
BUTINA_THRESH    = 0.7      # tanimoto >= 0.7 clusters
KMEDOIDS_ITERS   = 50
SUBCLUSTER_MIN   = 10
SUBCLUSTER_K_MAX = 5

# ---- fold-balance constraints ----
MIN_POS_PER_FOLD = 10
MIN_NEG_PER_FOLD = 10
FOLD_ASSIGN_TRIALS = 200

# ★ SampleWeight Configuration(Additional notes)
WEIGHT_ATTR_CANDIDATES = ["sample_weight","weight","w","SampleWeight","Weight"]
APPLY_WEIGHT_IN_VAL  = True   # Val- α*
APPLY_WEIGHT_IN_TEST = False  # set True to report weighted test metrics

# ====== utilities() ======
def pr_auc_score(y, p, sample_weight=None):
    prec, rec, _ = precision_recall_curve(y, p, sample_weight=sample_weight)
    return auc(rec, prec)

def _get_weight_from_data(d):
    for nm in WEIGHT_ATTR_CANDIDATES:
        if hasattr(d, nm):
            v = getattr(d, nm)
            return float(v.item()) if torch.is_tensor(v) else float(v)
    return 1.0

def _to_1d(w):
    if torch.is_tensor(w):
        if w.dim() == 2 and w.size(1) == 1: return w.view(-1)
        return w.view(-1)
    return torch.tensor(w, dtype=torch.float32)

def get_batch_weights(batch, device=None):
    for nm in WEIGHT_ATTR_CANDIDATES:
        if hasattr(batch, nm):
            w = getattr(batch, nm)
            w = _to_1d(w).to(device or batch.y.device).float()
            return w, nm
    return torch.ones(batch.y.numel(), device=device or batch.y.device, dtype=torch.float32), "CONST1"

def pos_weight_from(dataset):
    ys = torch.tensor([float(d.y.item()) for d in dataset])
    ws = torch.tensor([_get_weight_from_data(d) for d in dataset], dtype=torch.float32)
    pos_sum = ws[ys == 1].sum()
    neg_sum = ws[ys == 0].sum()
    w = (neg_sum / torch.clamp(pos_sum, min=1e-8)).item()
    return torch.tensor([min(max(w, 1.0), 10.0)], dtype=torch.float32)

@torch.no_grad()
def eval_probs(model, ds, batch_size=BATCH_TE):
    model.eval(); ys, ps, ws = [], [], []
    for bt in DataLoader(ds, batch_size=batch_size, shuffle=False):
        bt = bt.to(DEVICE)
        ys.append(bt.y.detach().cpu())
        ps.append(torch.sigmoid(model(bt)).detach().cpu())
        w, _ = get_batch_weights(bt, DEVICE)
        ws.append(w.detach().cpu())
    return torch.cat(ys).numpy(), torch.cat(ps).numpy(), torch.cat(ws).numpy()

def infer_dim(t):
    if t.dim() == 1: return t.numel()
    if t.dim() in (2,3): return t.size(-1)
    raise RuntimeError(f"Unknown tensor dimensionality: {t.size()}")

def best_alpha_for_prauc(y_val, p1, p2, alphas=ALPHAS, sample_weight=None):
    best_a, best_s = 0.5, -1.0
    for a in alphas:
        s = pr_auc_score(y_val, a*p1 + (1-a)*p2, sample_weight=sample_weight)
        if s > best_s:
            best_s, best_a = s, a
    return best_a, best_s

# ======  ======
def build_models_dims(sample):
    X_DIM = sample.x.size(-1)
    G_DIM = infer_dim(sample.g)
    R_DIM = infer_dim(sample.r)
    assert hasattr(sample, "edge_attr") and sample.edge_attr is not None, "edge_attr is required."
    E_DIM = sample.edge_attr.size(-1)
    HID   = 256
    return X_DIM, G_DIM, R_DIM, E_DIM, HID

class Transformer_DX_DG_DR(nn.Module):
    def __init__(self, X_DIM, G_DIM, R_DIM, E_DIM, HID=256):
        super().__init__()
        self.c1 = TransformerConv(X_DIM, HID, heads=4, concat=False, edge_dim=E_DIM, dropout=0.0, beta=False)
        self.c2 = TransformerConv(HID,  HID, heads=4, concat=False, edge_dim=E_DIM, dropout=0.0, beta=False)
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(HID + G_DIM + R_DIM, 1)
    def forward(self, d):
        x = F.relu(self.c1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.c2(x, d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim() == 3 else d.g
        r = d.r.squeeze(1) if d.r.dim() == 3 else d.r
        return self.fc(torch.cat([x, g, r], dim=1)).view(-1)

class GATv2_DX_DG_DR(nn.Module):
    def __init__(self, X_DIM, G_DIM, R_DIM, E_DIM, HID=256):
        super().__init__()
        # edge_attr  attention (edge_dim)
        self.g1 = GATv2Conv(X_DIM, HID, heads=4, concat=False, edge_dim=E_DIM, fill_value='mean', dropout=0.0)
        self.g2 = GATv2Conv(HID,  HID, heads=4, concat=False, edge_dim=E_DIM, fill_value='mean', dropout=0.0)
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(HID + G_DIM + R_DIM, 1)
    def forward(self, d):
        x = F.relu(self.g1(d.x, d.edge_index, d.edge_attr))
        x = F.relu(self.g2(x, d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim() == 3 else d.g
        r = d.r.squeeze(1) if d.r.dim() == 3 else d.r
        return self.fc(torch.cat([x, g, r], dim=1)).view(-1)


def train_one(model, train_set, val_set, save_path, tag):
    model.to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight_from(train_set).to(DEVICE),
                                reduction='none')  # per-sample
    opt  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    best_pr, wait = -1.0, 0
    for ep in range(1, MAX_EPOCH+1):
        model.train()
        for bt in DataLoader(train_set, batch_size=BATCH_TR, shuffle=True):
            bt = bt.to(DEVICE)
            logits  = model(bt)
            lossvec = crit(logits, bt.y.float())
            sw, _nm = get_batch_weights(bt, DEVICE)
            loss = (lossvec * sw).sum() / (sw.sum() + 1e-8)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()

        yv, pv, wv = eval_probs(model, val_set)
        prc = pr_auc_score(yv, pv, sample_weight=wv if APPLY_WEIGHT_IN_VAL else None)
        roc = roc_auc_score(yv, pv, sample_weight=wv if APPLY_WEIGHT_IN_VAL else None)
        f1  = f1_score(yv, (pv>=0.5).astype(int))  # 
        print(f"[{tag}] Ep{ep:03d} | PR={prc:.4f} ROC={roc:.4f} F1@0.5={f1:.4f}")

        if prc > best_pr:
            best_pr, wait = prc, 0
            torch.save(model.state_dict(), save_path)
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"[OK] EarlyStopping (best Val PR-AUC={best_pr:.4f}) [{tag}]")
                break

    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    return model, *eval_probs(model, val_set)

# ====== ECFP4 & Tanimoto ======

def mol_from_smiles(s):
    try:
        return Chem.MolFromSmiles(s) if s is not None else None
    except:
        return None

def fp_from_mol(m):
    if m is None:
        # 2048(=0)
        return AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(""), 2, nBits=2048)
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048)

# ====== clusters ======

def cluster_butina(fps, threshold=0.7):
    dists = []
    for i in range(1, len(fps)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1.0 - float(s) for s in sims])
    cs = Butina.ClusterData(dists, len(fps), 1.0 - float(threshold), isDistData=True)
    return [list(c) for c in cs]


def kmedoids_tanimoto(fps, k, max_iters=50, seed=SEED):
    rng = np.random.RandomState(seed)
    n = len(fps)
    medoids = rng.choice(n, size=min(k, n), replace=False).tolist()
    if len(medoids) < k: k = len(medoids)

    def assign(meds):
        assigns = np.empty(n, dtype=np.int64)
        for i in range(n):
            best_j, best_d = 0, 1e9
            for j, m in enumerate(meds):
                d = 1.0 - DataStructs.TanimotoSimilarity(fps[i], fps[m])
                if d < best_d: best_d, best_j = d, j
            assigns[i] = best_j
        return assigns

    def total_cost(assigns, meds):
        return sum(1.0 - DataStructs.TanimotoSimilarity(fps[i], fps[meds[assigns[i]]]) for i in range(n))

    assigns = assign(medoids)
    best_cost = total_cost(assigns, medoids)
    it, improved = 0, True
    while improved and it < max_iters:
        improved, it = False, it+1
        for mi in range(k):
            for cand in range(n):
                if cand in medoids: continue
                new_meds = medoids.copy(); new_meds[mi] = cand
                new_assigns = assign(new_meds)
                cost = total_cost(new_assigns, new_meds)
                if cost + 1e-9 < best_cost:
                    medoids, assigns, best_cost = new_meds, new_assigns, cost
                    improved = True
    clusters = [[] for _ in range(k)]
    for idx, a in enumerate(assigns): clusters[a].append(idx)
    return [c for c in clusters if len(c) > 0]


def scaffold_key(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaf, isomericSmiles=False) if scaf is not None else None
    except:
        return None


def cluster_scaffold_then_subcluster(smiles_list, fps, sub_min=10, sub_kmax=5):
    sc_groups = defaultdict(list)
    for i, smi in enumerate(smiles_list):
        key = scaffold_key(mol_from_smiles(smi))
        if key is None: key = f"_NOSCAF_{i}"
        sc_groups[key].append(i)

    clusters = []
    for _, idxs in sc_groups.items():
        if len(idxs) >= sub_min:
            k = max(1, min(sub_kmax, int(math.ceil(len(idxs)/sub_min))))
            sub_fps = [fps[i] for i in idxs]
            sub_clusters = kmedoids_tanimoto(sub_fps, k=k, max_iters=KMEDOIDS_ITERS, seed=SEED)
            for sc in sub_clusters:
                clusters.append([idxs[t] for t in sc])
        else:
            clusters.append(idxs)
    return clusters

# ====== fold () ======

def summarize_cluster_labels(data, clusters):
    rows = []
    for cid, idxs in enumerate(clusters):
        ys = [int(data[i].y) for i in idxs]
        rows.append({"cid":cid, "size":len(idxs), "pos":sum(ys), "neg":len(ys)-sum(ys)})
    return pd.DataFrame(rows)


def try_pack_clusters(clusters, labels_df, k=K, min_pos=MIN_POS_PER_FOLD, min_neg=MIN_NEG_PER_FOLD, trials=FOLD_ASSIGN_TRIALS):
    sizes = labels_df.set_index("cid")["size"].to_dict()
    pos_c = labels_df.set_index("cid")["pos"].to_dict()
    neg_c = labels_df.set_index("cid")["neg"].to_dict()
    cluster_ids = labels_df.sort_values("size", ascending=False)["cid"].tolist()

    best_solution, best_score = None, (1e9, 1e9)  # (violations, size_var)

    for _ in range(trials):
        folds = [{"cids":[], "pos":0, "neg":0, "size":0} for _ in range(k)]
        for cid in cluster_ids:
            best_fold, best_tuple = None, (1e9, 1e9)
            for f in range(k):
                p = folds[f]["pos"]  + pos_c[cid]
                n = folds[f]["neg"]  + neg_c[cid]
                s = folds[f]["size"] + sizes[cid]
                viol = int(p < min_pos) + int(n < min_neg)
                score = (viol, s)
                if score < best_tuple:
                    best_tuple, best_fold = score, f
            f = best_fold
            folds[f]["cids"].append(cid)
            folds[f]["pos"]  += pos_c[cid]
            folds[f]["neg"]  += neg_c[cid]
            folds[f]["size"] += sizes[cid]

        viol = sum(int(fd["pos"] < min_pos or fd["neg"] < min_neg) for fd in folds)
        sz = np.array([fd["size"] for fd in folds], dtype=float)
        size_var = float(np.var(sz))
        score = (viol, size_var)
        if score < best_score:
            best_score, best_solution = score, folds
            if viol == 0: break
    return best_solution, best_score


def assign_folds_with_relaxation(data, clusters, k=K, min_pos=MIN_POS_PER_FOLD, min_neg=MIN_NEG_PER_FOLD):
    labels_df = summarize_cluster_labels(data, clusters)
    folds, score = try_pack_clusters(clusters, labels_df, k=k, min_pos=min_pos, min_neg=min_neg)
    viol, size_var = score

    relax_steps = 0
    mpos, mneg = min_pos, min_neg
    while viol > 0 and (mpos > 0 or mneg > 0):
        relax_steps += 1
        if mpos > 0: mpos -= 1
        if mneg > 0: mneg -= 1
        folds2, score2 = try_pack_clusters(clusters, labels_df, k=k, min_pos=mpos, min_neg=mneg)
        if score2 < score:
            folds, score = folds2, score2
            viol, size_var = score
        if viol == 0: break

    node2fold = {}
    for f, info in enumerate(folds):
        for cid in info["cids"]:
            for idx in clusters[cid]:
                node2fold[idx] = f

    recs = [{"Fold": f, "size": fd["size"], "pos": fd["pos"], "neg": fd["neg"]} for f, fd in enumerate(folds)]
    fold_summary = pd.DataFrame(recs)
    labels_df["fold"] = -1
    for f, info in enumerate(folds):
        for cid in info["cids"]:
            labels_df.loc[labels_df["cid"]==cid, "fold"] = f

    return node2fold, fold_summary, labels_df, {
        "violations": int(viol),
        "size_var": float(size_var),
        "relax_steps": int(relax_steps),
        "min_pos_final": int(mpos),
        "min_neg_final": int(mneg)
    }

# ======  ======

def load_pyg_list(path):
    try:
        from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
        from torch_geometric.data.storage import GlobalStorage
        add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
    except Exception as e:
        print(f"[WARN] add_safe_globals failed: {e}")
    try:
        obj = torch.load(path)  # PyTorch 2.6: weights_only=True Default
    except Exception as e:
        print(f"[WARN] Safe loading failed({e}).retrying with weights_only=False (assuming a trusted file).")
        obj = torch.load(path, weights_only=False)

    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    try: return list(obj)
    except Exception as e:
        raise RuntimeError(f"Unknown format: {type(obj)}; failed to convert to list: {e}")

# ======  ======

def main():
    # 1) data loading
    data = load_pyg_list(PT_PATH)
    if len(data) == 0:
        raise RuntimeError("data  is empty.")
    print(f"[OK] Number of graphs: {len(data)}")
    sample = data[0]
    X_DIM, G_DIM, R_DIM, E_DIM, HID = build_models_dims(sample)

    # 2) SMILES / mol / fp
    smiles = [getattr(d, "smiles", None) for d in data]
    mols   = [mol_from_smiles(s) for s in smiles]
    fps    = [fp_from_mol(m) for m in mols]

    # 3) clusters
    cm = CLUSTER_METHOD.lower()
    if cm == "butina":
        clusters = cluster_butina(fps, threshold=BUTINA_THRESH)
        method_tag = f"butina_t{BUTINA_THRESH}"
    elif cm == "kmedoids":
        clusters = kmedoids_tanimoto(fps, k=K, max_iters=KMEDOIDS_ITERS, seed=SEED)
        method_tag = f"kmedoids_k{K}_it{KMEDOIDS_ITERS}"
    elif cm == "scaffold":
        clusters = cluster_scaffold_then_subcluster(smiles, fps, sub_min=SUBCLUSTER_MIN, sub_kmax=SUBCLUSTER_K_MAX)
        method_tag = f"scaffold_sub{SUBCLUSTER_K_MAX}"
    else:
        raise ValueError("CLUSTER_METHOD  'butina' | 'kmedoids' | 'scaffold'")

    # K
    while len(clusters) < K:
        clusters = sorted(clusters, key=len)
        a, b = clusters.pop(0), clusters.pop(0)
        clusters.append(a + b)
    print(f"[SETUP] Clusters={len(clusters)} (method={method_tag})")

    # 4)  fold 
    node2fold, fold_summary, cluster_labels, extra = assign_folds_with_relaxation(
        data, clusters, k=K, min_pos=MIN_POS_PER_FOLD, min_neg=MIN_NEG_PER_FOLD
    )
    print("[SUMMARY] Fold summary:\n", fold_summary)
    print("[WARN] Constraint:", extra)

    cluster_labels.to_csv(os.path.join(SAVE_DIR, f"clusters_{method_tag}.csv"), index=False)
    fold_summary.to_csv(os.path.join(SAVE_DIR, f"fold_summary_{method_tag}.csv"), index=False)

    for i, d in enumerate(data):
        d.cluster = int(node2fold[i])

    # 5) LCO-CV
    rows = []
    for fold in range(K):
        test_idx = [i for i,d in enumerate(data) if d.cluster==fold]
        rest_idx = [i for i,d in enumerate(data) if d.cluster!=fold]

        rng = np.random.default_rng(SEED + fold)
        rng.shuffle(rest_idx)

        n_val = max(1, int(0.1 * len(rest_idx)))
        val_idx   = rest_idx[:n_val]
        train_idx = rest_idx[n_val:]

        test_data  = [data[i] for i in test_idx]
        val_data   = [data[i] for i in val_idx]
        train_base = [data[i] for i in train_idx]

        # ---- positive(Trainingonly)----
        n_train = len(train_base)
        pos_cnt = sum(int(d.y) for d in train_base)
        neg_cnt = n_train - pos_cnt
        pos_rate = pos_cnt / max(1, n_train)
        target_pos = math.ceil(AUG_TARGET_POS_RATE * n_train)
        need_pos   = max(0, target_pos - pos_cnt)

        if pos_cnt > 0 and need_pos > 0:
            per_pos_aug = max(1, min(AUG_MAX_PER_POS, math.ceil(need_pos / pos_cnt)))
            train_aug = []
            for d in train_base:
                if int(d.y)==1:
                    for _ in range(per_pos_aug):
                        dd = d.clone()  #  clone 
                        train_aug.append(dd)
            train_data = train_base + train_aug
            print(f"[AUG] Fold {fold}: pos-oversample | base_pos={pos_cnt}, neg={neg_cnt}, "
                  f"pos_rate={pos_rate:.3f} -> target={AUG_TARGET_POS_RATE:.2f}, "
                  f"per_pos_aug={per_pos_aug}, +aug={len(train_aug)}, new_train={len(train_data)}")
        else:
            train_data = train_base
            print(f"[AUG] Fold {fold}: oversample skipped | base_pos={pos_cnt}, neg={neg_cnt}, pos_rate={pos_rate:.3f}")

        print(f"\n[FOLD] Fold {fold} | train={len(train_data)} val={len(val_data)} test={len(test_data)}")

        # ----  ----
        m_tr  = Transformer_DX_DG_DR(X_DIM, G_DIM, R_DIM, E_DIM, HID)
        m_gat = GATv2_DX_DG_DR(     X_DIM, G_DIM, R_DIM, E_DIM, HID)
        p_tr  = os.path.join(SAVE_DIR, f"{method_tag}_transformer_fold{fold}.pt")
        p_gat = os.path.join(SAVE_DIR, f"{method_tag}_gatv2_fold{fold}.pt")

        # ---- Training(PR-AUCSave；)----
        m_tr,  yv, pv_tr,  wv = train_one(m_tr,  train_data, val_data, p_tr,  f"Transformer/F{fold}")
        m_gat, _,  pv_gat, _  = train_one(m_gat, train_data, val_data, p_gat, f"GATv2/F{fold}")

        # ---- α*: Val PR maximum()----
        alpha_star, pr_val = best_alpha_for_prauc(
            yv, pv_tr, pv_gat, alphas=ALPHAS, sample_weight=wv if APPLY_WEIGHT_IN_VAL else None
        )
        print(f"⭐ Fold{fold} best α={alpha_star:.2f} (Val PR-AUC={pr_val:.4f}{' [weighted]' if APPLY_WEIGHT_IN_VAL else ''})")

        # ---- ()----
        yt, pt_tr,  wt  = eval_probs(m_tr,  test_data)
        _,  pt_gat, wt2 = eval_probs(m_gat, test_data)
        assert np.allclose(wt, wt2)
        pt_en = alpha_star * pt_tr + (1 - alpha_star) * pt_gat
        sw_use = wt if APPLY_WEIGHT_IN_TEST else None

        def pack(prefix, y, p, sw=None):
            return {
                f"{prefix}_ROC": roc_auc_score(y, p, sample_weight=sw) if sw is not None else roc_auc_score(y, p),
                f"{prefix}_PR":  pr_auc_score(y, p, sample_weight=sw),
                f"{prefix}_F1":  f1_score(y, (p>=0.5).astype(int)),  # 
            }

        rec = {"Fold": fold, "Alpha": float(alpha_star), "Method": CLUSTER_METHOD}
        rec.update(pack("Transformer", yt, pt_tr,  sw_use))
        rec.update(pack("GATv2",      yt, pt_gat, sw_use))
        rec.update(pack("Ensemble",    yt, pt_en,  sw_use))
        rows.append(rec)

        print(f"[OK] Fold{fold} [Transformer] ROC={rec['Transformer_ROC']:.4f} PR={rec['Transformer_PR']:.4f} F1={rec['Transformer_F1']:.4f}")
        print(f"[OK] Fold{fold} [GATv2]       ROC={rec['GATv2_ROC']:.4f} PR={rec['GATv2_PR']:.4f} F1={rec['GATv2_F1']:.4f}")
        print(f"[OK] Fold{fold} [Ensemble]    ROC={rec['Ensemble_ROC']:.4f} PR={rec['Ensemble_PR']:.4f} F1={rec['Ensemble_F1']:.4f}")

        del m_tr, m_gat; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # 6) Save- 
    df = pd.DataFrame(rows)
    out_csv = os.path.join(SAVE_DIR, f"lco_cv_metrics_{method_tag}.csv")
    df.to_csv(out_csv, index=False)

    metrics_cols = [c for c in df.columns if c not in ["Fold", "Alpha", "Method"]]
    mean_row = df[metrics_cols].mean().to_dict()
    std_row  = df[metrics_cols].std().to_dict()
    print("\n[DONE] LCO-CV done！")
    print("mean:", {k: float(v) for k, v in mean_row.items()})
    print("standard deviation:", {k: float(v) for k, v in std_row.items()})
    print("α*", df["Alpha"].round(2).tolist())
    print("Save:", out_csv)

# ====== Run ======
if __name__ == "__main__":
    from rdkit import RDLogger
    import warnings as _warnings
    RDLogger.DisableLog('rdApp.*')
    _warnings.filterwarnings("ignore", category=DeprecationWarning)
    main()

In [ ]:
# ================================
# AID_588834 -> externalset
# Requirements:
#  1) merged_for_model_consolidated.csv SMILES from
#     AID_588834_datatable.csv  PUBCHEM_EXT_DATASOURCE_SMILES match and remove overlaps
#  2) apply labels for the hERG inhibition model(Active=1, Inactive=0, drop Inconclusive entries)
#  3) save as an external dataset
#  4) print label crosstabs to the console(simple frequency table plus crosstabs against key metadata columns)
# -------------------------------
# Default output directory: /content/drive/MyDrive/Chemoinfo_QT/external_sets/
# If saving fails /content/ fallback
# ================================

import os, sys, re, json
import pandas as pd
import numpy as np
from pathlib import Path

# ==== Configuration(adjust as needed) ====
PATH_AID    = "/content/drive/MyDrive/Chemoinfo_QT/AID_588834_datatable.csv"           # e.g.: /mnt/data if uploaded there
PATH_MERGED = "/content/drive/MyDrive/Chemoinfo_QT/merged_for_model_consolidated.csv"   # e.g.: /mnt/data if uploaded there

# Output(Default)
ROOT_PRIMARY = Path("/content/drive/MyDrive/Chemoinfo_QT")
OUT_DIR_PRIMARY = ROOT_PRIMARY
OUT_NAME = "external_AID_588834_herg.csv"

# fallback
OUT_DIR_FALLBACK = Path("/content")
OUT_PATH_PRIMARY = OUT_DIR_PRIMARY / OUT_NAME
OUT_PATH_FALLBACK = OUT_DIR_FALLBACK / OUT_NAME

# ==== utilities ====
def _maybe_drop_meta_row(df: pd.DataFrame) -> pd.DataFrame:
    """AID_588834_datatable.csv at the top 'RESULT_TYPE' type-definition metadata rowremove it if present."""
    if df.empty:
        return df
    first_row = df.iloc[0].astype(str).str.upper().tolist()
    tokens = {"RESULT_TYPE", "STRING", "FLOAT", "INTEGER"}
    if any(any(tok in cell for tok in tokens) for cell in first_row):
        return df.iloc[1:].reset_index(drop=True)
    return df

def _norm_smiles(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip()  # onlyNormalization()

def _outcome_to_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "active":
        return 1
    if s == "inactive":
        return 0
    # (inconclusive) NaN(drop)
    return np.nan

# ==== Input ====
aid_raw = pd.read_csv(PATH_AID)
aid = _maybe_drop_meta_row(aid_raw.copy())

merged = pd.read_csv(PATH_MERGED)

# Required-column check
required_aid_cols = ["PUBCHEM_EXT_DATASOURCE_SMILES", "PUBCHEM_ACTIVITY_OUTCOME"]
missing_aid = [c for c in required_aid_cols if c not in aid.columns]
required_merged_cols = ["SMILES"]
missing_merged = [c for c in required_merged_cols if c not in merged.columns]

if missing_aid:
    raise ValueError(f"Required columns are missing in the AID file: {missing_aid}")
if missing_merged:
    raise ValueError(f"Required columns are missing in the merged file: {missing_merged}")

# ==== 1) SMILES ====
merged_smiles_set = set(merged["SMILES"].map(_norm_smiles).dropna().unique())
aid["PUBCHEM_EXT_DATASOURCE_SMILES_N"] = aid["PUBCHEM_EXT_DATASOURCE_SMILES"].map(_norm_smiles)

mask_overlap = aid["PUBCHEM_EXT_DATASOURCE_SMILES_N"].isin(merged_smiles_set)
aid_nodup = aid.loc[~mask_overlap].copy()

# AID(CID,SMILES)as a precaution
if "PUBCHEM_CID" in aid_nodup.columns:
    aid_nodup = aid_nodup.drop_duplicates(subset=["PUBCHEM_CID", "PUBCHEM_EXT_DATASOURCE_SMILES_N"], keep="first")
else:
    aid_nodup = aid_nodup.drop_duplicates(subset=["PUBCHEM_EXT_DATASOURCE_SMILES_N"], keep="first")

# ==== 2) (Inconclusivedrop) ====
aid_nodup["label"] = aid_nodup["PUBCHEM_ACTIVITY_OUTCOME"].map(_outcome_to_label)
external_df = aid_nodup.loc[aid_nodup["label"].isin([0, 1])].copy()

# SMILEScolumn(isomeric)
external_df.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "SMILES_ISO"}, inplace=True)

# ==== 3) save as an external dataset ====
preferred_cols = [
    "PUBCHEM_CID", "SMILES_ISO", "label",
    "Fit_LogAC50", "Efficacy", "Hill_Slope", "R2",
    "Curve_Description", "PUBCHEM_ACTIVITY_OUTCOME"
]
ordered_cols = [c for c in preferred_cols if c in external_df.columns] + \
               [c for c in external_df.columns if c not in preferred_cols]
external_df = external_df[ordered_cols]

# Output(Drive ->  /content)
out_path = None
try:
    OUT_DIR_PRIMARY.mkdir(parents=True, exist_ok=True)
    external_df.to_csv(OUT_PATH_PRIMARY, index=False)
    out_path = OUT_PATH_PRIMARY
except Exception as e:
    OUT_DIR_FALLBACK.mkdir(parents=True, exist_ok=True)
    external_df.to_csv(OUT_PATH_FALLBACK, index=False)
    out_path = OUT_PATH_FALLBACK

# ==== 4) labelOutput ====
print("\n===== [Frequency] label  =====")
print(external_df["label"].value_counts(dropna=False).sort_index().to_string())

# PUBCHEM_ACTIVITY_OUTCOME (as a precaution)
if "PUBCHEM_ACTIVITY_OUTCOME" in external_df.columns:
    print("\n===== [Crosstab] label × PUBCHEM_ACTIVITY_OUTCOME =====")
    ct1 = pd.crosstab(external_df["label"], external_df["PUBCHEM_ACTIVITY_OUTCOME"])
    print(ct1.to_string())

# curvequality()
if "Curve_Description" in external_df.columns:
    print("\n===== [Crosstab] label × Curve_Description =====")
    # columnonlycolumn
    ct2 = pd.crosstab(external_df["label"], external_df["Curve_Description"])
    # Ncolumnonly
    # top_cols = ct2.sum(axis=0).sort_values(ascending=False).head(12).index
    # ct2 = ct2[top_cols]
    print(ct2.to_string())

# Fitquality(: R2binarization)
if "R2" in external_df.columns:
    print("\n===== [Crosstab] label × (R2 >= 0.9) =====")
    ct3 = pd.crosstab(external_df["label"], (external_df["R2"] >= 0.9))
    print(ct3.to_string())

# ==== (at the end) ====
summary = {
    "input_rows_aid_raw": int(len(aid_raw)),
    "input_rows_aid_after_meta": int(len(aid)),
    "input_rows_merged": int(len(merged)),
    "overlap_removed_rows": int(mask_overlap.sum()),
    "rows_after_overlap_removal": int(len(aid_nodup)),
    "external_rows_after_label": int(len(external_df)),
    "label_counts": external_df["label"].value_counts(dropna=False).to_dict(),
    "output_path": str(out_path),
}
print("\n===== Summary =====")
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# =========================================
# External validation data -> PyG data generation(with scaler reuse)
# Input: /content/drive/MyDrive/Chemoinfo_QT/external_sets/external_AID_588834_herg.csv
# required columns: SMILES or SMILES_ISO, label(0/1)
# optionalcolumn: Sample_Weight/SampleWeight( 1.0), CID/PubChem_CID( -1)
# Specification:
#  - saved during training StandardScaler(gfeat7 / rdesc10)(do not refit)
#  - Because this is external validation SMILES do not apply augmentation
# Output:
#   /content/drive/MyDrive/Chemoinfo_QT/data_graph_external.pt
#   /content/drive/MyDrive/Chemoinfo_QT/data_graph_external_index.csv
# =========================================

import os, json, random
import numpy as np
import pandas as pd
from pathlib import Path

# ---- RDKit  ----
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.info')
RDLogger.DisableLog('rdApp.debug')

from rdkit import Chem
from rdkit.Chem import Descriptors, rdchem

# ---- PyTorch / PyG ----
import torch
from torch_geometric.data import Data

# ----  ----
import joblib

# ========== Configuration ==========
ROOT = Path("/content/drive/MyDrive/Chemoinfo_QT")
DEFAULT_INPUT = ROOT / "external_AID_588834_herg.csv"
ALT_INPUTS = [Path("/content/external_AID_588834_herg.csv"),
              Path("/mnt/data/external_AID_588834_herg.csv")]  # fallback path

SAVE_PATH = ROOT / "data_graph_external.pt"
INDEX_CSV = ROOT / "data_graph_external_index.csv"

SCALER_G_PATH = ROOT / "scaler_g.joblib"
SCALER_R_PATH = ROOT / "scaler_r.joblib"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ========== utilities ==========
def find_col(df, names):
    low = {str(c).strip().lower(): c for c in df.columns}
    for n in names:
        c = low.get(n.lower())
        if c: return c
    return None

# --- molecular features(d.g, d.r) ---
def gfeat7(mol):
    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.HeavyAtomCount(mol),
        Descriptors.RingCount(mol),
    ]

def rdesc10(mol):
    return [
        Descriptors.FpDensityMorgan1(mol),
        Descriptors.FpDensityMorgan2(mol),
        Descriptors.FpDensityMorgan3(mol),
        Descriptors.NumAliphaticRings(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumValenceElectrons(mol),
        Descriptors.BalabanJ(mol),
        Descriptors.BertzCT(mol),
        Descriptors.FractionCSP3(mol),
    ]

# --- -  ---
def atom_f(atom: rdchem.Atom):
    return torch.tensor([
        atom.GetAtomicNum(),
        atom.GetTotalDegree(),
        atom.GetFormalCharge(),
        float(atom.GetIsAromatic()),
        atom.GetTotalNumHs(includeNeighbors=True),
    ], dtype=torch.float)

def bond_f(bond: rdchem.Bond):
    return torch.tensor([
        bond.GetBondTypeAsDouble(),
        float(bond.GetIsConjugated()),
        float(bond.IsInRing()),
        int(bond.GetStereo()),
        bond.GetBeginAtom().GetAtomicNum(),
        bond.GetEndAtom().GetAtomicNum(),
    ], dtype=torch.float)

def to_int_or(x, default=-1):
    if pd.isna(x): return default
    try:
        return int(x)
    except Exception:
        return default

# ========== Resolve input paths ==========
input_path = None
if DEFAULT_INPUT.exists():
    input_path = DEFAULT_INPUT
else:
    for p in ALT_INPUTS:
        if p.exists():
            input_path = p
            break

if input_path is None:
    raise FileNotFoundError(
        f"External dataset not found.\n"
        f":\n - {DEFAULT_INPUT}\n - {ALT_INPUTS[0]}\n - {ALT_INPUTS[1]}"
    )

print(f"[INPUT] input: {input_path}")

# ==========  ==========
if not (SCALER_G_PATH.exists() and SCALER_R_PATH.exists()):
    raise FileNotFoundError(
        "Saved training scalers were not found. They are expected at:\n"
        f" - {SCALER_G_PATH}\n - {SCALER_R_PATH}\n"
        "(Save scaler_g.joblib and scaler_r.joblib in the training script.)"
    )
scaler_g = joblib.load(SCALER_G_PATH)
scaler_r = joblib.load(SCALER_R_PATH)
print("[OK] Scalers loaded successfully")

# ========== Data loading & Cleanup ==========
df0 = pd.read_csv(input_path, low_memory=False)

col_smiles = find_col(df0, ["SMILES_ISO","SMILES","smiles_iso"])
col_label  = find_col(df0, ["label","Label"])
if not col_smiles or not col_label:
    raise ValueError("required columns SMILES/SMILES_ISO  label  not found.")

col_sw  = find_col(df0, ["Sample_Weight","SampleWeight","sample_weight","sampleweight"])
col_cid = find_col(df0, ["PUBCHEM_CID","CID","PubChem_CID"])

# Normalization
df0["SMILES"] = df0[col_smiles].astype(str).str.strip()
df0["label"]  = pd.to_numeric(df0[col_label], errors="coerce").astype("Int64")

if col_sw:
    df0["SampleWeight"] = pd.to_numeric(df0[col_sw], errors="coerce").fillna(1.0).astype(float).clip(lower=0.0)
else:
    df0["SampleWeight"] = 1.0

if col_cid:
    df0["CID"] = pd.to_numeric(df0[col_cid], errors="coerce")
else:
    df0["CID"] = np.nan

# label(0/1 only)
before = len(df0)
df0 = df0[df0["label"].isin([0,1])]
df0["label"] = df0["label"].astype(int)
after = len(df0)
print(f"[INFO] rows: {before} -> {after} (drop NaN/non-binary labels)")

# Label distribution
print("[CHECK] label counts:", df0["label"].value_counts().to_dict())

# ========== RDKit  & descriptorscompute ==========
mols, keep, g7_raw, r10_raw = [], [], [], []
for i, (sm, ) in enumerate(df0[["SMILES"]].itertuples(index=False)):
    m = Chem.MolFromSmiles(sm)
    if m is None or m.GetNumAtoms() == 0:
        continue
    mols.append(m); keep.append(i)
    g7_raw.append(gfeat7(m)); r10_raw.append(rdesc10(m))

dff = df0.iloc[keep].reset_index(drop=True)
print(f"[OK] valid mols: {len(mols)} / {len(df0)}")

# ========== (do not fit) ==========
g7_scaled = scaler_g.transform(g7_raw)
r10_scaled = scaler_r.transform(r10_raw)

# ========== graph conversion ==========
data_list = []
for row_idx, m in enumerate(mols):
    y  = int(dff.loc[row_idx, "label"])
    sw = float(dff.loc[row_idx, "SampleWeight"])
    cid_val = to_int_or(dff.loc[row_idx, "CID"], default=-1)

    if m.GetNumBonds() == 0:
        # skip single-atom and related edge cases
        continue

    # 
    x = torch.stack([atom_f(a) for a in m.GetAtoms()])  # [N,5]

    # ()
    src, dst, eattr = [], [], []
    for b in m.GetBonds():
        i = b.GetBeginAtomIdx(); j = b.GetEndAtomIdx()
        f = bond_f(b)
        src += [i, j]; dst += [j, i]
        eattr += [f, f]
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr  = torch.stack(eattr)

    d = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([y], dtype=torch.float32)  # keep labels for evaluation
    )
    d.g = torch.tensor(g7_scaled[row_idx], dtype=torch.float32).unsqueeze(0)   # [1,7]
    d.r = torch.tensor(r10_scaled[row_idx], dtype=torch.float32).unsqueeze(0)  # [1,10]
    d.sample_weight = torch.tensor([sw], dtype=torch.float32)
    d.smiles = dff.loc[row_idx, "SMILES"]
    d.cid = cid_val
    data_list.append(d)

print(f"[OK] graphs: {len(data_list)}")
lbl = [int(d.y.item()) for d in data_list]
print(f"[CHECK] label counts (graphs): {pd.Series(lbl).value_counts().to_dict()}")

# ========== Save ==========
ROOT.mkdir(parents=True, exist_ok=True)
torch.save(data_list, SAVE_PATH)
print(f"[OK] saved: {SAVE_PATH}")

# 
idx_df = pd.DataFrame({
    "SMILES":        [d.smiles for d in data_list],
    "Label":         [int(d.y.item()) for d in data_list],
    "SampleWeight":  [float(d.sample_weight.item()) for d in data_list],
    "CID":           [int(getattr(d, "cid", -1)) for d in data_list],
})
idx_df.to_csv(INDEX_CSV, index=False)
print(f"[OK] index saved: {INDEX_CSV}")

# pos_weight ()
n_pos = (idx_df["Label"]==1).sum()
n_neg = (idx_df["Label"]==0).sum()
pos_w = (n_neg / max(n_pos,1)) if n_pos>0 else None
print(f"[INFO] pos_weight (neg/pos): {round(pos_w,4) if pos_w else 'N/A'}")

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

# =========================
# USER SETTINGS
# =========================
USE_GOOGLE_DRIVE = True
PROJECT_ROOT = "/content/drive/MyDrive/Chemoinfo_QT"  # <- Chemoinfo_QT folder
SCRIPT_PATH = "/content/drive/MyDrive/Chemoinfo_QT/comment5_split_similarity_analysis.py"  # <- analysis script (.py)
OUTPUT_DIR_NAME = "comment5_analysis"
STRICT_RULE = "cid_or_canonical"   # "cid_or_canonical" / "cid_only" / "canonical_only"
ZIP_OUTPUT = True

def run_cmd(cmd):
    print("[CMD]", " ".join(cmd))
    subprocess.check_call(cmd)

def pip_install_one(pkg):
    run_cmd([sys.executable, "-m", "pip", "install", pkg])

# update pip first
run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

# install required packages individually
for pkg in [
    "numpy",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "tabulate",
    "rdkit",
]:
    pip_install_one(pkg)

# mount Google Drive
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

project_root = Path(PROJECT_ROOT)
script_path = Path(SCRIPT_PATH)

if not project_root.exists():
    raise FileNotFoundError(f"PROJECT_ROOT not found: {project_root}")
if not script_path.exists():
    raise FileNotFoundError(f"SCRIPT_PATH not found: {script_path}")

# Run
cmd = [
    sys.executable,
    str(script_path),
    "--root", str(project_root),
    "--output_dir", OUTPUT_DIR_NAME,
    "--strict_overlap_rule", STRICT_RULE,
]

print("[RUN]", " ".join(cmd))
subprocess.check_call(cmd)

# zip the results
output_dir = project_root / OUTPUT_DIR_NAME
if not output_dir.exists():
    raise FileNotFoundError(f"Output directory not found after run: {output_dir}")

if ZIP_OUTPUT:
    zip_base = project_root / OUTPUT_DIR_NAME
    zip_file = project_root / f"{OUTPUT_DIR_NAME}.zip"
    if zip_file.exists():
        zip_file.unlink()
    shutil.make_archive(str(zip_base), "zip", root_dir=output_dir)
    print(f"[DONE] Output zip: {zip_file}")
else:
    print(f"[DONE] Output directory: {output_dir}")

# show the summary header
summary_md = output_dir / "comment5_summary.md"
if summary_md.exists():
    print("\n===== comment5_summary.md (head) =====\n")
    txt = summary_md.read_text(encoding="utf-8", errors="ignore")
    print(txt[:4000])

print("\n===== output files =====")
for p in sorted(output_dir.iterdir()):
    print(p.name)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%run /content/drive/MyDrive/Chemoinfo_QT/strict_external_eval_full_thr055_fig2layout.py

In [ ]:
# ==== QT-M2M4-cvpr: Internal evaluation with 95% CIs, ROC/PR curves, Confusion table ====
# Project: Chemoinfo_QT
# - Input: internal graphs at ROOT/data_graph_with_smiles.pt
# - Model: cv_pr weighted ensemble of M2 + M4
# - Outputs:
#   * stats csv with point estimates + 95% bootstrap CIs (ROC-AUC, PR-AUC, F1@0.5, F1@Best)
#   * confusion tables (TP/FP/TN/FN) @0.5 and @Best-F1 threshold
#   * ROC/PR curves (PNG + CSV)
#   * per-sample predictions CSV  (p_ens = cv_pr weighted M2+M4)
# -------------------------------------------------------------------------------------------
import os, glob, math, json, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path

# --- sklearn/scipy stack (quiet) ---
from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc, roc_auc_score,
    precision_score, recall_score, f1_score
)
from sklearn.exceptions import UndefinedMetricWarning
import warnings
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# matplotlib backend for headless
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------- Torch Geometric ----------
try:
    from torch_geometric.loader import DataLoader
    from torch_geometric.data import Data
    from torch_geometric.nn import TransformerConv, NNConv, AttentionalAggregation, GATv2Conv
    from torch.serialization import add_safe_globals
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    from torch_geometric.loader import DataLoader
    from torch_geometric.data import Data
    from torch_geometric.nn import TransformerConv, NNConv, AttentionalAggregation, GATv2Conv
    from torch.serialization import add_safe_globals

# ====== CONFIG ======
ROOT    = "/content/drive/MyDrive/Chemoinfo_QT"
DATA_PT = f"{ROOT}/data_graph_with_smiles.pt"   # internal(Training/Validation)

# --- Family dirs (M2 + M4) ---
M2_DIR  = f"{ROOT}/results_ens_trans_gatv2_scaffold5fold"
M4_DIR  = f"{ROOT}/results_lco5_trans_gat_ens_posaug_advanced"

OUT_DIR = f"{ROOT}/reports_internal_QT-M2M4-cvpr"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH = 256
FIXED_THR = 0.5           # threshold for confusion matrices and fixed-threshold F1
N_BOOT = 2000             # number of bootstrap iterations(95%CI)
SEED = 42                 # for reproducibility

# ====== PyG helpers ======
def load_pyg_list(path):
    try:
        from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
        from torch_geometric.data.storage import GlobalStorage
        add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
    except Exception:
        pass
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    return list(obj)

def dims_from_sample(d):
    x_dim = d.x.size(-1)
    e_dim = d.edge_attr.size(-1)
    g_dim = d.g.size(-1) if d.g.dim() in (2,3) else d.g.numel()
    r_dim = d.r.size(-1) if d.r.dim() in (2,3) else d.r.numel()
    return x_dim, e_dim, g_dim, r_dim

# ====== Models ======
def make_edge_mlp(in_attr_dim, out_channels, in_channels):
    hid = max(64, min(256, in_attr_dim*8))
    return nn.Sequential(nn.Linear(in_attr_dim,hid), nn.ReLU(),
                         nn.Linear(hid, in_channels*out_channels))

class Transformer_DX_DG_DR(nn.Module):
    def __init__(self,X_DIM,G_DIM,R_DIM,E_DIM,HID=256):
        super().__init__()
        self.c1=TransformerConv(X_DIM,HID,heads=4,concat=False,edge_dim=E_DIM,dropout=0.0,beta=False)
        self.c2=TransformerConv(HID, HID,heads=4,concat=False,edge_dim=E_DIM,dropout=0.0,beta=False)
        self.pool=AttentionalAggregation(gate_nn=nn.Linear(HID,1))
        self.drop=nn.Dropout(0.2)
        self.fc=nn.Linear(HID+G_DIM+R_DIM,1)
    def forward(self,d):
        x=F.relu(self.c1(d.x,d.edge_index,d.edge_attr))
        x=F.relu(self.c2(x,d.edge_index,d.edge_attr))
        x=self.pool(self.drop(x), d.batch)
        g=d.g.squeeze(1) if d.g.dim()==3 else d.g
        r=d.r.squeeze(1) if d.r.dim()==3 else d.r
        return self.fc(torch.cat([x,g,r],dim=1)).view(-1)

class NNConv_DX_DG_DR(nn.Module):
    def __init__(self,X_DIM,G_DIM,R_DIM,E_DIM,HID=256):
        super().__init__()
        e1=make_edge_mlp(E_DIM,HID,X_DIM); e2=make_edge_mlp(E_DIM,HID,HID)
        self.c1=NNConv(X_DIM,HID,e1,aggr='mean'); self.c2=NNConv(HID,HID,e2,aggr='mean')
        self.pool=AttentionalAggregation(gate_nn=nn.Linear(HID,1))
        self.drop=nn.Dropout(0.2)
        self.fc=nn.Linear(HID+G_DIM+R_DIM,1)
    def forward(self,d):
        x=F.relu(self.c1(d.x,d.edge_index,d.edge_attr))
        x=F.relu(self.c2(x,d.edge_index,d.edge_attr))
        x=self.pool(self.drop(x), d.batch)
        g=d.g.squeeze(1) if d.g.dim()==3 else d.g
        r=d.r.squeeze(1) if d.r.dim()==3 else d.r
        return self.fc(torch.cat([x,g,r],dim=1)).view(-1)

# GATv2 heads=1
class GATv2_DX_DG_DR(nn.Module):
    def __init__(self, X_DIM, G_DIM, R_DIM, E_DIM, HID=256):
        super().__init__()
        self.g1 = GATv2Conv(X_DIM, HID, edge_dim=E_DIM, dropout=0.0)              # heads=1
        self.g2 = GATv2Conv(HID,   HID, edge_dim=E_DIM, dropout=0.0)
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID, 1))
        self.drop = nn.Dropout(0.2)
        self.fc   = nn.Linear(HID + G_DIM + R_DIM, 1)
    def forward(self, d):
        x = F.elu(self.g1(d.x, d.edge_index, d.edge_attr))
        x = F.elu(self.g2(x,   d.edge_index, d.edge_attr))
        x = self.pool(self.drop(x), d.batch)
        g = d.g.squeeze(1) if d.g.dim()==3 else d.g
        r = d.r.squeeze(1) if d.r.dim()==3 else d.r
        return self.fc(torch.cat([x, g, r], dim=1)).view(-1)

# GATv2 heads=4 (concat=True -> output dimension 1024)
class GATv2_DX_DG_DR_H4(nn.Module):
    def __init__(self, X_DIM, G_DIM, R_DIM, E_DIM, HID=256):
        super().__init__()
        self.g1 = GATv2Conv(X_DIM, HID, heads=4, concat=True, edge_dim=E_DIM, dropout=0.0)
        self.g2 = GATv2Conv(HID*4, HID, heads=4, concat=True, edge_dim=E_DIM, dropout=0.0)
        self.pool = AttentionalAggregation(gate_nn=nn.Linear(HID*4, 1))
        self.drop = nn.Dropout(0.2)
        self.fc   = nn.Linear(HID*4 + G_DIM + R_DIM, 1)
    def forward(self, d):
        x = F.elu(self.g1(d.x, d.edge_index, d.edge_attr))   # [N, 1024]
        x = F.elu(self.g2(x,   d.edge_index, d.edge_attr))   # [N, 1024]
        x = self.pool(self.drop(x), d.batch)                 # [B, 1024]
        g = d.g.squeeze(1) if d.g.dim()==3 else d.g
        r = d.r.squeeze(1) if d.r.dim()==3 else d.r
        return self.fc(torch.cat([x, g, r], dim=1)).view(-1)

# ====== state_dict detection and safe loading ======
def detect_gatv2_heads_from_sd(sd: dict) -> int:
    for k, v in sd.items():
        if isinstance(v, torch.Tensor) and k.endswith("att") and v.dim() == 3:
            return int(v.shape[1])  # [1, heads, HID]
    return 1

def build_model_for_sd(sd: dict, dims):
    X_DIM, E_DIM, G_DIM, R_DIM = dims
    keys = list(sd.keys())
    # GATv2?
    if any(("g1." in k and "gat" in "_".join(keys).lower()) for k in keys):
        heads = detect_gatv2_heads_from_sd(sd)
        return (GATv2_DX_DG_DR_H4 if heads >= 4 else GATv2_DX_DG_DR)(X_DIM, G_DIM, R_DIM, E_DIM)
    # Transformer?
    if any(("c1." in k and "lin_edge" in k) for k in keys):
        return Transformer_DX_DG_DR(X_DIM, G_DIM, R_DIM, E_DIM)
    # fallback: NNConv 
    return NNConv_DX_DG_DR(X_DIM, G_DIM, R_DIM, E_DIM)

def safe_load_compatible_tensors(model: nn.Module, sd: dict):
    msd = model.state_dict()
    filt = {k: v for k, v in sd.items() if (k in msd and isinstance(v, torch.Tensor) and msd[k].shape == v.shape)}
    if len(filt) < len(msd):
        print(f"[WARN] load compatible tensors {len(filt)}/{len(msd)}; skip {len(msd)-len(filt)} keys")
    model.load_state_dict(filt, strict=False)
    return model

# ====== Family utilities ======
def list_weight_files(folder, patterns):
    files = []
    for pat in patterns:
        files += sorted(glob.glob(os.path.join(folder, pat)))
    return files

def fold_id(p):
    import re, os
    bn=os.path.basename(p).lower()
    m=re.findall(r"fold(\d+)", bn)
    return int(m[-1]) if m else -1

def infer_cv_pr_weight(csv_paths):
    if not csv_paths: return None
    vals=[]
    for p in csv_paths:
        try:
            df=pd.read_csv(p)
            for c in ["PR-AUC","Ensemble_PR","PR","Val PR-AUC","Ensemble_PR:","PR_auc","val_pr"]:
                if c in df.columns:
                    s=pd.to_numeric(df[c], errors="coerce")
                    vals += s[~s.isna()].tolist()
        except Exception:
            pass
    return float(np.nanmean(vals)) if vals else None

@torch.no_grad()
def family_probs(data_list, family_dir, dims):
    """Return mean probs across folds/models for a family, and its CV-PR weight."""
    X_DIM,E_DIM,G_DIM,R_DIM = dims
    tr_files = list_weight_files(family_dir, ["transformer_fold*.pt","*trans*fold*.pt"])
    nn_files = list_weight_files(family_dir, ["nnconv_fold*.pt","*nn*fold*.pt"])
    ga_files = list_weight_files(family_dir, ["*gatv2*fold*.pt","*gat*fold*.pt"])
    files = sorted(set(tr_files + nn_files + ga_files))
    assert files, f"No weight files found in: {family_dir}"

    # cv_pr weight
    fam_w = infer_cv_pr_weight(sorted(glob.glob(os.path.join(family_dir,"*.csv")))) or 1.0

    # fold grouping
    fmap = {}
    for p in files:
        f = fold_id(p)
        fmap.setdefault(f, []).append(p)

    dl=DataLoader(data_list, batch_size=BATCH, shuffle=False)
    fold_probs=[]
    for f, plist in sorted(fmap.items()):
        probs=[]
        for wt in plist:
            sd = torch.load(wt, map_location="cpu")
            if isinstance(sd, dict) and "state_dict" in sd:
                sd = sd["state_dict"]
            model = build_model_for_sd(sd, dims).to(DEVICE)
            model = safe_load_compatible_tensors(model, sd).eval()
            ps=[]
            for bt in dl:
                bt = bt.to(DEVICE)
                ps.append(torch.sigmoid(model(bt)).detach().cpu())
            probs.append(torch.cat(ps).numpy())
            del model
        if probs:
            fold_probs.append(np.mean(np.stack(probs,axis=0),axis=0))
    fam_probs=np.mean(np.stack(fold_probs,axis=0),axis=0)
    return fam_probs, float(fam_w)

def evaluable_subset(data_list):
    """Return indices, labels, smiles for samples with y in {0,1}."""
    idx=[i for i,d in enumerate(data_list) if int(getattr(d,"y",-1)) in (0,1)]
    if not idx:
        raise RuntimeError("No evaluable samples (y in {0,1}).")
    y=np.array([int(getattr(data_list[i],'y')) for i in idx], dtype=int)
    smiles=[getattr(data_list[i],"smiles","") for i in idx]
    return idx, y, smiles

# ====== metrics & CI ======
def pr_auc(y,p):
    prec, rec, _ = precision_recall_curve(y, p)
    return float(auc(rec, prec))

def roc_auc(y,p):
    try: return float(roc_auc_score(y,p))
    except Exception: return float("nan")

def f1_at(y,p,thr):
    return float(f1_score(y,(p>=thr).astype(int),zero_division=0))

def best_f1(y,p):
    prec,rec,thr=precision_recall_curve(y,p)
    thr=np.append(thr,1.0)
    f1=2*prec*rec/np.clip(prec+rec,1e-9,None)
    i=int(np.nanargmax(f1))
    return float(thr[i]), float(f1[i]), float(prec[i]), float(rec[i])

_compute_best_f1 = best_f1

def confusion_counts(y,p,thr):
    pred=(p>=thr).astype(int)
    tp=int(((pred==1)&(y==1)).sum())
    fp=int(((pred==1)&(y==0)).sum())
    tn=int(((pred==0)&(y==0)).sum())
    fn=int(((pred==0)&(y==1)).sum())
    return tp,fp,tn,fn

def bootstrap_ci(metric_fn, y, p, n_boot=2000, seed=42, stratified=True):
    """Return (point, lo95, hi95) using bootstrap. metric_fn(y, p) -> float"""
    rng = np.random.default_rng(seed)
    n=len(y)
    if stratified:
        pos_idx=np.where(y==1)[0]; neg_idx=np.where(y==0)[0]
        n_pos=len(pos_idx); n_neg=len(neg_idx)
    vals=[]
    for _ in range(n_boot):
        if stratified and n_pos>0 and n_neg>0:
            samp_pos = rng.choice(pos_idx, size=n_pos, replace=True)
            samp_neg = rng.choice(neg_idx, size=n_neg, replace=True)
            samp = np.concatenate([samp_pos, samp_neg])
        else:
            samp = rng.choice(np.arange(n), size=n, replace=True)
        yy=y[samp]; pp=p[samp]
        try:
            vals.append(float(metric_fn(yy,pp)))
        except Exception:
            continue
    vals=np.array(vals, dtype=float)
    point=float(metric_fn(y,p))
    lo=float(np.nanpercentile(vals, 2.5)) if len(vals)>0 else float("nan")
    hi=float(np.nanpercentile(vals,97.5)) if len(vals)>0 else float("nan")
    return point, lo, hi

# ====== curves saving ======
def save_roc(out_png, out_csv, y, p, title):
    if np.unique(y).size<2:
        print(f"[SKIP] ROC undefined (single-class labels). {title}")
        return None
    fpr,tpr,thr = roc_curve(y,p)
    roc = roc_auc_score(y,p)
    pd.DataFrame({"fpr":fpr,"tpr":tpr,"threshold":thr}).to_csv(out_csv, index=False)
    plt.figure()
    plt.plot(fpr,tpr,label=f"AUC={roc:.4f}")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(title); plt.legend(); plt.tight_layout()
    plt.savefig(out_png, dpi=160); plt.close()
    print(f"[ROC] {title} -> {out_png} | {out_csv}")
    return roc

def save_pr(out_png, out_csv, y, p, title):
    prec,rec,thr = precision_recall_curve(y,p)
    pr = auc(rec,prec)
    pd.DataFrame({"recall":rec, "precision":prec, "threshold":np.append(thr,np.nan)}).to_csv(out_csv, index=False)
    plt.figure()
    plt.plot(rec,prec,label=f"AUC={pr:.4f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(title); plt.legend(); plt.tight_layout()
    plt.savefig(out_png, dpi=160); plt.close()
    print(f"[PR ] {title} -> {out_png} | {out_csv}")
    return pr

# ====== run ======
def main():
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

    # Load internal data
    assert Path(DATA_PT).exists(), f"not found: {DATA_PT}"
    data_list = load_pyg_list(DATA_PT)
    idx, y, smiles = evaluable_subset(data_list)
    data_eval = [data_list[i] for i in idx]
    y = np.asarray(y, dtype=int)
    print(f"[INFO] Internal evaluable N={len(y)} | positives={y.sum()} | negatives={(y==0).sum()}")

    # Dims
    X_DIM,E_DIM,G_DIM,R_DIM = dims_from_sample(data_eval[0])
    dims = (X_DIM,E_DIM,G_DIM,R_DIM)
    print("[INFO] dims:", {"x":X_DIM,"edge":E_DIM,"g":G_DIM,"r":R_DIM})

    # --- Family probs & cv_pr weights (M2 + M4) ---
    print("🔎 Family directories:")
    print("  M2:", M2_DIR)
    print("  M4:", M4_DIR)
    pM2, wM2 = family_probs(data_eval, M2_DIR, dims)
    pM4, wM4 = family_probs(data_eval, M4_DIR, dims)

    # cv_pr normalization
    w2 = wM2 if np.isfinite(wM2) else 1.0
    w4 = wM4 if np.isfinite(wM4) else 1.0
    s  = w2 + w4 if (w2 + w4) > 0 else 1.0
    a2, a4 = w2/s, w4/s
    p = a2 * pM2 + a4 * pM4
    print(f"[INFO] Using M2+M4 (cv_pr) | alpha_M2={a2:.3f}, alpha_M4={a4:.3f}  (raw w: M2={wM2:.3f}, M4={wM4:.3f})")

    # Save per-sample predictions
    df_pred = pd.DataFrame({
        "SMILES": smiles,
        "y_true": y,
        "p_M2": pM2,
        "p_M4": pM4,
        "alpha_M2": a2,
        "alpha_M4": a4,
        "p_ens": p,   # final ensemble probability
    })
    pred_csv = f"{OUT_DIR}/predictions_internal_QT-M2M4-cvpr.csv"
    df_pred.to_csv(pred_csv, index=False)

    # ====== CIDaggregation(CID)internalEvaluationAdditional ======
    INDEX_CSV = f"{ROOT}/data_graph_with_smiles_index.csv"
    assert Path(INDEX_CSV).exists(), f"not found: {INDEX_CSV}"
    idx_df = pd.read_csv(INDEX_CSV)

    # assuming aligned row order: evaluable_subset idx  index filter the index side as well
    idx_df_sub = idx_df.iloc[idx].copy()

    if "Label" in idx_df_sub.columns:
        y_from_index = pd.to_numeric(idx_df_sub["Label"], errors="coerce").astype("Int64")
        mismatch = (y_from_index.notna()) & (y_from_index.astype(int).to_numpy() != y)
        if mismatch.sum() > 0:
            print(f"[WARN] index Label mismatch count = {int(mismatch.sum())} (check row alignment)")

    assert "CID" in idx_df_sub.columns, "index csv must contain CID column"

    df_eval = df_pred.copy()
    df_eval["CID"] = idx_df_sub["CID"].to_numpy()

    cid_agg = (
        df_eval.groupby("CID", dropna=False)
              .agg(y_true=("y_true", "max"),
                   p_ens=("p_ens", "mean"),
                   p_M2=("p_M2", "mean"),
                   p_M4=("p_M4", "mean"),
                   SMILES=("SMILES", "first"))
              .reset_index()
    )

    y_cid = cid_agg["y_true"].to_numpy(dtype=int)
    p_cid = cid_agg["p_ens"].to_numpy(dtype=float)

    print(f"[INFO] CID-aggregated N={len(y_cid)} | positives={int(y_cid.sum())} | negatives={int((y_cid==0).sum())}")

    roc_png_cid = f"{OUT_DIR}/roc_internal_m2m4_cvpr_CIDmean.png"
    roc_csv_cid = f"{OUT_DIR}/roc_internal_m2m4_cvpr_CIDmean.csv"
    pr_png_cid  = f"{OUT_DIR}/pr_internal_m2m4_cvpr_CIDmean.png"
    pr_csv_cid  = f"{OUT_DIR}/pr_internal_m2m4_cvpr_CIDmean.csv"

    _ = save_roc(roc_png_cid, roc_csv_cid, y_cid, p_cid, "QT-M2M4-cvpr - INTERNAL ROC (CID mean)")
    _ = save_pr (pr_png_cid , pr_csv_cid , y_cid, p_cid, "QT-M2M4-cvpr - INTERNAL PR (CID mean)")

    roc_point_c, roc_lo_c, roc_hi_c = bootstrap_ci(roc_auc, y_cid, p_cid, n_boot=N_BOOT, seed=SEED, stratified=True)
    pr_point_c , pr_lo_c , pr_hi_c  = bootstrap_ci(pr_auc , y_cid, p_cid, n_boot=N_BOOT, seed=SEED, stratified=True)
    f1_fixed_point_c, f1_fixed_lo_c, f1_fixed_hi_c = bootstrap_ci(
        lambda yy,pp: f1_at(yy,pp,FIXED_THR), y_cid, p_cid, n_boot=N_BOOT, seed=SEED, stratified=True
    )
    def f1_best_metric_c(yy,pp):
        _, f1b, _, _ = best_f1(yy,pp); return f1b
    f1_best_point_c, f1_best_lo_c, f1_best_hi_c = bootstrap_ci(
        f1_best_metric_c, y_cid, p_cid, n_boot=N_BOOT, seed=SEED, stratified=True
    )

    best_thr_c, best_f1_val_c, best_prec_c, best_rec_c = best_f1(y_cid, p_cid)
    tp_c, fp_c, tn_c, fn_c     = confusion_counts(y_cid, p_cid, FIXED_THR)
    tpb_c, fpb_c, tnb_c, fnb_c = confusion_counts(y_cid, p_cid, best_thr_c)

    df_stats_cid = pd.DataFrame([
        ["ROC-AUC", roc_point_c, roc_lo_c, roc_hi_c, ""],
        ["PR-AUC",  pr_point_c,  pr_lo_c,  pr_hi_c,  ""],
        [f"F1@{FIXED_THR:.2f}", f1_fixed_point_c, f1_fixed_lo_c, f1_fixed_hi_c, f"thr={FIXED_THR}"],
        ["F1@Best", f1_best_point_c, f1_best_lo_c, f1_best_hi_c, f"thr≈{best_thr_c:.4f}, Prec≈{best_prec_c:.4f}, Rec≈{best_rec_c:.4f}"],
    ], columns=["metric","point","ci_2.5","ci_97.5","note"])

    stats_csv_cid = f"{OUT_DIR}/stats_internal_QT-M2M4-cvpr_CIDmean.csv"
    df_stats_cid.to_csv(stats_csv_cid, index=False)
    print("[SAVE] stats (CIDmean) ->", stats_csv_cid)

    df_cm_cid = pd.DataFrame([
        ["@fixed", FIXED_THR, tp_c, fp_c, tn_c, fn_c],
        ["@bestF1", best_thr_c, tpb_c, fpb_c, tnb_c, fnb_c],
    ], columns=["mode","threshold","TP","FP","TN","FN"])
    cm_csv_cid = f"{OUT_DIR}/confusion_internal_QT-M2M4-cvpr_CIDmean.csv"
    df_cm_cid.to_csv(cm_csv_cid, index=False)
    print("[SAVE] confusion (CIDmean) ->", cm_csv_cid)

    cid_pred_csv = f"{OUT_DIR}/predictions_internal_QT-M2M4-cvpr_CIDmean.csv"
    cid_agg.to_csv(cid_pred_csv, index=False)
    print("[SAVE] predictions (CIDmean) ->", cid_pred_csv)
    # ====== end of CID-level aggregation block ======

    # Curves
    roc_png = f"{OUT_DIR}/roc_internal_m2m4_cvpr.png"
    roc_csv = f"{OUT_DIR}/roc_internal_m2m4_cvpr.csv"
    pr_png  = f"{OUT_DIR}/pr_internal_m2m4_cvpr.png"
    pr_csv  = f"{OUT_DIR}/pr_internal_m2m4_cvpr.csv"
    roc_auc_val = save_roc(roc_png, roc_csv, y, p, "QT-M2M4-cvpr - INTERNAL ROC")
    pr_auc_val  = save_pr (pr_png , pr_csv , y, p, "QT-M2M4-cvpr - INTERNAL PR")

    # Point metrics + 95% CIs (bootstrap, stratified)
    roc_point, roc_lo, roc_hi = bootstrap_ci(roc_auc, y, p, n_boot=N_BOOT, seed=SEED, stratified=True)
    pr_point , pr_lo , pr_hi  = bootstrap_ci(pr_auc , y, p, n_boot=N_BOOT, seed=SEED, stratified=True)
    # F1 @ fixed threshold
    f1_fixed_point, f1_fixed_lo, f1_fixed_hi = bootstrap_ci(lambda yy,pp: f1_at(yy,pp,FIXED_THR),
                                                            y, p, n_boot=N_BOOT, seed=SEED, stratified=True)
    # Best-F1(re-optimize the threshold within each bootstrap sample)
    def f1_best_metric(yy,pp):
        _, f1b, _, _ = best_f1(yy,pp); return f1b
    f1_best_point, f1_best_lo, f1_best_hi = bootstrap_ci(f1_best_metric, y, p, n_boot=N_BOOT, seed=SEED, stratified=True)

    # Confusion tables @0.5 and @Best-F1 threshold(BestThr determined on the full dataset)
    best_thr, best_f1_val, best_prec, best_rec = best_f1(y,p)
    tp,fp,tn,fn = confusion_counts(y,p,FIXED_THR)
    tp_b,fp_b,tn_b,fn_b = confusion_counts(y,p,best_thr)

    # Save metrics table
    rows = [
        ["ROC-AUC", roc_point, roc_lo, roc_hi, ""],
        ["PR-AUC",  pr_point,  pr_lo,  pr_hi,  ""],
        [f"F1@{FIXED_THR:.2f}", f1_fixed_point, f1_fixed_lo, f1_fixed_hi, f"thr={FIXED_THR}"],
        ["F1@Best", f1_best_point, f1_best_lo, f1_best_hi, f"thr≈{best_thr:.4f}, Prec≈{best_prec:.4f}, Rec≈{best_rec:.4f}"],
    ]
    df_stats = pd.DataFrame(rows, columns=["metric","point","ci_2.5","ci_97.5","note"])
    stats_csv = f"{OUT_DIR}/stats_internal_QT-M2M4-cvpr.csv"
    df_stats.to_csv(stats_csv, index=False)
    print("[SAVE] stats ->", stats_csv)

    # Save confusion tables
    df_cm = pd.DataFrame([
        ["@fixed", FIXED_THR, tp,fp,tn,fn],
        ["@bestF1", best_thr, tp_b,fp_b,tn_b,fn_b],
    ], columns=["mode","threshold","TP","FP","TN","FN"])
    cm_csv = f"{OUT_DIR}/confusion_internal_QT-M2M4-cvpr.csv"
    df_cm.to_csv(cm_csv, index=False)
    print("[SAVE] confusion ->", cm_csv)

    print("\n=== SUMMARY ===")
    print(df_stats.to_string(index=False))
    print("\nConfusion (counts):")
    print(df_cm.to_string(index=False))
    print(f"\nCurves saved to: {roc_png} , {pr_png}")
    print(f"Predictions: {pred_csv}")

if __name__=="__main__":
    main()

In [ ]:
# ============================================================
# Drug-level (pre-augmentation) Confusion Matrices  — Chemoinfo_QT
#   - Aggregates instance-level predictions per drug (CID or canonical SMILES)
#   - Models: (A) M4-HYB-like (auto-discover)  (B) QT-M2M4-cvpr (fixed)
# Inputs:
#   ROOT/data_graph_with_smiles.pt   (provides SMILES, y_true, cid/keys)
#   ROOT/reports_internal_*/predictions_internal_*.csv   (auto-discover)
# Outputs:
#   ROOT/figs_confusions_drug/
#     - confusion_drug_<model>.csv / .png
#     - confusion_drug_both_summary.csv  (present if 2both models available)
# Notes:
#   - Group key priority: CID > smiles_orig > canonical_smiles(RDKit) > raw SMILES
#   - Prob aggregation: mean (median optional)
#   - y_true aggregation: mode (warn on conflicts)
# ============================================================

import os, sys, subprocess, json, glob
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------- Config ----------
ROOT   = "/content/drive/MyDrive/Chemoinfo_QT"
DATA_PT= f"{ROOT}/data_graph_with_smiles.pt"

# Model B: QT-M2M4-cvpr(use a fixed path)
CSV_B   = f"{ROOT}/reports_internal_QT-M2M4-cvpr/predictions_internal_QT-M2M4-cvpr.csv"
NAME_B  = "QT-M2M4-cvpr"
PROB_B  = "p_ens"

# Model A: M4 single-model(HYB family)auto-discover(skip if missing)
# candidate filename patterns
CAND_A_PATTERNS = [
    f"{ROOT}/reports_internal_*M4*HYB*/predictions_internal_*M4*HYB*.csv",
    f"{ROOT}/reports_internal_*M4*hyb*/predictions_internal_*M4*hyb*.csv",
    f"{ROOT}/reports_internal_*M4*/predictions_internal_*M4*.csv",
]
NAME_A_FALLBACK = "M4-HYB-like"
PROB_A_DEFAULT  = "p_ens"   # switch to p_M4 automatically for single-model M4 CSV files

THRESH   = 0.5455
AGG_PROB = "mean"      # 'mean' or 'median'
OUTDIR   = f"{ROOT}/figs_confusions_drug"
os.makedirs(OUTDIR, exist_ok=True)

# ---------- Safe deps ----------
def _pip(cmd):
    try:
        subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

def _ensure(mod, pip_name=None):
    try:
        __import__(mod); return True
    except Exception:
        return _pip([sys.executable, "-m", "pip", "install", "-q", pip_name or mod])

_ensure("torch", "torch")
_ensure("scikit-learn", "scikit-learn")
import torch
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
from torch.serialization import add_safe_globals
try:
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
    add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
except Exception:
    pass

# RDKit (for canonical SMILES grouping if no CID)
HAVE_RDKIT = _ensure("rdkit", "rdkit")
if HAVE_RDKIT:
    from rdkit import Chem

# ---------- Utility: auto-discover Model A CSV ----------
def discover_model_a_csv():
    found = []
    for pat in CAND_A_PATTERNS:
        found.extend(glob.glob(pat))
    if not found:
        print("[INFO] Model-A CSV not found. Will skip Model-A and run Model-B only.")
        return None, None, None
    # use the most recent file
    found = sorted(found, key=lambda p: os.path.getmtime(p), reverse=True)
    csv = found[0]
    # derive the model name from the path
    bn = os.path.basename(csv).replace("predictions_internal_", "").replace(".csv","")
    name = bn
    # infer the probability column: p_ens ,  p_M4 ,  p search
    cols = pd.read_csv(csv, nrows=1).columns.tolist()
    if "p_ens" in cols:
        prob_col = "p_ens"
    elif "p_M4" in cols:
        prob_col = "p_M4"
    elif "p" in cols:
        prob_col = "p"
    else:
        # last resort: look for a probability-like column
        prob_col = next((c for c in cols if c.lower().startswith("p_")), None)
        if prob_col is None:
            raise ValueError(f"[Model-A] probability column not found in CSV: {csv} | cols={cols}")
    print(f"[INFO] Using Model-A CSV: {csv}  (prob='{prob_col}', name='{name}')")
    return csv, name, prob_col

# ---------- Load base (SMILES, y_true, cid, optional orig key) ----------
def load_pyg_list(path):
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    return list(obj)

assert os.path.exists(DATA_PT), f"Missing: {DATA_PT}"
dl = load_pyg_list(DATA_PT)

def _get_first_attr(d, keys, default=None):
    for k in keys:
        if hasattr(d, k):
            v = getattr(d, k)
            if v is not None:
                return v
    return default

rows = []
for d in dl:
    smi = getattr(d, "smiles", "")
    yv  = int(getattr(d, "y", -1))
    yv  = yv if yv in (0,1) else -1
    # collect CID from multiple attribute names
    cid = _get_first_attr(d, ["cid","CID","pubchem_cid","PubChem_CID"], None)
    smi_orig = _get_first_attr(d, ["smiles_orig","smiles_original","SMILES_orig"], None)
    rows.append({"SMILES": smi, "y_true": yv, "cid": cid, "smiles_orig": smi_orig})

base = pd.DataFrame(rows)
base = base[base["y_true"].isin([0,1])].copy()

# ---------- Build grouping key ----------
def canonical_smiles(s):
    if not HAVE_RDKIT or not isinstance(s, str): return None
    try:
        m = Chem.MolFromSmiles(s)
        if m is None: return None
        return Chem.MolToSmiles(m, isomericSmiles=False, canonical=True)
    except Exception:
        return None

if base["cid"].notna().any():
    base["group_key"] = base["cid"].astype("Int64").astype(str)
    group_tag = "CID"
elif base["smiles_orig"].notna().any():
    base["group_key"] = base["smiles_orig"].astype(str)
    group_tag = "smiles_orig"
else:
    if HAVE_RDKIT:
        base["group_key"] = base["SMILES"].map(canonical_smiles)
        # fallback: fall back to raw SMILES if canonicalization fails
        mask_nan = base["group_key"].isna()
        base.loc[mask_nan, "group_key"] = base.loc[mask_nan, "SMILES"].astype(str)
        group_tag = "canonical_smiles"
    else:
        base["group_key"] = base["SMILES"].astype(str)
        group_tag = "raw_smiles"

print(f"[INFO] Grouping by: {group_tag}")

# ---------- Helper: aggregate to drug-level ----------
def aggregate_drug_level(pred_csv, prob_col):
    assert os.path.exists(pred_csv), f"Missing CSV: {pred_csv}"
    pred = pd.read_csv(pred_csv)
    if "SMILES" not in pred.columns:
        raise ValueError(f"CSV needs 'SMILES' column. Columns: {list(pred.columns)}")
    if prob_col not in pred.columns:
        raise ValueError(f"Missing prob column '{prob_col}'. Columns: {list(pred.columns)}")

    df = base.merge(pred[["SMILES", prob_col]], on="SMILES", how="inner")
    if df.empty:
        raise RuntimeError("Empty merge by SMILES. Check keys/CSV.")

    # y_true aggregate at the drug level(mode).use the mode and warn if conflicts are detected
    def agg_mode(arr):
        vals, cnts = np.unique(arr, return_counts=True)
        pick = vals[np.argmax(cnts)]
        if len(vals) > 1:
            print(f"[WARN] y_true conflict in a group -> using mode={pick} (counts {dict(zip(vals, cnts))})")
        return int(pick)

    # probability aggregation
    if AGG_PROB == "median":
        prob_agg = lambda x: float(np.median(pd.to_numeric(x, errors="coerce").fillna(0.0).values))
    else:
        prob_agg = lambda x: float(np.mean(pd.to_numeric(x, errors="coerce").fillna(0.0).values))

    g = df.groupby("group_key", as_index=False).agg(
        y_true=("y_true", agg_mode),
        y_prob=(prob_col, prob_agg),
        n_instances=("SMILES", "count"),
    )
    return g

# ---------- Confusion + plot ----------
def confusion_and_plot(gdf, model_name, outdir=OUTDIR, thresh=THRESH):
    y_true = gdf["y_true"].values.astype(int)
    y_prob = gdf["y_prob"].values
    y_pred = (y_prob >= thresh).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_true, y_pred)
    p_pos, r_pos, f1_pos, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

    # CSV
    cm_df = pd.DataFrame(cm, index=["True_0","True_1"], columns=["Pred_0","Pred_1"])
    cm_csv = os.path.join(outdir, f"confusion_drug_{model_name.replace('/','-')}.csv")
    cm_df.to_csv(cm_csv, index=True)

    # PNG
    plt.figure(figsize=(4.2, 3.8))
    im = plt.imshow(cm, cmap="Blues")
    cb = plt.colorbar(im, fraction=0.046, pad=0.04); cb.set_label("Count")
    plt.xticks([0,1], ["Pred 0", "Pred 1"])
    plt.yticks([0,1], ["True 0", "True 1"])
    for i in range(2):
        for j in range(2):
            val = cm[i,j]
            plt.text(j, i, f"{val:,}", ha="center", va="center",
                     fontsize=12, color=("black" if val < cm.max()*0.6 else "white"))
    plt.title(f"{model_name} (Drug-level)\nthr={thresh:.2f}  Acc={acc:.3f}  P={p_pos:.3f}  R={r_pos:.3f}  F1={f1_pos:.3f}")
    plt.tight_layout()
    cm_png = os.path.join(outdir, f"confusion_drug_{model_name.replace('/','-')}.png")
    plt.savefig(cm_png, dpi=160); plt.close()

    return {
        "Model": f"{model_name} (drug-level)",
        "GroupKey": group_tag,
        "N_drugs": int(len(gdf)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Accuracy": float(acc),
        "Precision_pos": float(p_pos),
        "Recall_pos": float(r_pos),
        "F1_pos": float(f1_pos),
        "Threshold": float(thresh),
        "ProbAgg": AGG_PROB,
        "CSV": cm_csv,
        "PNG": cm_png,
    }

# ---------- Run ----------
def main():
    # Model-B fixed(M2+M4 ensemble)
    assert os.path.exists(CSV_B), f"[B] Missing prediction CSV: {CSV_B}"
    gB = aggregate_drug_level(CSV_B, PROB_B)
    sumB = confusion_and_plot(gB, NAME_B)

    # Model-A automaticsearch( B only)
    csvA, nameA, probA = discover_model_a_csv()
    results = []
    if csvA is not None:
        gA = aggregate_drug_level(csvA, probA)
        sumA = confusion_and_plot(gA, nameA or NAME_A_FALLBACK)
        results = [sumA, sumB]
    else:
        results = [sumB]

    # summary CSV
    pd.DataFrame(results).to_csv(os.path.join(OUTDIR, "confusion_drug_summary.csv"), index=False)

    # Output()
    if len(results) == 2:
        pd.DataFrame(results).to_csv(os.path.join(OUTDIR, "confusion_drug_both_summary.csv"), index=False)

    print(json.dumps({"results": results}, indent=2))
    print(f"\n[OK] Done. Saved under: {OUTDIR}")

if __name__ == "__main__":
    main()

In [ ]:
# search_threshold_precision_recall90_internal.py
# Internal (CID-level) threshold search for p_ens:
#   (A) Precision target:
#       - First try: Precision_pos >= 0.90 with PredPos >= MIN_PREDPOS_PREC
#       - If not found: pick threshold that MAXIMIZES Precision_pos under PredPos >= MIN_PREDPOS_PREC  <-- requested behavior
#   (B) Recall target:
#       - Find threshold where Recall_pos >= 0.90 with PredPos >= MIN_PREDPOS_REC
#       - If multiple, pick best Precision (then F1, then higher Threshold)
#
# Assumption:
#   Prefer using CID-aggregated CSV:
#     reports_internal_QT-M2M4-cvpr/predictions_internal_QT-M2M4-cvpr_CIDmean.csv
#
# Outputs (under OUTDIR):
#   - threshold_scan_internal_CID.csv
#   - threshold_targets_internal_CID.json
#
# No torch required.

import os
import json
from pathlib import Path
import numpy as np
import pandas as pd


# =========================
# CONFIG
# =========================
ROOT = "/content/drive/MyDrive/Chemoinfo_QT"

# Preferred (already CID-aggregated) predictions CSV
PRED_CID_CSV = f"{ROOT}/reports_internal_QT-M2M4-cvpr/predictions_internal_QT-M2M4-cvpr_CIDmean.csv"

# Fallback: any predictions CSV that already contains CID + y_true + p_ens
PRED_FALLBACK_CSV = f"{ROOT}/reports_internal_QT-M2M4-cvpr/predictions_internal_QT-M2M4-cvpr.csv"

# Output directory
OUTDIR = f"{ROOT}/reports_internal_threshold_search"
os.makedirs(OUTDIR, exist_ok=True)

# Columns
COL_PROB = "p_ens"
COL_Y    = "y_true"
COL_CID  = "CID"

# Targets
TARGET_PREC = 0.90
TARGET_REC  = 0.90
STRICT = False  # False: >= TARGET, True: > TARGET

# --- Minimum predicted positives constraint (key change) ---
MIN_PREDPOS_PREC = 50
MIN_PREDPOS_REC  = 50


# =========================
# Helpers
# =========================
def _cmp(x: float, thr: float) -> bool:
    return (x > thr) if STRICT else (x >= thr)

def find_existing_csv() -> str:
    """Return an existing CSV path, prefer CID-mean file."""
    if Path(PRED_CID_CSV).exists():
        return PRED_CID_CSV
    if Path(PRED_FALLBACK_CSV).exists():
        return PRED_FALLBACK_CSV

    hits = sorted(Path(ROOT).rglob("predictions_internal_*CIDmean*.csv"))
    if hits:
        return str(hits[0])
    hits = sorted(Path(ROOT).rglob("predictions_internal_*.csv"))
    if hits:
        return str(hits[0])

    raise FileNotFoundError("No predictions_internal_*.csv found under ROOT.")

def ensure_cid_level(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure CID-level table with columns: CID, y_true, p_ens
    - If multiple rows per CID: aggregate p_ens by mean; y_true by max (conservative)
    """
    required = {COL_PROB, COL_Y, COL_CID}
    if not required.issubset(df.columns):
        raise ValueError(f"CSV must contain columns {required}, but got {set(df.columns)}")

    df = df.copy()
    df[COL_PROB] = pd.to_numeric(df[COL_PROB], errors="coerce")
    df[COL_Y]    = pd.to_numeric(df[COL_Y], errors="coerce").astype("Int64")
    df = df.dropna(subset=[COL_PROB, COL_Y])

    # Aggregate if needed
    cid_counts = df[COL_CID].value_counts(dropna=False)
    if (cid_counts > 1).any():
        agg = (
            df.groupby(COL_CID, dropna=False)
              .agg(
                  y_true=(COL_Y, "max"),
                  p_ens=(COL_PROB, "mean"),
              )
              .reset_index()
        )
        return agg
    else:
        out = df[[COL_CID, COL_Y, COL_PROB]].copy()
        out = out.rename(columns={COL_Y: "y_true", COL_PROB: "p_ens"})
        return out

def scan_thresholds(y: np.ndarray, p: np.ndarray) -> pd.DataFrame:
    """
    Efficient scan over unique thresholds using sorting.
    For each unique p value as threshold t, predict positive if p >= t.
    """
    y = y.astype(int)
    p = p.astype(float)

    n = len(y)
    pos_total = int(y.sum())
    neg_total = int((y == 0).sum())

    order = np.argsort(-p)  # descending
    p_s = p[order]
    y_s = y[order]

    is_pos = (y_s == 1).astype(int)
    is_neg = (y_s == 0).astype(int)
    cum_tp = np.cumsum(is_pos)
    cum_fp = np.cumsum(is_neg)

    change_idx = np.where(np.diff(p_s) != 0)[0]
    cut_idx = np.concatenate([change_idx, np.array([n - 1])])

    rows = []
    for i in cut_idx:
        thr = float(p_s[i])
        tp = int(cum_tp[i])
        fp = int(cum_fp[i])
        fn = pos_total - tp
        tn = neg_total - fp

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        spec      = tn / (tn + fp) if (tn + fp) else 0.0
        acc       = (tp + tn) / n if n else 0.0

        rows.append({
            "Threshold": thr,
            "N": n,
            "Pos": pos_total,
            "Neg": neg_total,
            "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Precision_pos": precision,
            "Recall_pos": recall,
            "F1_pos": f1,
            "Specificity": spec,
            "Accuracy": acc,
            "PredPos": tp + fp,
            "PredNeg": tn + fn,
        })

    df = pd.DataFrame(rows).sort_values("Threshold").reset_index(drop=True)
    return df

def pick_precision_threshold(df: pd.DataFrame, target: float, min_predpos: int) -> dict:
    """
    Precision selection:
      1) Try Precision>=target AND PredPos>=min_predpos
      2) If none, return argmax Precision under PredPos>=min_predpos  <-- requested behavior
    """
    df2 = df[df["PredPos"] >= min_predpos].copy()
    if df2.empty:
        return {"found": False, "reason": f"No threshold with PredPos >= {min_predpos}", "min_predpos": min_predpos}

    ok = df2[df2["Precision_pos"] >= target]
    if not ok.empty:
        # among feasible, prefer higher recall, then higher F1, then higher threshold (more conservative)
        ok = ok.sort_values(["Recall_pos", "F1_pos", "Threshold"], ascending=[False, False, False]).head(1)
        r = ok.iloc[0].to_dict()
        r["found"] = True
        r["criterion"] = f"Precision_pos >= {target:.2f} with PredPos >= {min_predpos}"
        r["min_predpos"] = min_predpos
        return r

    # fallback: maximize Precision under PredPos constraint
    best = df2.sort_values(["Precision_pos", "Recall_pos", "F1_pos", "Threshold"],
                           ascending=[False, False, False, False]).head(1)
    r = best.iloc[0].to_dict()
    r["found"] = True
    r["criterion"] = f"MAX Precision_pos under PredPos >= {min_predpos} (fallback; no Precision_pos >= {target:.2f})"
    r["min_predpos"] = min_predpos
    return r

def pick_recall_threshold(df: pd.DataFrame, target: float, min_predpos: int) -> dict:
    """
    Recall selection:
      - Find Recall>=target AND PredPos>=min_predpos
      - Prefer higher precision, then higher F1, then higher threshold
    """
    df2 = df[df["PredPos"] >= min_predpos].copy()
    if df2.empty:
        return {"found": False, "reason": f"No threshold with PredPos >= {min_predpos}", "min_predpos": min_predpos}

    ok = df2[df2["Recall_pos"] >= target]
    if ok.empty:
        return {"found": False,
                "reason": f"No threshold achieves Recall_pos >= {target:.2f} with PredPos >= {min_predpos}",
                "min_predpos": min_predpos}

    ok = ok.sort_values(["Precision_pos", "F1_pos", "Threshold"], ascending=[False, False, False]).head(1)
    r = ok.iloc[0].to_dict()
    r["found"] = True
    r["criterion"] = f"Recall_pos >= {target:.2f} with PredPos >= {min_predpos}"
    r["min_predpos"] = min_predpos
    return r


# =========================
# Main
# =========================
def main():
    csv_path = find_existing_csv()
    print("[INFO] Using CSV:", csv_path)

    df = pd.read_csv(csv_path)
    df_cid = ensure_cid_level(df)

    y = df_cid["y_true"].to_numpy(dtype=int)
    p = df_cid["p_ens"].to_numpy(dtype=float)

    print(f"[INFO] CID-level N={len(y)} | Pos={int(y.sum())} | Neg={int((y==0).sum())}")

    scan = scan_thresholds(y, p)
    out_csv = os.path.join(OUTDIR, "threshold_scan_internal_CID.csv")
    scan.to_csv(out_csv, index=False)
    print("[SAVE] scan ->", out_csv)

    best_prec = pick_precision_threshold(scan, target=TARGET_PREC, min_predpos=MIN_PREDPOS_PREC)
    best_rec  = pick_recall_threshold(scan, target=TARGET_REC,  min_predpos=MIN_PREDPOS_REC)

    summary = {
        "ROOT": ROOT,
        "CSV_used": csv_path,
        "CID_level": True,
        "STRICT": STRICT,
        "TARGETS": {"Precision_pos": TARGET_PREC, "Recall_pos": TARGET_REC},
        "MIN_PREDPOS": {"Precision": MIN_PREDPOS_PREC, "Recall": MIN_PREDPOS_REC},
        "selected": {
            "Precision_pos_target_or_fallback": best_prec,
            "Recall_pos_target": best_rec,
        }
    }
    out_json = os.path.join(OUTDIR, "threshold_targets_internal_CID.json")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print("[SAVE] selected ->", out_json)

    print("\n=== Selected thresholds ===")
    if best_prec.get("found"):
        print("Precision_pos selection:", best_prec)
    else:
        print("Precision_pos selection:", best_prec.get("reason"))

    if best_rec.get("found"):
        print("Recall_pos selection   :", best_rec)
    else:
        print("Recall_pos selection   :", best_rec.get("reason"))

    print("\nDone.")


if __name__ == "__main__":
    main()


In [ ]:
def save_pr_curve(out_png, out_csv, y, p, title):
    if y is None or p is None or len(y)==0:
        print(f"[SKIP] PR undefined for {title}.")
        return False
    if np.unique(y).size < 2:
        print(f"[SKIP] PR undefined (single class) for {title}.")
        return False

    prec, rec, thr = precision_recall_curve(y, p)
    pr = auc(rec, prec)  # sklearn.metrics.auc returns a positive area even when recall is monotonically decreasing

    # CSV: pad the threshold array with NaN because it is one element shorter.
    thr_pad = np.r_[thr, np.nan]
    pd.DataFrame({"recall": rec, "precision": prec, "threshold": thr_pad}).to_csv(out_csv, index=False)

    # conventional layout(Recall->)Cleanup
    order = np.argsort(rec)
    rec_s  = rec[order]
    prec_s = prec[order]

    # Baseline (positive rate)
    base = float(np.mean(y))

    plt.figure()
    plt.plot(rec_s, prec_s, label=f"AUC={pr:.4f}")
    plt.plot([0, 1], [base, base], linestyle="--", label=f"Prevalence={base:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.xlim(0, 1); plt.ylim(0, 1)
    plt.title(title); plt.legend(); plt.tight_layout()
    plt.savefig(out_png, dpi=160); plt.close()

    print(f"[PR] Saved: {out_png} | {out_csv}")
    return True

# ROC (PNG & CSV) for ensemble
save_roc_curve(f"{OUT_DIR}/roc_union_m2m4_cvpr.png",
               f"{OUT_DIR}/roc_union_m2m4_cvpr.csv",
               y_u, pENS_u, "M2+M4 (cv_pr) ROC - UNION")

# PR (PNG & CSV) for ensemble  ★Additional
save_pr_curve(f"{OUT_DIR}/pr_union_m2m4_cvpr.png",
              f"{OUT_DIR}/pr_union_m2m4_cvpr.csv",
              y_u, pENS_u, "M2+M4 (cv_pr) PR - UNION")

if idx_k is not None:
    save_roc_curve(f"{OUT_DIR}/roc_kegg_m2m4_cvpr.png",
                   f"{OUT_DIR}/roc_kegg_m2m4_cvpr.csv",
                   y_k, pENS_k, "M2+M4 (cv_pr) ROC - KEGG-only")

    # PR (PNG & CSV) for ensemble  ★Additional
    save_pr_curve(f"{OUT_DIR}/pr_kegg_m2m4_cvpr.png",
                  f"{OUT_DIR}/pr_kegg_m2m4_cvpr.csv",
                  y_k, pENS_k, "M2+M4 (cv_pr) PR - KEGG-only")


In [ ]:
# === One-cell Figure Generator for QT-M2M4-cvpr (Index-safe, RDKit-robust) ===
# Generates: (A) UMAP, (B) Novelty vs PR-AUC, (C) Reliability (Brier/ECE),
#            (D) Top-K Precision, (E) Attribution (ECFP surrogate),
#            (F) Misclassification substructure heatmap.
# Inputs:
#   ROOT/data_graph_with_smiles.pt
#   ROOT/reports_internal_QT-M2M4-cvpr/predictions_internal_QT-M2M4-cvpr.csv
# Outputs under ROOT/figs_qt_final_m2m4_cvpr/

import os, sys, subprocess, json, numpy as np, pandas as pd, warnings, math, random
warnings.filterwarnings("ignore")

# Silence MorganGenerator deprecation warnings from RDKit.
warnings.filterwarnings("ignore", message=".*MorganGenerator.*", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# --------- CONFIG ---------
ROOT     = "/content/drive/MyDrive/Chemoinfo_QT"
DATA_PT  = f"{ROOT}/data_graph_with_smiles.pt"
PRED_CSV = f"{ROOT}/reports_internal_QT-M2M4-cvpr/predictions_internal_QT-M2M4-cvpr.csv"
OUT_DIR  = f"{ROOT}/figs_qt_final_m2m4_cvpr"
os.makedirs(OUT_DIR, exist_ok=True)

# Upper bound for novelty calculations to avoid O(N^2) cost on the full dataset.
NOVELTY_SAMPLE_MAX = 2500
RNG_SEED = 42
FIXED_THR = 0.5455  # ← M2+M4 fixed threshold

# --------- Helper: quiet pip ---------
def _pip_install(cmd_list):
    try:
        subprocess.check_call(cmd_list, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

def _ensure(pkg, pip_name=None):
    try:
        __import__(pkg)
        return True
    except Exception:
        return _pip_install([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

# sklearn / umap / matplotlib
_ensure("sklearn", "scikit-learn")
_ensure("umap", "umap-learn")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

# torch / pyg
_ensure("torch", "torch")
try:
    from torch.serialization import add_safe_globals
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
except Exception:
    _pip_install([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    from torch.serialization import add_safe_globals
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
import torch

# RDKit(optional)
HAVE_RDKIT = True
try:
    import rdkit  # quick probe
except Exception:
    if not _pip_install([sys.executable, "-m", "pip", "install", "-q", "rdkit==2023.9.5"]):
        if not _pip_install([sys.executable, "-m", "pip", "install", "-q", "rdkit"]):
            HAVE_RDKIT = False

if HAVE_RDKIT:
    from rdkit import RDLogger, rdBase
    RDLogger.DisableLog("rdApp.*")
    rdBase.DisableLog("rdApp.warning")
    from rdkit import Chem
    from rdkit.Chem import AllChem, Draw
    from rdkit import DataStructs

# --------- Load internal data ---------
def load_pyg_list(path):
    add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    return list(obj)

assert os.path.exists(DATA_PT), f"Missing: {DATA_PT}"
assert os.path.exists(PRED_CSV), f"Missing: {PRED_CSV}"

dl = load_pyg_list(DATA_PT)
pred = pd.read_csv(PRED_CSV)

# ---- Robust merge: attach gr_vec to each row before merging. ----
records = []
for d in dl:
    smiles = getattr(d, "smiles", "")
    yv = int(getattr(d, "y", -1))
    yv = yv if yv in (0,1) else -1
    g = np.asarray(getattr(d, "g")).reshape(-1)
    r = np.asarray(getattr(d, "r")).reshape(-1)
    # Dimension check (expected: g=7, r=10).
    if g.size == 0 or r.size == 0:
        continue
    gr_vec = np.concatenate([g, r], axis=0)  # 17-dim (g7 + r10)
    records.append({"SMILES": smiles, "y_true": yv, "gr_vec": gr_vec})
base = pd.DataFrame(records)
base = base[base["y_true"].isin([0,1])].copy()

# Expected columns for M2+M4 (cv_pr).
# Require at least 'SMILES' and 'p_ens'; the code still runs if p_M2/p_M4 are absent.
need_min = {"SMILES","p_ens"}
missing_min = sorted(list(need_min - set(pred.columns)))
assert not missing_min, f"Prediction CSV missing columns: {missing_min}"
has_m2 = "p_M2" in pred.columns
has_m4 = "p_M4" in pred.columns

# Merge by SMILES. Duplicate SMILES generate duplicate rows, and gr_vec is replicated accordingly.
use_cols = ["SMILES","p_ens"] + (["p_M2"] if has_m2 else []) + (["p_M4"] if has_m4 else [])
df = base.merge(pred[use_cols], on="SMILES", how="inner").reset_index(drop=True)
df = df[df["y_true"].isin([0,1])].reset_index(drop=True)

# Build the UMAP matrix from df['gr_vec'] so the row order always matches.
# Drop rows with missing gr_vec.
df = df[df["gr_vec"].notna()].reset_index(drop=True)
gr_mat = np.vstack(df["gr_vec"].to_numpy())

# --------- (A) UMAP on [g;r] ---------
A_PATH = None
try:
    import umap
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="euclidean", random_state=42)
    X2 = reducer.fit_transform(gr_mat)
    plt.figure()
    pos = df["y_true"].values==1
    neg = df["y_true"].values==0
    plt.scatter(X2[neg,0], X2[neg,1], s=8, alpha=0.7, label="Negative")
    plt.scatter(X2[pos,0], X2[pos,1], s=8, alpha=0.7, marker="x", label="Positive")
    plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2"); plt.title("A) UMAP of molecular descriptors (g+r)")
    plt.legend(); plt.tight_layout()
    A_PATH = os.path.join(OUT_DIR, "A_umap_gr.png")
    plt.savefig(A_PATH, dpi=160); plt.close()
except Exception as e:
    print("[WARN] UMAP failed:", e)

# --------- (B) Novelty proxy vs PR-AUC(only when RDKit is available; with a sampling safeguard) ---------
B_PATH = None
if HAVE_RDKIT:
    def morgan_fp(smiles, radius=2, nBits=2048):
        m = Chem.MolFromSmiles(smiles)
        if m is None: return None
        return AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=nBits)

    # Use a random subset when the sample size is large to approximate the O(M^2) calculation.
    rng = np.random.default_rng(RNG_SEED)
    N = len(df)
    if N > NOVELTY_SAMPLE_MAX:
        sample_idx = np.sort(rng.choice(np.arange(N), size=NOVELTY_SAMPLE_MAX, replace=False))
        df_bench = df.iloc[sample_idx].reset_index(drop=True)
        print(f"[INFO] Novelty panel: using subset {len(df_bench)}/{N} for speed.")
    else:
        df_bench = df.copy()

    smiles_list = df_bench["SMILES"].tolist()
    fps = [morgan_fp(s) for s in smiles_list]
    valid_idx = [i for i,f in enumerate(fps) if f is not None]

    # For each molecule, compute the maximum Tanimoto similarity to any other molecule.
    max_sim = np.zeros(len(df_bench), dtype=float)
    for i in valid_idx:
        fi = fps[i]
        sims = DataStructs.BulkTanimotoSimilarity(fi, [fps[j] for j in valid_idx if j != i])
        best = max(sims) if len(sims) else 0.0
        max_sim[i] = best

    df_bench["max_sim_rest"] = max_sim
    bins = [0.0, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 1.01]
    df_bench["sim_bin"] = pd.cut(df_bench["max_sim_rest"], bins=bins, right=False)

    rows_b = []
    for b in df_bench["sim_bin"].dropna().unique().sort_values():
        sub = df_bench[df_bench["sim_bin"]==b]
        if sub["y_true"].nunique()<2:
            pr = np.nan
        else:
            prec, rec, _ = precision_recall_curve(sub["y_true"].values, sub["p_ens"].values)
            pr = auc(rec, prec)
        rows_b.append({"bin": str(b), "n": len(sub), "pr_auc": pr})
    df_b = pd.DataFrame(rows_b)

    plt.figure()
    plt.plot(np.arange(len(df_b)), df_b["pr_auc"].values, marker="o")
    plt.xticks(np.arange(len(df_b)), df_b["bin"].tolist(), rotation=45, ha="right")
    plt.ylabel("PR-AUC"); plt.xlabel("Max similarity to others (bin)")
    plt.title("B) Novelty proxy vs PR-AUC")
    plt.tight_layout()
    B_PATH = os.path.join(OUT_DIR, "B_novelty_vs_prauc.png")
    plt.savefig(B_PATH, dpi=160); plt.close()
else:
    print("[SKIP] RDKit unavailable -> skip panel (B).")

# --------- (C) Reliability curve (Brier/ECE) ---------
frac_pos, mean_pred = calibration_curve(df["y_true"].values, df["p_ens"].values, n_bins=10, strategy="quantile")
brier = np.mean((df["p_ens"].values - df["y_true"].values)**2)
ece = np.mean(np.abs(frac_pos - mean_pred))
plt.figure()
plt.plot([0,1],[0,1], linestyle="--")
plt.plot(mean_pred, frac_pos, marker="o")
plt.xlabel("Predicted probability"); plt.ylabel("Observed frequency")
plt.title(f"C) Reliability (Brier={brier:.3f}, ECE={ece:.3f})")
plt.tight_layout()
C_PATH = os.path.join(OUT_DIR, "C_reliability.png")
plt.savefig(C_PATH, dpi=160); plt.close()

# --------- (D) Top-K precision ---------
def precision_at_k(y, p, k):
    k = max(1, min(k, len(p)))
    idx = np.argsort(-p)[:k]
    return float(np.mean(y[idx]))
Ks = [10,25,50,100,200,500,1000]
p_at = [precision_at_k(df["y_true"].values, df["p_ens"].values, k) for k in Ks]
plt.figure()
plt.plot(Ks, p_at, marker="o")
plt.xlabel("K"); plt.ylabel("Precision@K")
plt.title("D) Top-K Precision curve")
plt.tight_layout()
D_PATH = os.path.join(OUT_DIR, "D_topk_precision.png")
plt.savefig(D_PATH, dpi=160); plt.close()

# --------- (E) Attribution via ECFP surrogate(only when RDKit is available) ---------
E_PATHS = []
if HAVE_RDKIT:
    bit_infos, bit_mats = [], []
    ys = df["y_true"].values
    for s in df["SMILES"].tolist():
        m = Chem.MolFromSmiles(s)
        if m is None:
            bit_infos.append({})
            bit_mats.append(np.zeros((2048,), dtype=np.uint8))
            continue
        info = {}
        bv = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048, bitInfo=info)
        arr = np.zeros((2048,), dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(bv, arr)
        bit_infos.append(info); bit_mats.append(arr)
    X = np.vstack(bit_mats)
    try:
        clf = LogisticRegression(max_iter=200, n_jobs=1)
    except TypeError:
        clf = LogisticRegression(max_iter=200)
    clf.fit(X, ys)
    coef = clf.coef_[0]

    p = df["p_ens"].values
    pred_bin = (p>=FIXED_THR).astype(int)  # ← use 0.420554
    TP_idx = np.where((pred_bin==1)&(ys==1))[0][:2]
    FP_idx = np.where((pred_bin==1)&(ys==0))[0][:2]
    FN_idx = np.where((pred_bin==0)&(ys==1))[0][:2]

    def atom_importance_from_bits(mol, bit_info, coefs):
        n = mol.GetNumAtoms()
        scores = np.zeros(n, dtype=float)
        for b, envs in bit_info.items():
            w = coefs[b]
            for (atom_idx, radius) in envs:
                amap = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
                atoms = {atom_idx}
                for bond_idx in amap:
                    bnd = mol.GetBondWithIdx(bond_idx)
                    atoms.add(bnd.GetBeginAtomIdx()); atoms.add(bnd.GetEndAtomIdx())
                for a in atoms: scores[a] += w
        if (scores.max() - scores.min())>1e-9:
            scores = (scores - scores.min())/(scores.max()-scores.min())
        return scores

    def draw_with_atom_scores(smiles, scores, path, title):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return False
        atom_cols = {i:(float(scores[i]), 0.0, 0.0) for i in range(mol.GetNumAtoms())}
        d2d = Draw.MolDraw2DCairo(500, 400)
        Draw.PrepareAndDrawMolecule(d2d, mol, highlightAtoms=list(atom_cols.keys()), highlightAtomColors=atom_cols, legend=title)
        d2d.FinishDrawing()
        with open(path, "wb") as f: f.write(d2d.GetDrawingText())
        return True

    def make_attr_panels(indices, label):
        outs=[]
        for i in indices:
            s = df.loc[i,"SMILES"]
            m = Chem.MolFromSmiles(s)
            if m is None: continue
            scores = atom_importance_from_bits(m, bit_infos[i], coef)
            pth = os.path.join(OUT_DIR, f"E_attr_{label}_{i}.png")
            draw_with_atom_scores(s, scores, pth, title=f"{label} #{i}  p={p[i]:.2f}  y={ys[i]}")
            outs.append(pth)
        return outs

    E_PATHS += make_attr_panels(TP_idx, "TP")
    E_PATHS += make_attr_panels(FP_idx, "FP")
    E_PATHS += make_attr_panels(FN_idx, "FN")
else:
    print("[SKIP] RDKit unavailable -> skip panel (E).")

# --------- (F) Misclassification substructure heatmap(only when RDKit is available) ---------
F_PATH = None
if HAVE_RDKIT:
    SMARTS = {
        "AromaticRing": "a1aaaaa1",
        "AliphaticRing": "[R;!a]",
        "Halogen": "[F,Cl,Br,I]",
        "QuaternaryAmmonium": "[N+](C)(C)(C)C",
        "TertiaryAmine": "[NX3;H0;!$(NC=O)]",
        "CarboxylicAcid": "C(=O)[OH]",
        "Amide": "C(=O)N",
        "Ether": "COC",
        "Sulfonamide": "S(=O)(=O)N",
        "Nitrile": "C#N",
    }
    ps = {k: Chem.MolFromSmarts(v) for k,v in SMARTS.items()}
    ys = df["y_true"].values
    p = df["p_ens"].values
    pred_bin = (p>=FIXED_THR).astype(int)  # ← use 0.420554
    groups = {
        "TP": np.where((pred_bin==1)&(ys==1))[0],
        "FP": np.where((pred_bin==1)&(ys==0))[0],
        "TN": np.where((pred_bin==0)&(ys==0))[0],
        "FN": np.where((pred_bin==0)&(ys==1))[0],
    }
    counts = pd.DataFrame(0, index=list(SMARTS.keys()), columns=list(groups.keys()))
    totals = {g: len(idx) for g, idx in groups.items()}

    for name, patt in ps.items():
        for g, idxs in groups.items():
            c = 0
            for i in idxs:
                m = Chem.MolFromSmiles(df.loc[i,"SMILES"])
                if m is None: continue
                if m.HasSubstructMatch(patt): c += 1
            counts.loc[name, g] = c

    rates = counts.copy().astype(float)
    for g in groups.keys():
        rates[g] = rates[g] / max(1, totals[g])  # avoid division by zero and keep values on the 0-1 scale

    # ★ low=blue, high=red, with legend
    plt.figure(figsize=(7, 5))
    im = plt.imshow(
        rates.values,
        aspect="auto",
        cmap="bwr",     # blue-white-red(: -> :)
        vmin=0.0,
        vmax=1.0        # fixed to 0-1 because the values are proportions
    )
    plt.yticks(np.arange(len(rates.index)), rates.index.tolist())
    plt.xticks(np.arange(len(rates.columns)), rates.columns.tolist())
    plt.title("F) Misclassification substructure heatmap (rate)")
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label("Rate", rotation=90)
    plt.tight_layout()

    F_PATH = os.path.join(OUT_DIR, "F_misclf_substruct_heatmap.png")
    plt.savefig(F_PATH, dpi=160); plt.close()
else:
    print("[SKIP] RDKit unavailable -> skip panel (F).")

# --------- Save working table & report paths ---------
WORK_TBL = os.path.join(OUT_DIR, "working_table.csv")
# If RDKit is unavailable, initialize max_sim and sim_bin as NaN.
if "max_sim_rest" not in locals() and "max_sim_rest" not in df.columns:
    df["max_sim_rest"] = np.nan
    df["sim_bin"] = np.nan
df[["SMILES","y_true","p_ens","max_sim_rest","sim_bin"]].to_csv(WORK_TBL, index=False)

outputs = {
    "A_umap_gr.png": A_PATH,
    "B_novelty_vs_prauc.png": B_PATH,
    "C_reliability.png": os.path.join(OUT_DIR, "C_reliability.png"),
    "D_topk_precision.png": os.path.join(OUT_DIR, "D_topk_precision.png"),
    "E_attr_paths": E_PATHS,
    "F_misclf_heatmap.png": F_PATH,
    "working_table.csv": WORK_TBL,
    "note": "Panels B/E/F require RDKit. Index mismatch is avoided by carrying gr_vec with each row. p_ens is cv_pr-weighted M2+M4 probability. Novelty uses subset if N is large."
}
print(json.dumps(outputs, indent=2))

In [ ]:
# === Cell 2 (QT): SHAP + PDP on g(7) & r(10) features (surrogate on p_ens) ===
# Target: M2+M4 cv_pr ensemble (p_ens is the final probability of M2+M4(cv_pr)).
# scikit-learn  partial_dependence Specification

import os, sys, subprocess, json, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

# -------- CONFIG --------
ROOT     = "/content/drive/MyDrive/Chemoinfo_QT"
DATA_PT  = f"{ROOT}/data_graph_with_smiles.pt"
# ★ internalEvaluation(M2+M4, cv_pr)CSV(p_ens column)
PRED_CSV = f"{ROOT}/reports_internal_QT-M2M4-cvpr/predictions_internal_QT-M2M4-cvpr.csv"
OUT_DIR  = f"{ROOT}/figs_shap_pdp_QT-M2M4_cvpr"
os.makedirs(OUT_DIR, exist_ok=True)

# ----- deps -----
def _pip(cmd):
    try:
        subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); return True
    except Exception:
        return False

def _ensure(mod, pip_name=None):
    try:
        __import__(mod); return True
    except Exception:
        return _pip([sys.executable, "-m", "pip", "install", "-q", pip_name or mod])

_ensure("sklearn", "scikit-learn")
_ensure("shap", "shap")
_ensure("torch", "torch")
try:
    from torch.serialization import add_safe_globals
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
except Exception:
    _pip([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    from torch.serialization import add_safe_globals
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage

import torch
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import partial_dependence
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- load ----
def load_pyg_list(path):
    add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    return list(obj)

assert os.path.exists(DATA_PT), f"Missing: {DATA_PT}"
assert os.path.exists(PRED_CSV), f"Missing: {PRED_CSV}"
dl = load_pyg_list(DATA_PT)
pred = pd.read_csv(PRED_CSV)

# p_ens (the final probability of M2+M4(cv_pr)) is required.
if "p_ens" not in pred.columns:
    raise RuntimeError("The predictions CSV does not contain a 'p_ens' column. Please provide the internal-evaluation predictions for M2+M4 (cv_pr).")

# carry (g,r) inside rows and merge by SMILES
rows = []
for d in dl:
    sm = getattr(d, "smiles", "")
    y  = int(getattr(d, "y", -1)); y  = y if y in (0,1) else -1
    g = np.asarray(getattr(d, "g")).reshape(-1)  # 7
    r = np.asarray(getattr(d, "r")).reshape(-1)  # 10
    if g.size != 7 or r.size != 10:  # sanity check
        continue
    gr = np.concatenate([g, r], axis=0)          # (17,)
    rows.append({"SMILES": sm, "y_true": y, "gr_vec": gr})
base = pd.DataFrame(rows)

df = base.merge(pred[["SMILES","p_ens"]], on="SMILES", how="inner")
df = df[df["y_true"].isin([0,1])].reset_index(drop=True)
if df.empty:
    raise RuntimeError("The merge result is empty. Please verify that the SMILES keys are aligned.")

X = np.vstack(df["gr_vec"].to_numpy())  # (n,17)
y = df["p_ens"].values.astype(float)
feat_names = [f"g{i+1}" for i in range(7)] + [f"r{i+1}" for i in range(10)]

# ---- surrogate (GBR) ----
sur = GradientBoostingRegressor(random_state=42)
sur.fit(X, y)

# ---- SHAP summary ----
explainer = shap.TreeExplainer(sur)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X, feature_names=feat_names, show=False, plot_size=(9,6))
SUM_PATH = os.path.join(OUT_DIR, "shap_summary_gr.png")
plt.tight_layout(); plt.savefig(SUM_PATH, dpi=180); plt.close()

# ---- PDP: top-3 features by |SHAP| ----
imp = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(-imp)[:3]

def _pd_xy(pd_bunch):
    """sklearn handle return-spec differences(grid_values/values/x_values  average )"""
    if hasattr(pd_bunch, "grid_values"):
        xs = pd_bunch.grid_values[0]
    elif isinstance(pd_bunch, dict) and "grid_values" in pd_bunch:
        xs = pd_bunch["grid_values"][0]
    elif isinstance(pd_bunch, dict) and "values" in pd_bunch:
        xs = pd_bunch["values"][0]
    elif hasattr(pd_bunch, "values"):
        xs = pd_bunch.values[0]
    elif hasattr(pd_bunch, "x_values"):
        xs = pd_bunch.x_values[0]
    elif isinstance(pd_bunch, dict) and "x_values" in pd_bunch:
        xs = pd_bunch["x_values"][0]
    else:
        raise KeyError("No x grid found in partial_dependence result.")
    if hasattr(pd_bunch, "average"):
        ys = pd_bunch.average[0]
    elif isinstance(pd_bunch, dict) and "average" in pd_bunch:
        ys = pd_bunch["average"][0]
    else:
        raise KeyError("No 'average' in partial_dependence result.")
    return np.asarray(xs), np.asarray(ys)

pdp_paths = []
for k in top_idx:
    pd_res = partial_dependence(sur, X, [k], kind="average")
    xs, ys_pd = _pd_xy(pd_res)
    plt.figure()
    plt.plot(xs, ys_pd)
    plt.xlabel(feat_names[k]); plt.ylabel("p_ens (surrogate of M2+M4 cv_pr)")
    plt.title(f"PDP: {feat_names[k]}")
    plt.tight_layout()
    pth = os.path.join(OUT_DIR, f"pdp_{feat_names[k]}.png")
    plt.savefig(pth, dpi=180); plt.close()
    pdp_paths.append(pth)

print(json.dumps({
    "out_dir": OUT_DIR,
    "shap_summary": SUM_PATH,
    "pdp_top3": pdp_paths,
}, indent=2))

In [ ]:
# === CiPA 28 probes (High 8 + Low 9) -> Figure5a / Figure5b (submission-ready PNG+PDF) ===
# Outputs:
#   - OUT_DIR/cipa28_probes_predictions.csv
#   - OUT_DIR/Figure5a_CiPA28_Probes_Attribution.png / .pdf
#   - OUT_DIR/Figure5b_CiPA28_Probes_Attribution.png / .pdf
#
# Resolution policy (AIM/Elsevier practical-safe):
#   - Combination figures: >= 500 dpi recommended.
#   - Ensure width >= 3740 px (full-width equivalent at 500 dpi).
#   - Avoid bbox_inches="tight" (can shrink output width unexpectedly).
#   - Embed DPI metadata into PNG explicitly (PIL).
#   - Create PDF by embedding the final PNG (guaranteed effective resolution).

import os, sys, subprocess
import numpy as np
import pandas as pd

# ----------------- CONFIG -----------------
ROOT     = "/content/drive/MyDrive/Chemoinfo_QT"
DATA_PT  = f"{ROOT}/data_graph_with_smiles.pt"
PRED_CSV = f"{ROOT}/reports_internal_QT-M2M4-cvpr/predictions_internal_QT-M2M4-cvpr.csv"
OUT_DIR  = f"{ROOT}/figs_qt_final_m2m4_cvpr"
os.makedirs(OUT_DIR, exist_ok=True)

FIXED_THR = 0.5455
RNG_SEED  = 42

OUT_TABLE = os.path.join(OUT_DIR, "cipa28_probes_predictions.csv")

# Output base names
FIG5A_BASE = os.path.join(OUT_DIR, "Figure5a_CiPA28_Probes_Attribution")
FIG5B_BASE = os.path.join(OUT_DIR, "Figure5b_CiPA28_Probes_Attribution")

# --- Submission-ready raster spec ---
# Combination figures: 500 dpi is the Elsevier recommendation baseline.
OUT_DPI   = 500
FIG_W_IN  = 10.5
FIG_H_IN  = 14.0

# Minimum width requirement (safe-side):
# Full-width at 500 dpi ~ 7.48 inch -> 3740 px.
MIN_WIDTH_PX = 3740

# --- Increase RDKit panel render size to keep molecule panels crisp ---
# (Resolution only; same molecule, same colors, same caption text.)
MOL_W = 2600  # was 520
MOL_H = 2100  # was 420

# CiPA probes (17 shown here)
HIGH8 = ["Ajmaline","Azimilide","Bepridil","Disopyramide","Dofetilide","Ibutilide","Quinidine","Vandetanib"]
LOW9  = ["Diltiazem","Loratadine","Metoprolol","Mexiletine","Nifedipine","Nitrendipine","Ranolazine","Tamoxifen","Verapamil"]

QUERY_OVR = {}

# ----------------- helper: quiet pip -----------------
def _pip_install(cmd_list):
    try:
        subprocess.check_call(cmd_list, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

def _ensure(pkg, pip_name=None):
    try:
        __import__(pkg)
        return True
    except Exception:
        return _pip_install([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

# deps
_ensure("numpy", "numpy")
_ensure("pandas", "pandas")
_ensure("torch", "torch")
_ensure("sklearn", "scikit-learn")
_ensure("scipy", "scipy")
_ensure("matplotlib", "matplotlib")
_ensure("pubchempy", "pubchempy")
_ensure("PIL", "pillow")

# safe display (works in notebooks; no-op in pure script)
try:
    from IPython.display import display
except Exception:
    def display(x):
        pass

# torch/pyg safe load
try:
    from torch.serialization import add_safe_globals
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
except Exception:
    _pip_install([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    from torch.serialization import add_safe_globals
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
import torch

# RDKit
HAVE_RDKIT = True
try:
    import rdkit
except Exception:
    if not _pip_install([sys.executable, "-m", "pip", "install", "-q", "rdkit==2023.9.5"]):
        if not _pip_install([sys.executable, "-m", "pip", "install", "-q", "rdkit"]):
            HAVE_RDKIT = False
assert HAVE_RDKIT, "RDKit install failed."

from rdkit import RDLogger, rdBase
RDLogger.DisableLog("rdApp.*")
rdBase.DisableLog("rdApp.warning")
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

# optional standardizer
HAVE_STD = True
try:
    from rdkit.Chem.MolStandardize import rdMolStandardize
except Exception:
    HAVE_STD = False

from scipy.sparse import csr_matrix, vstack
from sklearn.linear_model import Ridge, LogisticRegression

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpecFromSubplotSpec

import pubchempy as pcp
from PIL import Image
import io

# ----------------- basic checks -----------------
assert os.path.exists(DATA_PT), f"Missing: {DATA_PT}"
assert os.path.exists(PRED_CSV), f"Missing: {PRED_CSV}"

np.random.seed(RNG_SEED)

# ----------------- load internal files -----------------
def load_pyg_list(path):
    add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, list): return obj
    if isinstance(obj, dict) and "data_list" in obj: return obj["data_list"]
    return list(obj)

dl = load_pyg_list(DATA_PT)
pred_internal = pd.read_csv(PRED_CSV)

need_min = {"SMILES","p_ens"}
missing = sorted(list(need_min - set(pred_internal.columns)))
assert not missing, f"PRED_CSV missing columns: {missing}"

# internal y_true table (for attribution surrogate)
records=[]
for d in dl:
    s = getattr(d, "smiles", "")
    y = int(getattr(d, "y", -1))
    if y in (0,1):
        records.append({"SMILES": s, "y_true": y})
base = pd.DataFrame(records).dropna()

df_y = base.merge(pred_internal[["SMILES","p_ens"]], on="SMILES", how="inner").reset_index(drop=True)

# ----------------- standardization / keys -----------------
def can_smiles(s):
    m = Chem.MolFromSmiles(str(s))
    if m is None: return None
    return Chem.MolToSmiles(m, isomericSmiles=True, canonical=True)

def parent_can_smiles(s):
    if not HAVE_STD:
        return can_smiles(s)
    m = Chem.MolFromSmiles(str(s))
    if m is None: return None
    try:
        m = rdMolStandardize.FragmentParent(m)
        m = rdMolStandardize.Uncharger().uncharge(m)
    except Exception:
        pass
    return Chem.MolToSmiles(m, isomericSmiles=True, canonical=True)

def inchi_key(s):
    try:
        from rdkit.Chem import inchi
        m = Chem.MolFromSmiles(str(s))
        if m is None: return None
        return inchi.MolToInchiKey(m)
    except Exception:
        return None

def inchi_key14(k):
    if k is None or (isinstance(k,float) and np.isnan(k)): return None
    ks = str(k)
    return ks[:14] if len(ks) >= 14 else None

df_int = pred_internal[["SMILES","p_ens"]].copy()
df_int["SMILES_CAN"]  = df_int["SMILES"].map(can_smiles)
df_int["PARENT_CAN"]  = df_int["SMILES"].map(parent_can_smiles)
df_int["InChIKey"]    = df_int["SMILES"].map(inchi_key)
df_int["InChIKey14"]  = df_int["InChIKey"].map(inchi_key14)

df_int_key   = df_int.dropna(subset=["InChIKey"]).drop_duplicates("InChIKey")
df_int_key14 = df_int.dropna(subset=["InChIKey14"]).drop_duplicates("InChIKey14")
df_int_can   = df_int.dropna(subset=["SMILES_CAN"]).drop_duplicates("SMILES_CAN")
df_int_pcan  = df_int.dropna(subset=["PARENT_CAN"]).drop_duplicates("PARENT_CAN")

# ----------------- PubChem SMILES fetch -----------------
def fetch_pubchem_smiles(query):
    cs = pcp.get_compounds(query, "name")
    if not cs:
        return None
    c0 = cs[0]
    smi = getattr(c0, "isomeric_smiles", None) or getattr(c0, "canonical_smiles", None)
    return smi

def resolve_smiles(drug):
    q = QUERY_OVR.get(drug, drug)
    return fetch_pubchem_smiles(q)

# ----------------- ECFP sparse row -----------------
def fp_row_sparse(smiles, radius=2, nBits=2048):
    m = Chem.MolFromSmiles(str(smiles))
    if m is None:
        return None
    bv = AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=nBits)
    on = list(bv.GetOnBits())
    if len(on) == 0:
        return csr_matrix((1, nBits), dtype=np.float32)
    data = np.ones(len(on), dtype=np.float32)
    rows = np.zeros(len(on), dtype=np.int32)
    cols = np.asarray(on, dtype=np.int32)
    return csr_matrix((data, (rows, cols)), shape=(1, nBits), dtype=np.float32)

# ----------------- Surrogate for p_ens (ECFP -> p_ens) -----------------
Xr_rows=[]
yr_vals=[]
for smi, pens in zip(pred_internal["SMILES"].tolist(), pred_internal["p_ens"].tolist()):
    r = fp_row_sparse(smi)
    if r is None:
        continue
    Xr_rows.append(r)
    yr_vals.append(float(pens))
Xr = vstack(Xr_rows)
yr = np.asarray(yr_vals, dtype=np.float32)
reg = Ridge(alpha=1.0, solver="sparse_cg", random_state=RNG_SEED)
reg.fit(Xr, yr)

# ----------------- Attribution surrogate (ECFP -> y_true) -----------------
Xa_rows=[]
ya_vals=[]
for smi, y in zip(df_y["SMILES"].tolist(), df_y["y_true"].tolist()):
    r = fp_row_sparse(smi)
    if r is None:
        continue
    Xa_rows.append(r)
    ya_vals.append(int(y))
Xa = vstack(Xa_rows)
ya = np.asarray(ya_vals, dtype=np.int32)

clf = LogisticRegression(max_iter=300, solver="saga", n_jobs=1)
clf.fit(Xa, ya)
coef = clf.coef_[0]  # 2048

# ----------------- Attribution drawing (NO legend inside image) -----------------
def atom_importance_from_bits(mol, bit_info, coefs):
    n = mol.GetNumAtoms()
    scores = np.zeros(n, dtype=float)
    for b, envs in bit_info.items():
        w = float(coefs[b])
        for (atom_idx, radius) in envs:
            amap = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
            atoms = {atom_idx}
            for bond_idx in amap:
                bnd = mol.GetBondWithIdx(bond_idx)
                atoms.add(bnd.GetBeginAtomIdx()); atoms.add(bnd.GetEndAtomIdx())
            for a in atoms:
                scores[a] += w
    if (scores.max() - scores.min()) > 1e-9:
        scores = (scores - scores.min())/(scores.max()-scores.min())
    return scores

def draw_attr_no_legend(smiles, scores, w=MOL_W, h=MOL_H):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    atom_cols = {i:(float(scores[i]), 0.0, 0.0) for i in range(mol.GetNumAtoms())}
    d2d = Draw.MolDraw2DCairo(w, h)
    opts = d2d.drawOptions()
    try:
        opts.legendFontSize = 0
    except Exception:
        pass
    Draw.PrepareAndDrawMolecule(
        d2d, mol,
        highlightAtoms=list(atom_cols.keys()),
        highlightAtomColors=atom_cols,
        legend=""
    )
    d2d.FinishDrawing()
    return d2d.GetDrawingText()

def pngbytes_to_array(png_bytes):
    im = Image.open(io.BytesIO(png_bytes)).convert("RGBA")
    return np.array(im)

# ----------------- internal lookup function -----------------
def lookup_pens(smiles_probe):
    if smiles_probe is None:
        return (np.nan, "missing_smiles", "none")

    smi_can  = can_smiles(smiles_probe)
    smi_pcan = parent_can_smiles(smiles_probe)
    k   = inchi_key(smiles_probe)
    k14 = inchi_key14(k)

    if k is not None and k in set(df_int_key["InChIKey"]):
        pens = float(df_int_key.loc[df_int_key["InChIKey"]==k, "p_ens"].iloc[0])
        return (pens, "internal", "InChIKey")
    if k14 is not None and k14 in set(df_int_key14["InChIKey14"]):
        pens = float(df_int_key14.loc[df_int_key14["InChIKey14"]==k14, "p_ens"].iloc[0])
        return (pens, "internal", "InChIKey14")
    if smi_can is not None and smi_can in set(df_int_can["SMILES_CAN"]):
        pens = float(df_int_can.loc[df_int_can["SMILES_CAN"]==smi_can, "p_ens"].iloc[0])
        return (pens, "internal", "SMILES_CAN")
    if smi_pcan is not None and smi_pcan in set(df_int_pcan["PARENT_CAN"]):
        pens = float(df_int_pcan.loc[df_int_pcan["PARENT_CAN"]==smi_pcan, "p_ens"].iloc[0])
        return (pens, "internal", "PARENT_CAN")

    xr = fp_row_sparse(smiles_probe)
    if xr is None:
        return (np.nan, "unparsed_smiles", "none")
    pens = float(reg.predict(xr)[0])
    pens = float(np.clip(pens, 0.0, 1.0))
    return (pens, "surrogate", "ECFP->p_ens")

# ----------------- build probe table -----------------
rows=[]
for grp, drug_list in [("High", HIGH8), ("Low", LOW9)]:
    for drug in drug_list:
        smi = resolve_smiles(drug)
        pens, src, keyhit = lookup_pens(smi)
        pred_bin = (int(pens >= FIXED_THR) if pd.notna(pens) else np.nan)
        rows.append({
            "group": grp,
            "drug": drug,
            "query": QUERY_OVR.get(drug, drug),
            "SMILES_probe": smi,
            "p_ens": pens,
            "pred_bin@thr": pred_bin,
            "thr": FIXED_THR,
            "p_ens_source": src,
            "match_rule": keyhit
        })

probe_tbl = pd.DataFrame(rows)
probe_tbl.to_csv(OUT_TABLE, index=False)
print("Saved probe table:", OUT_TABLE)
display(probe_tbl[["group","drug","p_ens","pred_bin@thr","p_ens_source","match_rule"]])

# ----------------- caption -----------------
def caption_text(row):
    drug = row["drug"]
    grp  = row["group"]
    pens = row["p_ens"]

    pens_str = "NA" if (pens is None or pd.isna(pens)) else f"{float(pens):.3f}"
    if pd.isna(row["pred_bin@thr"]):
        cls_str = "NA"
    else:
        cls_str = "Positive" if int(row["pred_bin@thr"])==1 else "Negative"

    line1 = f"{drug} ({grp})"
    line2 = f"p_ens={pens_str} | pred={cls_str}"
    return line1 + "\n" + line2

# ----------------- robust saver (PNG with DPI metadata + PDF from PNG) -----------------
def save_png_with_dpi(fig, out_png, dpi, fig_w_in, fig_h_in):
    # Save without bbox_inches="tight" to prevent unexpected shrink.
    fig.savefig(out_png, dpi=dpi, facecolor="white")
    # Embed DPI metadata explicitly (no pixel change; PNG is lossless)
    im = Image.open(out_png)
    w, h = im.size
    if w < MIN_WIDTH_PX:
        raise RuntimeError(
            f"Output PNG width too small: {w}px (< {MIN_WIDTH_PX}px). "
            f"Do NOT use bbox_inches='tight'. Check FIG_W_IN/OUT_DPI."
        )
    im.save(out_png, format="PNG", dpi=(dpi, dpi), optimize=False)
    return w, h

def save_pdf_from_png(out_png, out_pdf):
    im = Image.open(out_png).convert("RGB")
    # PDF page size is derived from DPI metadata in the PNG
    im.save(out_pdf, "PDF", resolution=OUT_DPI)

# ----------------- make page -----------------
def make_page(fig_title, subset_df, nrows, ncols, out_base, page_note=None):
    fig = plt.figure(figsize=(FIG_W_IN, FIG_H_IN))
    fig.subplots_adjust(top=0.94, bottom=0.04, left=0.04, right=0.98)

    gs = fig.add_gridspec(nrows=nrows, ncols=ncols, wspace=0.05, hspace=0.25)

    for i in range(nrows*ncols):
        cell = gs[i // ncols, i % ncols]
        sub = GridSpecFromSubplotSpec(
            2, 1, subplot_spec=cell,
            height_ratios=[18, 3], hspace=0.02
        )
        ax_img = fig.add_subplot(sub[0, 0])
        ax_cap = fig.add_subplot(sub[1, 0])
        ax_img.axis("off")
        ax_cap.axis("off")

        if i >= len(subset_df):
            continue

        row = subset_df.iloc[i]
        smi = row["SMILES_probe"]

        ax_cap.text(0.5, 0.55, caption_text(row), ha="center", va="center", fontsize=9)

        if not isinstance(smi, str) or not smi:
            ax_img.text(0.5, 0.5, "SMILES not available", ha="center", va="center", fontsize=12)
            ax_img.set_xlim(0,1); ax_img.set_ylim(0,1)
            continue

        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            ax_img.text(0.5, 0.5, "SMILES parse failed", ha="center", va="center", fontsize=12)
            ax_img.set_xlim(0,1); ax_img.set_ylim(0,1)
            continue

        info={}
        _ = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048, bitInfo=info)
        scores = atom_importance_from_bits(mol, info, coef)
        pngb = draw_attr_no_legend(smi, scores, w=MOL_W, h=MOL_H)
        if pngb is None:
            ax_img.text(0.5, 0.5, "Attribution draw failed", ha="center", va="center", fontsize=12)
            ax_img.set_xlim(0,1); ax_img.set_ylim(0,1)
            continue

        arr = pngbytes_to_array(pngb)
        ax_img.imshow(arr, interpolation="nearest")

    fig.suptitle(fig_title, fontsize=14, y=0.985)
    if page_note:
        fig.text(0.5, 0.012, page_note, ha="center", va="bottom", fontsize=9)

    out_png = out_base + ".png"
    out_pdf = out_base + ".pdf"

    w, h = save_png_with_dpi(fig, out_png, dpi=OUT_DPI, fig_w_in=FIG_W_IN, fig_h_in=FIG_H_IN)
    plt.close(fig)

    save_pdf_from_png(out_png, out_pdf)

    print(f"Saved PNG: {out_png}  ({w} x {h} px, dpi={OUT_DPI})")
    print(f"Saved PDF: {out_pdf}  (from PNG)")

# Keep order
high_df = probe_tbl[probe_tbl["group"]=="High"].copy().set_index("drug").loc[HIGH8].reset_index()
low_df  = probe_tbl[probe_tbl["group"]=="Low"].copy().set_index("drug").loc[LOW9].reset_index()

# Figure5a
make_page(
    fig_title="CiPA 28 compounds: High TdP risk probes (n=8)",
    subset_df=high_df,
    nrows=4, ncols=2,
    page_note="",
    out_base=FIG5A_BASE
)

# Figure5b
make_page(
    fig_title="CiPA 28 compounds: Low TdP risk probes (n=9)",
    subset_df=low_df,
    nrows=5, ncols=2,
    page_note="",
    out_base=FIG5B_BASE
)

print("Done.")

In [ ]:
# triage_external_sim_fixed_thresholds.py
# ============================================================
# External triage simulation (GREEN / YELLOW / RED) — Chemoinfo_QT
#
# Policy: 
#   - internal(JSON) triage Rulesfixed
#   - external(CSV),  triage 「」Output
#
# (Colab/Jupyter): 
#   - ipykernel injects "-f <kernel.json>" so they can be ignored, 
#     argparse  parse_known_args() use(unknown argv )
#
# Input: 
#   1) external prediction CSV(: predictions_external_*.csv)
#      required columns(one of the following): 
#        - y_true / Label / label / y(0/1)
#        - p_ens(default)user-specified probability column
#        - CID(recommended) SMILES
#   2) internal-threshold JSON(threshold_targets_internal_CID.json)
#      typical structure(expected from scripts in this project): 
#        selected:
#          Precision_pos_target_or_fallback: { Threshold: ... }
#          Recall_pos_target:               { Threshold: ... }
#      -> triage typically: 
#         thr_green = Recall side threshold(lower threshold: used for GREEN calls)
#         thr_red   = Precision side threshold(higher threshold: used for RED calls)
#
# 3Rules: 
#   - GREEN : p <  thr_green
#   - YELLOW: thr_green <= p < thr_red
#   - RED   : p >= thr_red
#
# Output(OUTDIR under OUTDIR): 
#   - triage_assignments_external_<GroupTag>.csv
#   - triage_summary_external_<GroupTag>.csv
#   - triage_rule_and_kpi_external_<GroupTag>.json
# ============================================================

import os
import json
import argparse
from pathlib import Path
import numpy as np
import pandas as pd


# =========================
# Defaults
# =========================
DEFAULT_ROOT = "/content/drive/MyDrive/Chemoinfo_QT"
DEFAULT_INTERNAL_THR_JSON = f"{DEFAULT_ROOT}/reports_internal_threshold_search/threshold_targets_internal_CID.json"
DEFAULT_OUTDIR = f"{DEFAULT_ROOT}/reports_triage_external_fixed"


# =========================
# utilities
# =========================
def discover_external_csv(root: str) -> str:
    """
    externalCSVauto-discover()
    - Search broadly using common filename patterns used in this project.
    """
    pats = [
        f"{root}/reports_external_confusions/predictions_external*.csv",
        f"{root}/reports_external*/predictions_external*.csv",
        f"{root}/reports_external*/*external*pred*.csv",
        f"{root}/**/predictions_external*.csv",
    ]
    hits = []
    for pat in pats:
        hits.extend([str(p) for p in Path(root).glob(pat.replace(f"{root}/", ""))] if "*" not in pat else [])
    # Ensure recursive globbing works.
    if not hits:
        hits = [str(p) for p in Path(root).rglob("predictions_external*.csv")]
    if not hits:
        raise FileNotFoundError("External predictions CSV was not found. Please pass --external_csv explicitly.")
    hits = sorted(hits, key=lambda p: os.path.getmtime(p), reverse=True)
    return hits[0]


def _pick_first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def detect_columns(df: pd.DataFrame, prob_col_hint: str = "p_ens"):
    """
    Infer the required columns from the prediction CSV.
    Returns: 
      (y_col, p_col, cid_col, smiles_col)
    """
    cols = list(df.columns)

    # y
    y_col = _pick_first_existing(
        cols,
        ["y_true", "Label", "label", "Y", "y", "target", "Target", "class", "Class"]
    )
    if y_col is None:
        raise ValueError(f"Could not detect label column. Available columns: {cols}")

    # prob
    if prob_col_hint in cols:
        p_col = prob_col_hint
    else:
        p_col = _pick_first_existing(
            cols,
            ["p_ens", "prob", "probability", "score", "p", "pred", "prediction"]
        )
    if p_col is None:
        # last resort: p_ columns starting with
        p_col = next((c for c in cols if str(c).lower().startswith("p_")), None)
    if p_col is None:
        raise ValueError(f"Could not detect probability column. Available columns: {cols}")

    # group key (CID preferred)
    cid_col = _pick_first_existing(
        cols,
        ["CID", "cid", "PubChem_CID", "pubchem_cid", "compound_cid", "drug_cid", "group_key"]
    )
    smiles_col = _pick_first_existing(
        cols,
        ["SMILES", "smiles", "smiles_orig", "smiles_original", "canonical_smiles"]
    )

    return y_col, p_col, cid_col, smiles_col


def load_thresholds_from_json(json_path: str):
    """
    Extract triage thresholds from the internal-threshold JSON.
    Expected: 
      - precision threshold -> thr_red(higher threshold)
      - recall threshold     -> thr_green(lower threshold)
    """
    jp = Path(json_path)
    if not jp.exists():
        raise FileNotFoundError(f"internal_thr_json not found: {json_path}")

    with open(jp, "r", encoding="utf-8") as f:
        obj = json.load(f)

    meta = {"json_path": str(jp)}

    # Direct-key path (for future extensions)
    if "thr_green" in obj and "thr_red" in obj:
        thr_green = float(obj["thr_green"])
        thr_red = float(obj["thr_red"])
        meta["source"] = "direct_keys"
        return thr_green, thr_red, meta

    selected = obj.get("selected", {})
    if not isinstance(selected, dict) or not selected:
        raise ValueError("threshold json missing 'selected' dict and also missing direct thr_green/thr_red keys.")

    # recall candidate
    recall_dict = None
    for k, v in selected.items():
        if isinstance(v, dict) and ("Recall" in k or "recall" in k):
            recall_dict = v
            meta["recall_key"] = k
            break

    # precision candidate
    prec_dict = None
    for k, v in selected.items():
        if isinstance(v, dict) and ("Precision" in k or "precision" in k):
            prec_dict = v
            meta["precision_key"] = k
            break

    if recall_dict is None or prec_dict is None:
        raise ValueError(f"Could not locate recall/precision entries in json['selected']. Keys={list(selected.keys())}")

    if "Threshold" not in recall_dict or "Threshold" not in prec_dict:
        raise ValueError("selected recall/precision dict must contain 'Threshold'.")

    thr_green = float(recall_dict["Threshold"])
    thr_red = float(prec_dict["Threshold"])

    meta["source"] = "selected_dict"
    meta["thr_green_from"] = recall_dict.get("criterion", "recall_threshold")
    meta["thr_red_from"] = prec_dict.get("criterion", "precision_threshold_or_fallback")

    return thr_green, thr_red, meta


def aggregate_drug_level(df: pd.DataFrame,
                         y_col: str,
                         p_col: str,
                         key_col: str,
                         prob_agg: str = "mean",
                         y_agg: str = "max") -> pd.DataFrame:
    """
    Aggregate at the drug level using group_key (e.g., CID).
      - y_true: max (conservative) or mode
      - p: mean or median
    """
    d = df.copy()

    d[y_col] = pd.to_numeric(d[y_col], errors="coerce")
    d[p_col] = pd.to_numeric(d[p_col], errors="coerce")
    d = d.dropna(subset=[y_col, p_col, key_col])

    # label clean
    d[y_col] = d[y_col].astype(int)
    d = d[d[y_col].isin([0, 1])].copy()

    # y aggregation
    if y_agg == "mode":
        def agg_mode(arr):
            vals, cnts = np.unique(arr, return_counts=True)
            pick = vals[np.argmax(cnts)]
            # Surface label conflicts here; restrict to top cases if the output becomes too large.
            if len(vals) > 1:
                print(f"[WARN] y_true conflict in a group -> using mode={pick} (counts {dict(zip(vals, cnts))})")
            return int(pick)
        y_fn = agg_mode
    else:
        # conservative: any positive -> positive
        y_fn = "max"

    # prob aggregation
    if prob_agg == "median":
        p_fn = lambda x: float(np.median(pd.to_numeric(x, errors="coerce").dropna().values))
    else:
        p_fn = lambda x: float(np.mean(pd.to_numeric(x, errors="coerce").dropna().values))

    g = (
        d.groupby(key_col, dropna=False, as_index=False)
         .agg(
            y_true=(y_col, y_fn),
            p_ens=(p_col, p_fn),
            n_instances=(p_col, "count")
         )
    )
    return g


def confusion_counts(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tp, fp, tn, fn


def metrics_from_counts(tp, fp, tn, fn):
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    npv = tn / (tn + fn) if (tn + fn) else 0.0
    acc = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Precision_pos": precision,
        "Recall_pos": recall,
        "Specificity": specificity,
        "NPV": npv,
        "Accuracy": acc,
        "F1_pos": f1
    }


def summarize_tiers(g: pd.DataFrame, thr_green: float, thr_red: float):
    """
    Assign GREEN/YELLOW/RED tiers and return tier-wise summaries and KPIs.
    """
    d = g.copy()
    p = d["p_ens"].to_numpy(dtype=float)

    tier = np.where(p < thr_green, "GREEN",
            np.where(p >= thr_red, "RED", "YELLOW"))
    d["tier"] = tier

    # tier summary
    tier_df = (
        d.groupby("tier", as_index=False)
         .agg(
            N=("tier", "size"),
            Pos=("y_true", "sum"),
            Neg=("y_true", lambda x: int((np.array(x) == 0).sum())),
            MeanProb=("p_ens", "mean"),
            MedianProb=("p_ens", "median"),
            MeanInstances=("n_instances", "mean")
         )
    )
    tier_df["PosRate"] = tier_df["Pos"] / tier_df["N"]

    # KPI 1: RED treat as a positive prediction(binarization: RED vs not-RED)
    y_true = d["y_true"].to_numpy(dtype=int)
    y_pred_red = (d["tier"].to_numpy() == "RED").astype(int)
    tp, fp, tn, fn = confusion_counts(y_true, y_pred_red)
    kpi_red = metrics_from_counts(tp, fp, tn, fn)
    kpi_red["Definition"] = "Predict positive = RED only"

    # KPI 2: YELLOW+RED treat as a positive prediction(binarization: not-GREEN)
    y_pred_not_green = (d["tier"].to_numpy() != "GREEN").astype(int)
    tp2, fp2, tn2, fn2 = confusion_counts(y_true, y_pred_not_green)
    kpi_not_green = metrics_from_counts(tp2, fp2, tn2, fn2)
    kpi_not_green["Definition"] = "Predict positive = (YELLOW or RED)"

    # KPI 3: GREEN 「negative-assurance」(GREENtreat only GREEN as a negative call)
    # reference: GREEN NPV / FN compute separately
    green = d[d["tier"] == "GREEN"].copy()
    if len(green) > 0:
        green_pos = int(green["y_true"].sum())
        green_neg = int((green["y_true"] == 0).sum())
        green_pos_rate = green_pos / len(green)
    else:
        green_pos = green_neg = 0
        green_pos_rate = np.nan

    # capture rate
    pos_total = int(y_true.sum())
    pos_red = int(d.loc[d["tier"] == "RED", "y_true"].sum())
    pos_yellow_red = int(d.loc[d["tier"].isin(["YELLOW", "RED"]), "y_true"].sum())
    capture = {
        "Pos_total": pos_total,
        "Pos_in_RED": pos_red,
        "Pos_in_YELLOW_or_RED": pos_yellow_red,
        "Capture_RED": (pos_red / pos_total) if pos_total else np.nan,
        "Capture_YELLOW_or_RED": (pos_yellow_red / pos_total) if pos_total else np.nan,
        "GREEN_Pos": green_pos,
        "GREEN_Neg": green_neg,
        "GREEN_PosRate": green_pos_rate
    }

    kpi = {"RED_only": kpi_red, "YELLOW_or_RED": kpi_not_green, "capture": capture}
    return tier_df.sort_values("tier"), kpi, d


# =========================
# Main
# =========================
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=DEFAULT_ROOT)
    ap.add_argument("--external_csv", default=None, help="external predictions csv (if omitted, auto-discover)")
    ap.add_argument("--internal_thr_json", default=DEFAULT_INTERNAL_THR_JSON, help="threshold_targets_internal_CID.json")
    ap.add_argument("--thr_green", type=float, default=None, help="override GREEN upper threshold")
    ap.add_argument("--thr_red", type=float, default=None, help="override RED lower threshold")
    ap.add_argument("--prob_col", default="p_ens", help="probability column (default: p_ens)")
    ap.add_argument("--prob_agg", choices=["mean", "median"], default="mean")
    ap.add_argument("--y_agg", choices=["max", "mode"], default="max")
    ap.add_argument("--outdir", default=None, help=f"output directory (default: {DEFAULT_OUTDIR})")

    # ★Colab/Jupyter compatibility: ignore unknown arguments such as -f kernel.json.
    args, unknown = ap.parse_known_args(args=argv)
    if unknown:
        print("[INFO] Ignoring unknown argv from notebook kernel:", unknown)

    root = args.root
    outdir = args.outdir or DEFAULT_OUTDIR
    os.makedirs(outdir, exist_ok=True)

    external_csv = args.external_csv or discover_external_csv(root)
    if not Path(external_csv).exists():
        raise FileNotFoundError(f"not found: {external_csv}")

    thr_green, thr_red, thr_meta = load_thresholds_from_json(args.internal_thr_json)
    if args.thr_green is not None:
        thr_green = float(args.thr_green); thr_meta["thr_green_override"] = True
    if args.thr_red is not None:
        thr_red = float(args.thr_red); thr_meta["thr_red_override"] = True

    if not (thr_green < thr_red):
        raise ValueError(f"threshold order invalid: thr_green={thr_green}, thr_red={thr_red}")

    print("[INFO] External CSV:", external_csv)
    print("[INFO] Thresholds fixed:", {"thr_green": thr_green, "thr_red": thr_red})
    print("[INFO] Threshold meta:", thr_meta)

    df = pd.read_csv(external_csv)
    y_col, p_col, cid_col, smiles_col = detect_columns(df, prob_col_hint=args.prob_col)

    # grouping key
    if cid_col is not None:
        key_col = cid_col
        group_tag = "CID"
    elif smiles_col is not None:
        key_col = smiles_col
        group_tag = "SMILES"
    else:
        raise ValueError("Neither CID nor SMILES column found for grouping.")

    print("[INFO] Detected columns:", {"y_col": y_col, "p_col": p_col, "group_key": f"{group_tag}({key_col})"})

    g = aggregate_drug_level(
        df=df,
        y_col=y_col,
        p_col=p_col,
        key_col=key_col,
        prob_agg=args.prob_agg,
        y_agg=args.y_agg,
    )

    print(f"[INFO] Drug-level N={len(g)} | Pos={int(g['y_true'].sum())} | Neg={int((g['y_true']==0).sum())}")

    tier_df, kpi, assign = summarize_tiers(g, thr_green=thr_green, thr_red=thr_red)

    assign_csv = f"{outdir}/triage_assignments_external_{group_tag}.csv"
    assign.to_csv(assign_csv, index=False)

    summary_csv = f"{outdir}/triage_summary_external_{group_tag}.csv"
    tier_df.to_csv(summary_csv, index=False)

    rule_json = f"{outdir}/triage_rule_and_kpi_external_{group_tag}.json"
    with open(rule_json, "w", encoding="utf-8") as f:
        json.dump(
            {
                "external_csv": external_csv,
                "group_tag": group_tag,
                "prob_col": p_col,
                "prob_agg": args.prob_agg,
                "y_agg": args.y_agg,
                "thresholds": {"thr_green": thr_green, "thr_red": thr_red},
                "threshold_meta": thr_meta,
                "tier_summary": tier_df.to_dict(orient="records"),
                "kpi": kpi,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("\n=== Tier summary (external) ===")
    print(tier_df.to_string(index=False))
    print("\n=== KPI (reference) ===")
    print(json.dumps(kpi, ensure_ascii=False, indent=2))
    print("\n[SAVE] assignments:", assign_csv)
    print("[SAVE] summary    :", summary_csv)
    print("[SAVE] kpi json    :", rule_json)
    print("\nDone.")


if __name__ == "__main__":
    main()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%run /content/drive/MyDrive/Chemoinfo_QT/triage_external_strict_sim_fixed_thresholds.py